# Mathematical Foundations of Language Model Systems

*A derivations-first textbook, with the Gibbs variational principle as the recurring thread.*

This notebook is a single-file textbook built on the `llm_from_scratch` teaching implementation in this repository. It follows the same 13-section journey as the notebook sequence, written for a mathematically trained reader in the style of a mathematical-physics text: every object is introduced with its domain and codomain, every central formula is derived rather than asserted, and every derivation must eventually land in a tensor, a training objective, or a behavior of the toy decoder-only GPT implemented in `src/llm_from_scratch`.

The reader I have in mind is comfortable with linear algebra, multivariable calculus, basic probability, and proof-style reasoning. The reader may not yet be comfortable with machine learning habits: empirical risk, stochastic optimization, logits, masking, sampling, validation loss, or the difference between a model class and a training procedure. This notebook bridges those habits without hiding the algebra.

## Conventions of this text

**Numbered environments.** Definitions, lemmas, propositions, and theorems share one counter per chapter: Proposition 4.4 is the fourth numbered statement of Chapter 4. Proofs end with $\blacksquare$. Cross-references always cite the number.

**Typing.** Every map is introduced with an explicit domain and codomain, in the style of mathematical physics. All spaces here are finite-dimensional: a tensor of shape `(B, T, C)` is a point of $\mathbb{R}^{B\times T\times C}$, and a neural network layer is a smooth map between such spaces, usually depending on parameters. When the product structure of a space matters (batch, position, channel, vocabulary), we say which axis carries which role before writing code.

**Indices.** Token IDs are 0-based to match the code: $\mathcal{V}=\{0,1,\ldots,V-1\}$. Sequence positions are 1-based: $x_{1:T}=(x_1,\ldots,x_T)$. This convention is used consistently, including inside cross-entropy.

**Masks.** All masks are additive: a mask is a matrix $M \in \{0,-\infty\}^{T\times T}$ added to scores before softmax. The `masked_fill` idiom in the code is the implementation of this additive convention.

**Physics bridges.** The Gibbs distribution appears three times in this text, each time derived from the same variational principle (Theorem 1.6): as the softmax output head (Chapter 1), as the attention weights over prefix positions (Chapter 4), and as the optimum of KL-regularized preference alignment (Chapter 11). Watching one structure recur at three scales is the main aesthetic payoff of the mathematical treatment.

## The central object

The central object is an autoregressive language model. Given a finite vocabulary of token IDs and a prefix of tokens, it returns a probability distribution for the next token. In symbols, if

$$
x_{1:T} = (x_1, x_2, \ldots, x_T), \qquad x_t \in \mathcal{V} = \{0,1,\ldots,V-1\},
$$

then the model represents conditional probabilities of the form

$$
p_\theta(x_t \mid x_{<t}),
$$

where $\theta$ denotes all trainable parameters: embedding tables, attention projections, MLP weights, layer norm parameters, and the final language-model head. The curriculum builds a tiny version of this map from scratch, then shows how the same ideas appear in PyTorch, Hugging Face, quantization, and modern LLM practice.

A good way to use this walkthrough:

1. Read a section once without running code.
2. Write down the mathematical object being defined, with its domain and codomain.
3. Predict every tensor shape in the next code cell.
4. Run the code cell.
5. Change one small dimension or hyperparameter and explain exactly which formula or tensor contract changed.
6. Treat the exercises as self-checks, not as a separate homework set.

The notebook is intentionally CPU-friendly. It is not trying to train a useful assistant. It is trying to make the toy implementation mathematically legible.


## Map To The Existing Curriculum

The original notebooks are still the best way to study one topic at a time. This file is the continuous pass. It keeps the same arc, but makes the mathematical and statistical contracts explicit.

| Chapter in this notebook | Original notebook | Main mathematical question | Main code path |
| --- | --- | --- | --- |
| 0. Orientation | `00_project_orientation.ipynb` | What distribution does an autoregressive model represent? | `TinyGPT.forward` returns logits for next-token prediction |
| 1. Tensors, autograd, probability | `01_tensors_autograd_and_probability.ipynb` | How do logits become a differentiable negative log-likelihood? | `stable_softmax`, `cross_entropy_from_logits` |
| 2. Tokenization | `02_text_tokenization_char_to_subword.ipynb` | How does text become a finite sequence over a vocabulary? | `CharTokenizer`, `train_bpe_tokenizer` |
| 3. Embeddings and language modeling | `03_embeddings_and_language_modeling.ipynb` | How does a finite set map into a vector space, and how does one sequence become many labels? | `nn.Embedding`, `get_batch`, `TinyGPT.token_embedding` |
| 4. Attention | `04_attention_from_raw_tensors.ipynb` | How does each position form a probability distribution over earlier positions? | `scaled_dot_product_attention`, `CausalSelfAttention` |
| 5. Transformer block | `05_transformer_block_from_scratch.ipynb` | How are residual maps, normalization, attention, and MLPs composed? | `GPTBlock`, `FeedForward`, `TinyGPT` |
| 6. Training | `06_training_loop_loss_and_optimization.ipynb` | How does empirical risk minimization change parameters? | `train_steps`, `overfit_tiny_batch` |
| 7. Generation and evaluation | `07_generation_sampling_and_evaluation.ipynb` | How does a trained conditional distribution produce a sequence? | `generate`, `sample_next_token`, `perplexity` |
| 8. Toy fine-tuning | `08_finetuning_toy_instruction_task.ipynb` | How does masking turn some positions into supervised targets and others into context? | `toy_instruction_examples`; masked risk illustrated directly |
| 9. Library translation | `09_hugging_face_translation_layer.ipynb` | Which from-scratch objects become library abstractions? | `datasets`, `tokenizers`, `GPT2LMHeadModel` |
| 10. Quantization | `10_quantization_deep_dive.ipynb` | How does a real tensor map to a finite integer grid? | `quantize_tensor`, `dequantize_tensor`, `estimate_kv_cache_bytes` |
| 11. Modern orientation | `11_modern_llm_orientation.ipynb` | How do objectives, data distributions, retrieval, evaluation, and serving wrap the core model? | Orientation map in this notebook |
| 12. Beyond transformers | `12_beyond_transformers_and_world_models.ipynb` | Which mathematical bottleneck is a new architecture trying to relax? | Attention cost comparison in this notebook |
| 13. Study path | this notebook | How should a mathematical reader keep studying without losing the implementation thread? | Exercises and detailed notebooks |

### Source note

This revision cross-checks the curriculum against Tong Xiao and Jingbo Zhu, *Foundations of Large Language Models* (arXiv:2501.09223v2, CC BY 4.0). The additions are paraphrased and folded into this repo's toy implementation rather than copied as textbook excerpts. The most useful imported emphases are: causal language modeling as right-context masking, the distinction between prefill and decoding during inference, KV-cache sharing through MQA/GQA, RAG/tool-use as problem decomposition, and DPO as a preference objective that avoids an explicit reward model.

The repeating pattern is:

1. State the mathematical object.
2. Attach dimensions and tensor shapes.
3. Explain its statistical or optimization role.
4. Derive the formula at an undergraduate level.
5. Point to the exact toy code path.
6. Interpret the abstraction.
7. Warn about common ML failure modes.

When a topic does not satisfy that pattern, it is probably still too vague.


In [ ]:
from pathlib import Path
import sys

# ── Environment setup ────────────────────────────────────────────────
# On Google Colab this cell fetches the curriculum and its dependencies;
# on a local checkout it only sets up the import path.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/zeroexstrat/lm-foundations.git"

if IN_COLAB:
    if not Path("lm-foundations").exists():
        !git clone --depth 1 {REPO_URL} lm-foundations
    %pip install -q tokenizers transformers datasets
    PROJECT_ROOT = Path("lm-foundations").resolve()
else:
    PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PROJECT_ROOT


## 0. What A Language Model Computes

**Why this chapter.** You have sent a prompt to a model and read a completion. This chapter strips away everything — the chat interface, the servers, the safety layers, the very idea of "answering" — until what remains is the single question the machine actually computes: *given this prefix of tokens, what probability distribution governs the next one?* Everything else in this book is machinery for representing, training, and sampling that one map. Getting its type exactly right, before any neural network appears, is the chapter's entire job.

### Mathematical object

**Definition 0.1 (vocabulary and sequence spaces).** Fix an integer $V\ge 2$. The *vocabulary* is the finite set

$$
\mathcal{V} = \{0,1,\ldots,V-1\}.
$$

For $T\ge 1$, the space of token sequences of length $T$ is the finite set $\mathcal{V}^T$, and the space of nonempty finite sequences is $\mathcal{V}^{*} = \bigcup_{t\ge 1}\mathcal{V}^{t}$. A *prefix* of length $t$ is an element of $\mathcal{V}^t$; we write $\mathcal{V}^0 = \{\varnothing\}$ for the empty prefix.

**Definition 0.2 (probability simplex).** The probability simplex over $\mathcal{V}$ is

$$
\Delta^{V-1} = \Big\{p \in \mathbb{R}^{V}: p_i \ge 0,\ \sum_{i=0}^{V-1}p_i=1\Big\},
$$

a compact convex $(V-1)$-dimensional subset of $\mathbb{R}^V$. Its relative interior $\mathring{\Delta}^{V-1}$ consists of the strictly positive probability vectors.

**Definition 0.3 (autoregressive language model).** An autoregressive language model with maximal context $T_{\max}$ is a map

$$
p_\theta : \bigcup_{t=0}^{T_{\max}-1} \mathcal{V}^{t} \longrightarrow \mathring{\Delta}^{V-1},
\qquad
x_{1:t} \longmapsto p_\theta(\,\cdot \mid x_{1:t}),
$$

depending on a parameter $\theta$ (the parameter space $\Theta$ is defined precisely in Definition 1.1). It assigns to each prefix a strictly positive distribution over the next token. The induced joint distribution on $\mathcal{V}^T$ is defined by the product

$$
P_\theta(X_{1:T}=x_{1:T})
= \prod_{t=1}^{T} p_\theta(x_t \mid x_{<t}).
$$

The following proposition makes precise the sense in which this factorization loses no generality: autoregression is not itself a modeling approximation.

**Proposition 0.4 (chain-rule factorization is a bijection).** The map that sends a strictly positive probability measure $P$ on $\mathcal{V}^T$ to its family of conditionals

$$
p(x_t \mid x_{<t}) = \frac{P(X_{1:t} = x_{1:t})}{P(X_{1:t-1}=x_{1:t-1})},
\qquad 1\le t\le T,
$$

is a bijection between strictly positive measures on $\mathcal{V}^T$ and families of maps $\mathcal{V}^{t-1}\to\mathring{\Delta}^{V-1}$, $1\le t \le T$.

*Proof.* Given $P$ strictly positive, each conditional above is well defined (the denominator is a positive marginal) and strictly positive, and the telescoping product

$$
\prod_{t=1}^{T} \frac{P(x_{1:t})}{P(x_{1:t-1})} = P(x_{1:T})
$$

recovers $P$. Conversely, given any family of conditionals valued in $\mathring{\Delta}^{V-1}$, the product defines a strictly positive function on $\mathcal{V}^T$; summing over $x_T$, then $x_{T-1}$, and so on, each sum collapses to $1$ by normalization of the conditionals, so the total mass is $1$ and the function is a probability measure whose conditionals are the given family. The two constructions are mutually inverse. $\blacksquare$

The modeling choice, therefore, is not the factorization. It is that a *single* parameterized map $p_\theta$, with shared weights, computes every conditional factor from its prefix. Taking logarithms converts the product into the sum

$$
\log P_\theta(x_{1:T})
= \sum_{t=1}^{T} \log p_\theta(x_t \mid x_{<t}),
$$

and maximum-likelihood training minimizes the negative of this sum. This is the first reason language modeling becomes a per-token supervised learning problem: one text sequence of length $T$ yields $T$ prediction terms (Worked Example 0.1 computes all three for the word `"cat"`). (In practice a beginning-of-sequence token gives even the first factor a nonempty context.)

### Causal language modeling among pretraining tasks

Decoder-only next-token training is one member of a broader family of self-supervised pretraining tasks. Causal language modeling is a mask-predict task in which every position is prevented from using its right context: for token $x_t$ the visible context is $x_{<t}$ and the hidden part is the future.

Masked language modeling, used by encoder-style systems, chooses a subset of positions $A(x)$, replaces those positions with a mask symbol, and predicts only the masked tokens from the corrupted sequence $\bar{x}$:

$$
\sum_{i\in A(x)} \log p_\theta(x_i\mid \bar{x}).
$$

The important contrast is context direction. A causal decoder predicts from left context only; a masked encoder may use both left and right unmasked context; permuted language modeling changes the prediction order via attention masks while preserving token positions. This notebook keeps the decoder-only case because it is the path implemented by `TinyGPT`: a lower-triangular mask, next-token targets, and logits at every position.

### Dimensions and tensor shapes

In the toy model, a batch of input token IDs is a point of $\mathcal{V}^{B\times T}$, stored as an integer tensor of shape `(B, T)`:

$$
\texttt{idx}[b,t] \in \mathcal{V}.
$$

The model returns logits in $\mathbb{R}^{B\times T\times V}$, shape `(B, T, V)`. The vector `logits[b, t, :]` $\in \mathbb{R}^V$ contains one real score per vocabulary item as the candidate target for position $t$; after softmax (Definition 1.2) it becomes a point of $\mathring{\Delta}^{V-1}$.

The code path is `TinyGPT.forward` in `src/llm_from_scratch/model.py`. It computes token and position embeddings, applies transformer blocks, normalizes the residual stream, and applies `lm_head` to obtain logits:

```python
logits = self.lm_head(x)  # shape (B, T, V)
```

If targets are supplied, the same method calls PyTorch cross-entropy on flattened token positions:

```python
loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
```

Flattening from `(B, T, V)` to `(B*T, V)` means every batch-position pair becomes one multiclass classification example.

### Statistical role

The statistical problem is not to memorize a deterministic next-token function. Natural language is not a function from prefixes to unique continuations: the same prefix admits many plausible next tokens. The model estimates a conditional distribution, and training asks whether the observed next token received high probability under it.

Write $y_{b,t} := x_{b,t+1}$ for the shifted target (Chapter 3 constructs this shift explicitly). For a dataset of token sequences, the empirical risk is the average negative log-likelihood over observed positions:

$$
\widehat{R}(\theta)
= \frac{1}{N}\sum_{n=1}^{N}\frac{1}{T_n-1}\sum_{t=1}^{T_n-1}
-\log p_\theta\big(x^{(n)}_{t+1}\mid x^{(n)}_{1:t}\big).
$$

The toy batching code samples fixed windows, so the implementation-level objective is

$$
\widehat{R}_{\text{batch}}(\theta)
= \frac{1}{BT}\sum_{b=1}^{B}\sum_{t=1}^{T}
-\log p_\theta\big(y_{b,t}\mid x_{b,1:t}\big).
$$

That is exactly the scalar loss used for backpropagation.

> **Mathematician's lens.** The model is a map from a finite prefix space into the interior of a probability simplex; Proposition 0.4 says this representation is fully general. The transformer is one parameterization of that map. Attention, embeddings, MLPs, and layer norms are not separate statistical models; they are coordinate-level machinery for computing the conditional probability vector.

> **ML practitioner's warning.** A low training loss says only that the model assigned high probability to observed tokens under the training distribution. It does not prove factuality, reasoning, instruction-following, calibration, robustness, or safe behavior. Those require additional data, objectives, evaluations, and constraints.

### Self-checks

1. If logits have shape `(8, 16, 100)`, how many multiclass classification examples are used by the cross-entropy call after flattening?
2. Why is the sequence log-likelihood a sum but the sequence probability a product?
3. Where in `TinyGPT.forward` does the model become a distribution over vocabulary items rather than a hidden representation?
4. (*) In Proposition 0.4, where does strict positivity get used, and what fails for measures with zeros?


**Worked Example 0.1 (the chain rule on `"cat"`, term by term).** Take the running vocabulary of this book, $\mathcal{V} = \{\texttt{' '}\!:0,\ \texttt{a}\!:1,\ \texttt{c}\!:2,\ \texttt{t}\!:3\}$ ($V=4$; built in Worked Example 2.1), and suppose some model assigns the three conditionals

$$
p(\texttt{c}) = 0.4, \qquad
p(\texttt{a}\mid\texttt{c}) = 0.5, \qquad
p(\texttt{t}\mid\texttt{ca}) = 0.6 .
$$

The joint probability of Definition 0.3 is the product

$$
P(\texttt{cat}) = 0.4 \times 0.5 \times 0.6 = 0.12,
$$

and the negative log-likelihood is the sum, one term per position:

$$
-\log P(\texttt{cat})
= \underbrace{0.9163}_{-\ln 0.4} + \underbrace{0.6931}_{-\ln 0.5} + \underbrace{0.5108}_{-\ln 0.6}
= 2.1203 \text{ nats}.
$$

The per-token average is $2.1203/3 = 0.7068$ nats — this single number is what every loss curve in Chapter 6 plots, and $e^{0.7068} = 2.0274 = (1/0.12)^{1/3}$ is its perplexity (Proposition 6.4): the model is as uncertain, on geometric average, as a uniform choice among about two continuations. One sequence, three supervised terms, one scalar — the entire training pipeline in miniature.

In [ ]:
import math

# Worked Example 0.1 — verify the hand arithmetic
p_c, p_a_given_c, p_t_given_ca = 0.4, 0.5, 0.6

joint = p_c * p_a_given_c * p_t_given_ca
nll_terms = [-math.log(p) for p in (p_c, p_a_given_c, p_t_given_ca)]
total_nll = sum(nll_terms)
mean_nll = total_nll / 3
ppl = math.exp(mean_nll)

assert abs(joint - 0.12) < 1e-12
assert abs(total_nll - 2.1203) < 5e-5
assert abs(ppl - (1 / joint) ** (1 / 3)) < 1e-12   # PPL = geometric mean of inverse probs
print(f"P(cat) = {joint}")
print(f"-log terms: {[round(t, 4) for t in nll_terms]}  sum = {total_nll:.4f} nats")
print(f"mean = {mean_nll:.4f} nats/token   perplexity = {ppl:.4f}")

### Classical Baseline: N-Grams Before Neural LMs

Before transformers, the cleanest language-model baseline was the n-gram model. It uses the same chain-rule target as the neural model, but it replaces the full prefix with a short fixed history and estimates probabilities from counts.

The exact chain rule is

$$
P(x_{1:T}) = \prod_{t=1}^{T} P(x_t \mid x_{<t}).
$$

An n-gram model approximates each factor by conditioning only on the previous $n-1$ tokens:

$$
P(x_t \mid x_{<t}) \approx P(x_t \mid x_{t-n+1:t-1}).
$$

The maximum-likelihood count estimate has the form

$$
\hat P(w_t \mid h) = \frac{\operatorname{count}(h, w_t)}{\operatorname{count}(h)},
$$

where $h$ is the history, such as the previous one or two tokens. This simple ratio exposes three issues that still matter for LLMs:

1. **Data sparsity.** Many valid histories never appear in a finite training corpus.
2. **Generalization.** A model must assign probability to held-out text, not just memorized training strings.
3. **Evaluation protocol.** Train, development, and test splits must stay separate, or probability-based metrics become misleading.

Neural LMs keep the same autoregressive question but replace sparse count tables with learned embeddings, attention, nonlinear layers, and gradient descent. The transformer is therefore not a new probability problem. It is a more flexible estimator of the same conditional factors.

Source note: this bridge follows the CS124 n-gram material and SLP3 Chapter 3, especially the chain-rule, train/test, smoothing, and perplexity discussion in `../data/cs124/slp3_chapters/3.pdf` and `../data/cs124/lectures/lm_jan25.pdf`.


### Smoothing: The Classical Fix For Sparse Counts

The maximum-likelihood n-gram estimate is brutally literal:

$$
\hat P(w \mid h)=\frac{\operatorname{count}(h,w)}{\operatorname{count}(h)}.
$$

If a valid continuation never appears after history $h$ in the training corpus, this estimate assigns it probability zero. In a sequence model, one zero factor makes the whole sequence probability zero and the log-likelihood unusable. This is not just an old n-gram problem. It is the finite-data problem in its simplest form.

Classical smoothing changes the estimate so the model reserves some probability mass for rare or unseen events. Three common moves are:

| Method | Core idea | Undergrad intuition |
| --- | --- | --- |
| Additive smoothing | add a small pseudo-count before normalizing | pretend every event had at least tiny evidence |
| Interpolation | mix higher-order and lower-order n-gram estimates | trust long histories when available, fall back when sparse |
| Backoff | use a shorter history when the longer history is unreliable | if `in San` is too specific, try just `San` or the unigram |

Kneser-Ney smoothing adds a deeper idea: a word's usefulness is not only how often it appears, but how many different contexts it can complete. A token like `Francisco` may be frequent after `San` but not a generally likely continuation everywhere.

The neural version of this story is parameter sharing. Embeddings, attention heads, and MLPs let evidence from one context affect predictions in related contexts. They do not remove the need for held-out evaluation; they are a more flexible way to generalize beyond exact count tables.

Source note: this section comes from the CS124 n-gram and smoothing material in `../data/cs124/slp3_chapters/3.pdf`, the Kneser-Ney appendix `../data/cs124/slp3_chapters/C.pdf`, and `../data/cs124/lectures/lm_jan25.pdf`.


**Worked Example 0.2 (counts, a zero, and its repair — `"the cat"` at word level).** Take a six-word vocabulary $\{\texttt{the}, \texttt{cat}, \texttt{sat}, \texttt{ran}, \texttt{dog}, \texttt{.}\}$ and the three-sentence corpus

$$
\texttt{the cat sat . the cat ran . the dog sat .}
$$

Counting gives $\operatorname{count}(\texttt{the}) = 3$, $\operatorname{count}(\texttt{the}, \texttt{cat}) = 2$, $\operatorname{count}(\texttt{the}, \texttt{dog}) = 1$, so the bigram MLE estimates are

$$
\hat P(\texttt{cat} \mid \texttt{the}) = \frac{2}{3}, \qquad
\hat P(\texttt{dog} \mid \texttt{the}) = \frac{1}{3},
$$

count-and-divide, exactly as promised. Now the catastrophe: $\texttt{dog}$ occurs once, followed by $\texttt{sat}$, so

$$
\hat P(\texttt{ran} \mid \texttt{dog}) = \frac{0}{1} = 0,
$$

and the perfectly grammatical sentence $\texttt{the dog ran .}$ receives probability $0 \cdot (\ldots) = 0$ — log-likelihood $-\infty$. One unseen bigram poisons the entire sequence. Add-1 smoothing repairs the row by pretending every continuation was seen once more:

$$
\hat P_{+1}(w \mid \texttt{dog}) = \frac{\operatorname{count}(\texttt{dog}, w) + 1}{\operatorname{count}(\texttt{dog}) + V}
\quad\Longrightarrow\quad
\hat P_{+1}(\texttt{ran} \mid \texttt{dog}) = \frac{0 + 1}{1 + 6} = \frac{1}{7} \approx 0.143,
$$

and note the price, visible in the same row: the *seen* continuation fell from $\hat P(\texttt{sat}\mid\texttt{dog}) = 1$ to $2/7 \approx 0.286$. Smoothing does not create probability; it taxes observed events to fund unobserved ones — the row still sums to $1$. Every neural language model in this book is, at bottom, a subtler answer to this same zero: instead of taxing counts, share parameters so that evidence about $\texttt{cat ran}$ informs $\texttt{dog ran}$.

In [ ]:
from collections import Counter

# Worked Example 0.2 — bigram counts, the zero, and add-1 smoothing
words = "the cat sat . the cat ran . the dog sat .".split()
uni = Counter(words)
bi = Counter(zip(words, words[1:]))
V = len(uni)

assert uni["the"] == 3 and bi[("the", "cat")] == 2
assert bi[("the", "cat")] / uni["the"] == 2 / 3          # P(cat | the)
assert bi[("dog", "ran")] / uni["dog"] == 0.0            # the zero
p_add1 = lambda w2, w1: (bi[(w1, w2)] + 1) / (uni[w1] + V)
assert abs(p_add1("ran", "dog") - 1 / 7) < 1e-12         # repaired
assert abs(p_add1("sat", "dog") - 2 / 7) < 1e-12         # ...and the seen event taxed
assert abs(sum(p_add1(w, "dog") for w in uni) - 1.0) < 1e-9   # row still sums to 1
print(f"P(cat|the) = {bi[('the','cat')]}/{uni['the']} = {2/3:.4f}")
print(f"P(ran|dog) = 0  →  add-1: {1/7:.4f}   (P(sat|dog): 1 → {2/7:.4f})")

### The Three Levels We Will Keep Separating

A recurring discipline in this notebook is to separate three levels of description.

| Level | Example statement | What can go wrong if ignored |
| --- | --- | --- |
| Mathematical object | Softmax maps $z \in \mathbb{R}^V$ to $p \in \Delta^{V-1}$. | You may forget which axis is normalized. |
| Tensor implementation | `logits` has shape `(B, T, V)` and softmax acts on `dim=-1`. | You may normalize over positions instead of vocabulary. |
| Software abstraction | `TinyGPT.forward(idx, targets)` returns `(logits, loss)`. | You may misunderstand what the API has already shifted or flattened. |

A mathematically correct sentence can still be insufficient for programming. For example, saying "attention computes $\operatorname{softmax}(QK^T)V$" omits the batch dimension, head dimension, mask broadcasting, and the axis of softmax. In this repo, the single-head teaching function `scaled_dot_product_attention` works with tensors shaped `(B, T, D)`, while the module `CausalSelfAttention` reshapes a `(B, T, C)` residual stream into `(B, H, T, D)`.

We will use the following notation throughout:

| Symbol | Meaning | Implementation name |
| --- | --- | --- |
| $B$ | batch size | first dimension of `idx` |
| $T$ | context length, block size, or sequence length | second dimension of `idx` |
| $V$ | vocabulary size | `config.vocab_size` |
| $C$ | embedding width, channel dimension | `config.n_embd` |
| $H$ | number of attention heads | `config.n_head` |
| $D$ | per-head dimension | `C // H` |
| $\theta$ | all trainable parameters | `model.parameters()` |

Two shape habits prevent many mistakes:

1. Name the semantic role of every axis before coding.
2. Check that any reduction, softmax, mean, or gather is applied along the intended axis.

For example, cross-entropy over vocabulary should reduce the `V` axis. Averaging the loss should reduce over token positions and batch examples. Causal masking should modify the source-position axis for each receiving position.

> **Mathematician's lens.** Many neural network operations are familiar maps between finite-dimensional vector spaces, but the product structure of the space matters. A tensor in $\mathbb{R}^{B\times T\times C}$ is not just a vector in $\mathbb{R}^{BTC}$ when the operation is supposed to preserve batch examples, respect causal order, or normalize over vocabulary.

> **ML practitioner's warning.** Shape-compatible code can still be semantically wrong. Broadcasting may make an incorrect mask run without error. A loss may decrease while the model is learning from leaked future tokens or shifted labels.


In [ ]:
import math
import torch

from llm_from_scratch.configs import ModelConfig, TrainConfig, set_seed, get_device

set_seed(123)
device = torch.device("cpu")  # CPU keeps this notebook predictable and lightweight.
print("Project device preference would choose:", get_device())
print("This walkthrough executes small demonstrations on:", device)


### Where We Stand

**What we now have.** A language model, stripped of every implementation detail, is an answer to one question: *given this prefix, what distribution governs the next token?* Proposition 0.4 showed this is fully general — any distribution over sequences is exactly a family of such answers — and the n-gram sections showed the classical way to produce the answers (count and divide), along with its failure mode (zeros) and its patch (smoothing). Training reduced to a single scalar: average negative log-likelihood of observed next tokens.

> **Definition so far.** *An autoregressive language model is an assignment of a next-token distribution to every prefix.*

**What is still missing.** Everything computational. What kind of machine turns a prefix into a probability vector, if not a count table? What does "make the observed token likely" mean as an operation on that machine? We asserted that a scalar loss gets minimized — minimized *how*, and with respect to *what*?

**Where Chapter 1 goes.** The calculus layer: the three-step bridge that every neural language model crosses on every call — real-valued scores to probabilities (softmax), probabilities to a scalar loss (cross-entropy), and the loss to parameter motion (gradients). By the end of the chapter, "make the observed token likely" will be a specific smooth map with a specific derivative, and the Gibbs variational principle behind softmax will have been proved once — to be recognized twice more before the book ends.

## 1. Tensors, Autograd, And Probability

**Why this chapter.** Every loss curve you have ever watched, every `logits` array in an API response, every gradient update inside a training job runs across one three-step bridge: unconstrained scores become probabilities, probabilities become a scalar loss, and the loss becomes parameter motion. This chapter builds the bridge once, carefully, with proofs — because all thirteen chapters cross it and none will re-derive it.

A tensor is a rectangular array together with a shape: a point of a finite-dimensional space $\mathbb{R}^{d_1\times\cdots\times d_k}$ whose axes carry semantic roles. A differentiable program is a composition of smooth (or piecewise-smooth) maps between such spaces, for which automatic differentiation computes derivatives. A probabilistic classifier is a differentiable program whose output is interpreted as a point of a probability simplex. A language model is a probabilistic classifier repeated over token positions with shared parameters and causal context.

**Definition 1.1 (parameter space).** The parameter space of the toy model is the product of the spaces of its weight tensors,

$$
\Theta \;=\; \underbrace{\mathbb{R}^{V\times C}}_{W_E}
\times \underbrace{\mathbb{R}^{T_{\max}\times C}}_{W_P}
\times \prod_{l=1}^{L}\Big(
\underbrace{\mathbb{R}^{C\times 3C}}_{W^{(l)}_{QKV}}\times
\underbrace{\mathbb{R}^{C\times C}}_{W^{(l)}_{O}}\times
\underbrace{\mathbb{R}^{C\times 4C}\times\mathbb{R}^{4C\times C}}_{\text{MLP}}\times
\underbrace{\mathbb{R}^{C}\times\mathbb{R}^{C}\times\cdots}_{\text{LN } \gamma,\beta,\ \text{biases}}
\Big)
\;\cong\; \mathbb{R}^{P},
$$

where $P$ is the total parameter count (`count_parameters(model)`). A parameter $\theta\in\Theta$ is one point of this space; training is a trajectory in $\Theta$. The model with parameters attached is a map

$$
F : \Theta \times \mathcal{V}^{B\times T} \longrightarrow \mathbb{R}^{B\times T\times V},
\qquad (\theta, \texttt{idx}) \longmapsto \texttt{logits},
$$

smooth in $\theta$ for each fixed integer input.

The common LLM shapes:

| Symbol | Meaning | Typical tensor shape |
| --- | --- | --- |
| `B` | batch size | number of independent windows processed together |
| `T` | sequence length or context length | number of token positions per window |
| `C` | channel or embedding dimension | vector width inside the model |
| `V` | vocabulary size | number of possible token IDs |
| `H` | number of attention heads | parallel attention subspaces |
| `D` | per-head dimension | usually `C / H` |

If `idx` has shape `(B, T)`, it contains integer token IDs. If `x = embedding(idx)`, then `x` has shape `(B, T, C)`. If the model returns logits for next-token prediction, logits have shape `(B, T, V)`.

### Empirical risk as the scalar objective

Autograd needs a scalar objective, i.e. a map $L:\Theta\to\mathbb{R}$. In supervised learning, the scalar is an empirical average of per-example losses. For language modeling with a batch of windows,

$$
L(\theta)
= \frac{1}{BT}\sum_{b=1}^{B}\sum_{t=1}^{T}
\ell\big(z_\theta(b,t),\, y_{b,t}\big),
$$

where $z_\theta(b,t) \in \mathbb{R}^V$ is the logit vector at one token position, $y_{b,t}\in\mathcal{V}$ is the target token ID, and $\ell:\mathbb{R}^V\times\mathcal{V}\to\mathbb{R}_{\ge 0}$ is the cross-entropy loss

$$
\ell(z,y) = -\log \operatorname{softmax}(z)_y,
$$

studied in detail below (Definition 1.7, Proposition 1.8). The toy function `cross_entropy_from_logits` implements this directly for logits with a final vocabulary axis. The model class `TinyGPT` uses PyTorch's optimized `F.cross_entropy`, but the mathematical object is the same.

### Autograd's role, and gradient descent as a discretized flow

The training objective is a composition

$$
\Theta \xrightarrow{\;F(\cdot,\,\texttt{idx})\;} \mathbb{R}^{B\times T\times V} \xrightarrow{\;\text{cross-entropy}\;} \mathbb{R},
\qquad \theta \mapsto z_\theta \mapsto L(\theta).
$$

Backpropagation computes $\nabla_\theta L \in \Theta$ (identifying $\Theta\cong\mathbb{R}^P$ with its dual via the standard inner product) by the chain rule applied mechanically through the computation graph. The optimizer then chooses an update rule. Plain gradient descent,

$$
\theta_{k+1} = \theta_k - \eta \nabla_\theta L(\theta_k), \qquad \eta>0,
$$

is precisely the explicit Euler discretization, with step $\eta$, of the *gradient flow*

$$
\dot\theta(s) = -\nabla_\theta L(\theta(s)),
$$

the ODE along which $L$ decreases at the fastest instantaneous rate: $\frac{d}{ds}L(\theta(s)) = -\lVert\nabla L\rVert^2 \le 0$. Stochastic gradient descent replaces $\nabla L$ by a noisy unbiased estimate from a minibatch (Proposition 6.2); AdamW additionally preconditions the step coordinatewise (Algorithm 6.3). None of these change the objective — they change the trajectory through $\Theta$.

### Code path

The code cell below uses a vector `w` and a scalar `loss = mean(w_i^2)` because it isolates the mechanism. Later, `TinyGPT.forward` creates a scalar loss from `(B, T, V)` logits and `(B, T)` targets. In both cases, `.backward()` applies the chain rule to the tensor operations that produced the scalar.

> **Mathematician's lens.** Autograd does not eliminate derivatives. It represents a large composed function and applies the chain rule mechanically. You should still know what scalar function is being differentiated, and on which space its gradient lives.

> **ML practitioner's warning.** If the scalar loss averages over the wrong elements, uses misaligned labels, or includes tokens that should be masked out, autograd will faithfully optimize the wrong objective.

### Self-checks

1. Why must the training objective be scalar before `.backward()`?
2. If `logits` has shape `(B, T, V)`, why does cross-entropy treat the final axis differently from the first two?
3. In the empirical risk formula, what changes when prompt tokens are masked during supervised fine-tuning?
4. (*) Verify the identity $\frac{d}{ds}L(\theta(s)) = -\lVert\nabla L(\theta(s))\rVert^2$ along the gradient flow, and explain why a large Euler step $\eta$ can break this monotonicity.


### Broadcasting And Scalar Objectives

Broadcasting is a rule for applying operations to tensors whose shapes are compatible but not identical. For example, subtracting a tensor of shape `(B, T, 1)` from a tensor of shape `(B, T, V)` subtracts one value from every vocabulary logit at the same batch-position pair. This is exactly the kind of operation used in numerically stable softmax, where the maximum logit is subtracted from every logit in the same row.

The key point is that broadcasting is syntactic convenience, not mathematical forgiveness. You still need to know which mathematical object is being copied along which axis. In attention, a causal mask of shape `(T, T)` can broadcast across batch and head dimensions to mask scores of shape `(B, H, T, T)`. That is correct only because every batch example and head obeys the same causal ordering.

The scalar objective in the code cell is

$$
L(w) = \frac{1}{3}\sum_{i=1}^{3} w_i^2.
$$

Its derivative is

$$
\frac{\partial L}{\partial w_i} = \frac{2w_i}{3}.
$$

The cell verifies that PyTorch stores this derivative in `w.grad`. This small check mirrors what happens in training: the loss is more complex, the parameters are matrices rather than a length-3 vector, but the chain rule contract is the same.

### Connection to the toy LLM

In `TinyGPT`, parameters appear in embedding tables, linear projections, layer norms, and the final head. The loss is the average cross-entropy over token positions. Calling `loss.backward()` populates gradients for all tensors that contributed to that loss. The training helper `train_steps` then calls:

```python
optimizer.zero_grad(set_to_none=True)
loss.backward()
optimizer.step()
```

This three-line pattern is the optimization heartbeat of the toy model.

> **Mathematician's lens.** The training loop repeatedly evaluates a scalar-valued function on a high-dimensional parameter space and follows local derivative information.

> **ML practitioner's warning.** A nonzero gradient does not guarantee useful learning. Gradients can point in noisy directions, explode, vanish, or optimize a flawed data pipeline.


In [ ]:
w = torch.tensor([1.0, -2.0, 3.0], requires_grad=True)
loss = (w ** 2).mean()
loss.backward()

# For mean(w_i^2), the derivative is 2*w_i / 3.
expected_grad = 2 * w.detach() / w.numel()
assert torch.allclose(w.grad, expected_grad)
loss.item(), w.grad


### Softmax: Scores To Probabilities

**Definition 1.2 (softmax).** Softmax is the map

$$
\sigma: \mathbb{R}^V \longrightarrow \mathring{\Delta}^{V-1},
\qquad
\sigma(z)_i
= \frac{\exp(z_i)}{\sum_{j=0}^{V-1}\exp(z_j)}.
$$

The numerator makes every component strictly positive; the denominator normalizes the sum to $1$, so the image indeed lies in the open simplex.

A vector $z\in\mathbb{R}^V$ of unconstrained real scores is called a *logit vector*. Softmax is the bridge from the linear-algebraic world (where the model computes) to the probabilistic world (where the loss and the sampling live).

**Proposition 1.3 (shift invariance; softmax as a diffeomorphism).** For any $c\in\mathbb{R}$,

$$
\sigma(z+c\mathbf{1}) = \sigma(z),
$$

and $\sigma$ descends to a diffeomorphism from the quotient $\mathbb{R}^V/\mathbb{R}\mathbf{1}$ onto $\mathring{\Delta}^{V-1}$.

*Proof.* Invariance is a one-line computation:

$$
\sigma(z+c\mathbf{1})_i
= \frac{e^{z_i+c}}{\sum_j e^{z_j+c}}
= \frac{e^c\, e^{z_i}}{e^c \sum_j e^{z_j}}
= \sigma(z)_i.
$$

For the second claim, given $p\in\mathring{\Delta}^{V-1}$ set $z_i = \log p_i$; then $\sigma(z)=p$, so $\sigma$ is onto. If $\sigma(z)=\sigma(z')$ then $z_i - z'_i = \log\frac{\sum_j e^{z_j}}{\sum_j e^{z'_j}}$ is independent of $i$, so $z-z' \in \mathbb{R}\mathbf{1}$: the fibers of $\sigma$ are exactly the lines $z+\mathbb{R}\mathbf{1}$. Smoothness of $\sigma$ and of the inverse $p\mapsto (\log p_i)_i + \mathbb{R}\mathbf{1}$ gives the diffeomorphism. $\blacksquare$

Two consequences. First, logits are identifiable only up to an additive constant: the model's "score" for a token has no absolute meaning, only logit *differences* do (they are log odds ratios). Second, the invariance is what the numerically stable implementation exploits, choosing $c=-\max_j z_j$:

$$
\sigma(z)_i
= \frac{\exp(z_i - m)}{\sum_j \exp(z_j - m)}, \qquad m = \max_j z_j,
$$

so the largest exponent is $e^0=1$ and overflow is avoided.

### The log-partition function

**Definition 1.4 (log-sum-exp / log-partition function).**

$$
A: \mathbb{R}^V \to \mathbb{R},
\qquad
A(z) = \log \sum_{j=0}^{V-1} e^{z_j}.
$$

In statistical mechanics $Z=\sum_j e^{z_j}$ is the partition function and $A = \log Z$; with energies $\varepsilon_i = -z_i$ and inverse temperature $\beta = 1$, $-A$ is the (dimensionless) free energy. The stable evaluation uses the same shift as softmax:

$$
A(z) = m + \log \sum_j e^{z_j-m}, \qquad m=\max_j z_j,
$$

which is what `torch.logsumexp` implements.

**Proposition 1.5 (softmax is the gradient of the log-partition function).** $A$ is smooth and convex, with

$$
\nabla A(z) = \sigma(z),
\qquad
\nabla^2 A(z) = \operatorname{diag}(p) - pp^{\top}, \quad p = \sigma(z),
$$

and $\nabla^2 A(z) \succeq 0$. In particular the Jacobian of softmax is

$$
\frac{\partial \sigma(z)_i}{\partial z_k}
= p_i(\delta_{ik} - p_k).
$$

*Proof.* Differentiate directly:

$$
\frac{\partial A}{\partial z_i} = \frac{e^{z_i}}{\sum_j e^{z_j}} = p_i = \sigma(z)_i .
$$

For the second derivative, apply the quotient rule to $p_i = e^{z_i}/Z$ with $Z=\sum_j e^{z_j}$, $\partial Z/\partial z_k = e^{z_k}$:

$$
\frac{\partial p_i}{\partial z_k}
= \frac{\delta_{ik}e^{z_i} Z - e^{z_i} e^{z_k}}{Z^2}
= p_i\delta_{ik} - p_i p_k
= p_i(\delta_{ik} - p_k),
$$

which is the $(i,k)$ entry of $\operatorname{diag}(p)-pp^\top$. Positive semidefiniteness: for $u\in\mathbb{R}^V$,

$$
u^\top\big(\operatorname{diag}(p)-pp^\top\big)u
= \sum_i p_i u_i^2 - \Big(\sum_i p_i u_i\Big)^2
= \operatorname{Var}_{i\sim p}(u_i) \;\ge\; 0,
$$

the variance of the function $u$ under the distribution $p$. Convexity of $A$ follows. $\blacksquare$

The Hessian degenerates exactly along $\mathbf{1}$ (constants have zero variance), which is Proposition 1.3 seen infinitesimally. The Jacobian's interpretation: increasing one logit raises its own probability and must lower mass elsewhere, because probabilities are coupled by normalization. The rank-one term $-pp^\top$ is that coupling.

### The Gibbs variational principle

The deepest characterization of softmax is variational, and it will recur verbatim in attention (Proposition 4.4) and in preference alignment (Proposition 11.1).

**Theorem 1.6 (maximum-entropy / Gibbs).** Fix scores $z\in\mathbb{R}^V$ and a temperature $\tau>0$. The problem

$$
\max_{p\,\in\,\Delta^{V-1}}\;
\Big\{ \langle p, z\rangle + \tau H(p) \Big\},
\qquad
H(p) = -\sum_i p_i\log p_i,
$$

has the unique solution $p^\ast = \sigma(z/\tau)$, with optimal value $\tau A(z/\tau)$.

*Proof.* The objective is strictly concave on the simplex ($\langle p,z\rangle$ is linear; $H$ is strictly concave, since its Hessian $\operatorname{diag}(-1/p_i)$ is negative definite on the interior), so a critical point of the Lagrangian in the interior is the unique maximizer. Form the Lagrangian with multiplier $\mu$ for the constraint $\sum_i p_i = 1$ (the positivity constraints are inactive at an interior point):

$$
\mathcal{L}(p,\mu) = \sum_i p_i z_i - \tau \sum_i p_i \log p_i + \mu\Big(\sum_i p_i - 1\Big).
$$

Stationarity in $p_i$:

$$
z_i - \tau(\log p_i + 1) + \mu = 0
\quad\Longrightarrow\quad
p_i = \exp\!\Big(\frac{z_i}{\tau}\Big)\,\exp\!\Big(\frac{\mu-\tau}{\tau}\Big).
$$

The second factor is a constant fixed by normalization, giving $p_i = e^{z_i/\tau}/\sum_j e^{z_j/\tau} = \sigma(z/\tau)_i$. Substituting back:

$$
\langle p^\ast, z\rangle + \tau H(p^\ast)
= \sum_i p^\ast_i z_i - \tau\sum_i p^\ast_i\Big(\frac{z_i}{\tau} - A(z/\tau)\Big)
= \tau A(z/\tau). \qquad\blacksquare
$$

In physical language: among all distributions, the Gibbs distribution is the one that maximizes entropy at a given expected score (equivalently, minimizes free energy $\langle p,\varepsilon\rangle - \tau H(p)$ with $\varepsilon=-z$). Softmax at temperature $\tau$ is not an arbitrary squashing function; it is the unique resolution of the competition between *score-seeking* (put all mass on the argmax) and *entropy* (spread out). As $\tau\to 0$ the score term dominates and $p^\ast$ converges to the uniform distribution on the argmax set; as $\tau\to\infty$, entropy dominates and $p^\ast\to$ uniform on $\mathcal{V}$ (proved as Proposition 7.1).

### Dimensions and tensor shapes

For language modeling, logits have shape `(B, T, V)`. Softmax is applied along the vocabulary axis `dim=-1`, producing a tensor of the same shape whose slice `probs[b,t,:]` lies in $\mathring{\Delta}^{V-1}$ for every fixed `(b,t)` — a field of simplex points indexed by batch and position.

The notebook code calls `stable_softmax(logits, dim=-1)` from `src/llm_from_scratch/functional.py`. That function subtracts the row maximum, exponentiates, and divides by the row sum, exactly as in the proof of Proposition 1.3.

> **Mathematician's lens.** Softmax is the gradient map of a convex potential (Proposition 1.5) and a diffeomorphism from logit space modulo constants onto the open simplex (Proposition 1.3). It turns additive logit differences into multiplicative odds ratios.

> **ML practitioner's warning.** Softmax probabilities are not automatically calibrated probabilities. A model can assign confident probabilities for bad reasons, especially off-distribution or after overfitting.

### Self-checks

1. Prove that adding the same constant to every logit does not change softmax, and identify where this appears in `stable_softmax`.
2. If all logits are equal, what probability does each vocabulary item receive? Reconcile with Theorem 1.6 at $\tau\to\infty$.
3. Why would applying softmax over `dim=1` be wrong for a `(B, T, V)` language-model logit tensor?
4. (*) Show directly that $u^\top \nabla^2 A(z)\, u = \operatorname{Var}_{i\sim p}(u_i)$ and conclude that $A$ is strictly convex transverse to $\mathbf 1$.


In [ ]:
from llm_from_scratch.functional import stable_softmax, cross_entropy_from_logits

logits = torch.tensor([[1.0, 2.0, 3.0]])
probs = stable_softmax(logits, dim=-1)

assert probs.shape == logits.shape
assert torch.allclose(probs.sum(dim=-1), torch.ones(1))
assert torch.argmax(probs, dim=-1).item() == 2
probs


### Cross-Entropy As Negative Log-Likelihood

**Definition 1.7 (cross-entropy loss).** The per-example loss is the map

$$
\ell : \mathbb{R}^V \times \mathcal{V} \longrightarrow \mathbb{R}_{\ge 0},
\qquad
\ell(z,y) = -\log \sigma(z)_y .
$$

Substituting Definition 1.2 and using the log-partition function $A$ of Definition 1.4,

$$
\ell(z,y)
= -\log\left(\frac{e^{z_y}}{\sum_j e^{z_j}}\right)
= A(z) - z_y .
$$

This is the form stable implementations use: gather the correct logit and subtract it from a log-sum-exp.

**Proposition 1.8 (positivity and gradient).** For all $z\in\mathbb{R}^V$, $y\in\mathcal{V}$:

(i) $\ell(z,y) \ge 0$, with $\ell(z,y)\to 0$ iff $\sigma(z)_y \to 1$;

(ii) writing $p=\sigma(z)$ and $e_y$ for the one-hot vector of $y$,

$$
\nabla_z\, \ell(z,y) = p - e_y,
$$

so in coordinates $\partial\ell/\partial z_i = p_i - \delta_{iy}$;

(iii) the gradient components sum to zero: $\sum_i (p_i - \delta_{iy}) = 1 - 1 = 0$.

*Proof.* (i) $A(z) = \log\sum_j e^{z_j} \ge \log e^{z_y} = z_y$, with equality only in the limit where the other terms vanish relative to $e^{z_y}$. (ii) By Proposition 1.5, $\nabla A = \sigma$, and $\nabla_z z_y = e_y$; subtract. (iii) Immediate, since $p\in\Delta^{V-1}$. $\blacksquare$

The gradient formula deserves contemplation: it is the *residual* $p - e_y$, the difference between the predicted distribution and the observed one-hot target. The gradient is dense — every logit receives a derivative. The correct class receives $p_y - 1 \le 0$, so gradient descent increases that logit; each incorrect class $i$ receives $p_i \ge 0$ and is pushed down in proportion to the probability the model wrongly assigned it. Property (iii) says the gradient lies in the hyperplane $\mathbf{1}^\perp$: by shift invariance (Proposition 1.3) the loss cannot see, and therefore cannot generate, motion along $\mathbf{1}$.

### KL and the information view

Let $q\in\Delta^{V-1}$ be a target distribution and $p\in\mathring\Delta^{V-1}$ the model distribution. Cross-entropy between distributions is

$$
H(q,p) = -\sum_i q_i \log p_i,
$$

and it decomposes as

$$
H(q,p) = H(q) + \mathrm{KL}(q\,\|\,p),
$$

since

$$
\mathrm{KL}(q\|p)
= \sum_i q_i \log\frac{q_i}{p_i}
= \sum_i q_i\log q_i - \sum_i q_i\log p_i
= -H(q) + H(q,p).
$$

For a one-hot target $q = e_y$ we get $H(q,p) = -\log p_y = \ell(z,y)$ and $H(e_y)=0$, so minimizing the loss on that token is exactly minimizing $\mathrm{KL}(e_y\|p)$.

The population version is the statement that actually justifies maximum likelihood, and it is worth deriving once in full.

**Theorem 1.9 (maximum likelihood minimizes conditional KL; the entropy floor).** Let $q$ denote the data-generating distribution over sequences, with conditionals $q(\cdot\mid x_{<t})$. Then the population risk of the model satisfies

$$
R(\theta)
:= \mathbb{E}_{x\sim q}\Big[-\log p_\theta(x_t \mid x_{<t})\Big]
= \underbrace{\mathbb{E}_{x_{<t}}\big[H\big(q(\cdot\mid x_{<t})\big)\big]}_{\text{irreducible}}
\;+\;
\underbrace{\mathbb{E}_{x_{<t}}\Big[\mathrm{KL}\big(q(\cdot\mid x_{<t})\,\big\|\,p_\theta(\cdot\mid x_{<t})\big)\Big]}_{\ge 0,\ =0 \text{ iff } p_\theta = q}.
$$

*Proof.* Condition on the prefix and take the inner expectation over the next token $x_t \sim q(\cdot\mid x_{<t})$:

$$
\mathbb{E}_{x_t}\big[-\log p_\theta(x_t\mid x_{<t})\big]
= -\sum_i q(i\mid x_{<t})\log p_\theta(i\mid x_{<t})
= H\big(q(\cdot\mid x_{<t}),\, p_\theta(\cdot\mid x_{<t})\big),
$$

which by the decomposition above equals $H(q(\cdot\mid x_{<t})) + \mathrm{KL}(q(\cdot\mid x_{<t})\|p_\theta(\cdot\mid x_{<t}))$. Take the outer expectation over prefixes. Nonnegativity of KL is Gibbs' inequality; equality holds iff the distributions agree $q$-almost surely. $\blacksquare$

Two morals. First, the achievable minimum of the language-modeling loss is not $0$ but the *intrinsic conditional entropy of language* — an entropy floor set by the data distribution, not by the model. Second, training loss (in nats per token) is an estimate of $R(\theta)$, so improvements measure reduction of the KL term only. Chapter 6 exponentiates this quantity to define perplexity.

### Dimensions and code path

`cross_entropy_from_logits(logits, targets)` accepts logits whose last dimension is vocabulary. If logits have shape `(B, T, V)` and targets have shape `(B, T)`, it flattens them to `(B*T, V)` and `(B*T)`. The final scalar is the mean loss over token positions.

`TinyGPT.forward` uses `F.cross_entropy` with the same flattening. This is the implementation of

$$
\frac{1}{BT}\sum_{b,t}-\log p_\theta\big(y_{b,t}\mid x_{b,1:t}\big).
$$

> **Mathematician's lens.** Cross-entropy is the composition of the smooth convex function $A(z)-z_y$ with the model map; its gradient is the residual $p - e_y$; and in expectation it is intrinsic entropy plus a KL divergence (Theorem 1.9). Everything the optimizer sees follows from these three facts.

> **ML practitioner's warning.** Cross-entropy rewards probability assigned to observed tokens. If the data contain leakage, duplication, bad formatting, or harmful targets, the loss will optimize those patterns too.

### Self-checks

1. Derive $\nabla_z \ell = p - e_y$ from $\ell=A(z)-z_y$ using Proposition 1.5.
2. Why does the gradient sum to zero across logits, and how does this relate to shift invariance?
3. In a language-model batch, what does one row of the flattened `(B*T, V)` tensor correspond to?
4. (*) In Theorem 1.9, suppose the data distribution is deterministic ($q(\cdot\mid x_{<t})$ one-hot for every prefix). What is the entropy floor, and what loss value certifies a perfect model?


### Logistic Regression As The One-Logit Warmup

Binary logistic regression is the smallest version of the same idea. It computes one real-valued score

$$
z = w\cdot x + b,
$$

then maps it to a probability with the sigmoid

$$
\sigma(z) = \frac{1}{1 + e^{-z}}.
$$

A vocabulary softmax generalizes this from one logit to a vector of $V$ logits. The binary model asks, "how much evidence is there for class 1?" The language model asks, "how much evidence is there for each possible next token?"

This bridge is useful because it separates probability from action. A probability distribution is not yet a decision. In binary classification, one still chooses a threshold. In generation, one still chooses a decoding policy. In both cases, the model's probabilities become behavior only after an extra rule is applied.

Source note: this follows the CS124 logistic-regression lab's logit/sigmoid framing and its warning that probability and decision thresholds are different objects: `../data/cs124/labs/Lab2_LogisticRegression.md`.


In [ ]:
logits = torch.tensor([[1.0, 2.0, 3.0]], requires_grad=True)
target = torch.tensor([2])
loss = cross_entropy_from_logits(logits, target)
loss.backward()

with torch.no_grad():
    p = stable_softmax(logits, dim=-1)
    expected_grad = p.clone()
    expected_grad[0, target.item()] -= 1.0

assert torch.allclose(logits.grad, expected_grad, atol=1e-6)
loss.item(), logits.grad


**Worked Example 1.1 (softmax and its stable form, digit by digit).** Let the model emit logits over the running vocabulary $(\texttt{' '}, \texttt{a}, \texttt{c}, \texttt{t})$:

$$
z = (2,\ 0,\ -1,\ 1).
$$

Exponentiate: $e^2 = 7.3891$, $e^0 = 1$, $e^{-1} = 0.3679$, $e^{1} = 2.7183$. The partition function (Definition 1.4) is

$$
Z = 7.3891 + 1 + 0.3679 + 2.7183 = 11.4752,
\qquad
A(z) = \ln Z = 2.4402,
$$

and dividing gives the probability vector

$$
\sigma(z) = (0.6439,\ 0.0871,\ 0.0321,\ 0.2369) \in \mathring{\Delta}^{3},
$$

which sums to $1$ exactly. The stable form (Proposition 1.3) subtracts $m = 2$ first: the shifted logits $(0,-2,-3,-1)$ produce the *same* vector, but now the largest exponent is $e^0 = 1$.

**Worked Example 1.2 (cross-entropy and its gradient).** Suppose the observed next token is $\texttt{t}$ (class $y = 3$). By Definition 1.7,

$$
\ell(z, 3) = A(z) - z_3 = 2.4402 - 1 = 1.4402 \text{ nats},
$$

and by Proposition 1.8 the gradient is the residual $p - e_3$:

$$
\nabla_z \ell = (0.6439,\ 0.0871,\ 0.0321,\ \mathbf{-0.7631}),
$$

whose components sum to $0.6439+0.0871+0.0321-0.7631 = 0$ exactly — the conservation law of shift invariance, visible in the digits. Read the vector as forces: the space token, wrongly favored with probability $0.64$, receives the largest downward push; the true token $\texttt{t}$ is pushed up by $1 - 0.2369 = 0.7631$; the two tokens the model barely considered are barely touched.

In [ ]:
import torch
from llm_from_scratch.functional import stable_softmax, cross_entropy_from_logits

# Worked Examples 1.1 / 1.2 — verify against the hand arithmetic
z = torch.tensor([[2.0, 0.0, -1.0, 1.0]], requires_grad=True)   # logits for (' ', 'a', 'c', 't')

p = stable_softmax(z, dim=-1)
expected_p = torch.tensor([[0.6439, 0.0871, 0.0321, 0.2369]])
assert torch.allclose(p, expected_p, atol=5e-5)
assert torch.allclose(p.sum(dim=-1), torch.ones(1))

y = torch.tensor([3])                                            # observed token: 't'
loss = cross_entropy_from_logits(z, y)
assert abs(loss.item() - 1.4402) < 5e-5                          # A(z) - z_y

loss.backward()
expected_grad = torch.tensor([[0.6439, 0.0871, 0.0321, -0.7631]])
assert torch.allclose(z.grad, expected_grad, atol=5e-5)          # gradient = p - e_y
assert abs(z.grad.sum().item()) < 1e-6                           # components sum to zero
print("p        =", [round(v, 4) for v in p[0].tolist()])
print("loss     =", round(loss.item(), 4), "nats")
print("gradient =", [round(v, 4) for v in z.grad[0].tolist()], "  (sums to 0)")

### Where We Stand

**What we now have.** The three-step calculus is in place, and each step earned a theorem. Scores become probabilities through softmax, which is not an arbitrary squashing but the unique optimum of score-plus-entropy (Theorem 1.6) and the gradient of the free energy $A$ (Proposition 1.5). Probability of the observed token becomes the loss $\ell = A(z) - z_y$, whose gradient is the residual $p - e_y$ (Proposition 1.8) — prediction minus truth, the cleanest learning signal imaginable. And in expectation, minimizing that loss is minimizing the KL divergence to the true conditionals, above an irreducible entropy floor (Theorem 1.9).

> **Definition so far.** *An autoregressive language model is a smooth parameterized family $p_\theta$ of next-token distributions, trained by descending the average negative log-likelihood — equivalently, by pushing $p_\theta$ toward the data's conditionals in KL divergence.*

**What is still missing.** The sample space itself. We have been saying "token" as if tokens fell from the sky. Real data is text — strings over Unicode — and nothing so far explains how strings become the integer sequences that $p_\theta$ actually conditions on, or why that choice is consequential rather than clerical.

**Where Chapter 2 goes.** Tokenization as a modeling decision: the maps $\operatorname{enc}$ and $\operatorname{dec}$, what losslessness does and does not guarantee, and why the tokenizer silently sets the vocabulary size $V$, the sequence length $T$, the entropy floor, and the cost of everything downstream.

## 2. Tokenization: Text To Integer IDs

### Mathematical object

Neural networks in this curriculum do not operate on strings directly. They operate on finite sequences of integers. Let $\mathcal{A}$ be the alphabet of raw text (Unicode characters, or bytes) and $\mathcal{A}^*$ the set of finite strings over it.

**Definition 2.1 (tokenizer).** A tokenizer over vocabulary $\mathcal{V}$ is a pair of maps

$$
\operatorname{enc}: \mathcal{A}^* \longrightarrow \mathcal{V}^*,
\qquad
\operatorname{dec}: \mathcal{V}^* \longrightarrow \mathcal{A}^*,
$$

and it is called *lossless* if

$$
\operatorname{dec} \circ \operatorname{enc} = \operatorname{id}_{\mathcal{A}^*}.
$$

**Remark 2.2 (asymmetry of the round trip).** Losslessness makes $\operatorname{enc}$ a *section* of $\operatorname{dec}$: it is injective, and $\operatorname{dec}$ is surjective. The opposite composition generally fails, $\operatorname{enc}\circ\operatorname{dec} \ne \operatorname{id}_{\mathcal{V}^*}$: many token sequences decode to the same string that re-encodes canonically (e.g. `["at","tention"]` and `["att","ention"]` may both decode to `"attention"`, which re-encodes to only one of them). The model, however, assigns probability to *token sequences*, not strings; the probability of a string is in principle a sum over all token sequences decoding to it, which practical systems approximate by the canonical encoding alone. This is a genuine, usually ignored, modeling gap.

For a character tokenizer, the vocabulary is the finite set of characters found in the corpus, each assigned an integer ID. For a subword tokenizer such as BPE, the vocabulary contains learned string fragments. The output is in either case a finite integer sequence.

### Dimensions and tensor shapes

After tokenization, a text sample becomes a 1D sequence of token IDs:

```python
tokens.shape == (N,)
```

Batching later slices this into windows:

```python
x.shape == (B, T)
y.shape == (B, T)
```

The vocabulary size `V` determines the number of rows in the token embedding table and the number of logits emitted at each position.

### Statistical role

Tokenization defines the sample space of the model's categorical predictions. If the vocabulary has size $V$, then every next-token distribution is a point of $\Delta^{V-1}$. A different tokenizer changes both the labels and the sequence length: the same raw text can become many character tokens or fewer subword tokens.

This matters because the model does not optimize probability of raw Unicode strings directly. It optimizes probability of token sequences (Remark 2.2). If two tokenizers represent the same string with different token sequences, they create different conditional prediction problems, different entropy floors (Theorem 1.9), and different per-token losses — which is why per-token loss is not comparable across tokenizers.

### Character versus subword tokenization

A character tokenizer is transparent:

```text
"cat" -> [id("c"), id("a"), id("t")]
```

The advantage is conceptual simplicity. The disadvantage is longer sequences. A subword tokenizer learns frequent chunks:

```text
"attention" -> [id("att"), id("ention")]  # illustrative, not exact
```

The advantage is shorter sequences and reusable pieces for rare words. The disadvantage is that the learned vocabulary and merge rules become part of the data pipeline.

### Code path

The repo exposes both teaching forms:

- `CharTokenizer.from_text(text)` builds a simple character vocabulary.
- `char_tok.encode(...)` maps text to integer IDs.
- `char_tok.decode(...)` maps IDs back to text.
- `train_bpe_tokenizer([text], vocab_size=80)` trains a tiny BPE tokenizer for demonstration.

The next code cell checks the lossless property $\operatorname{dec}\circ\operatorname{enc}=\operatorname{id}$ on a text prefix for the character tokenizer and compares character-token length with BPE-token length on the same prefix.

> **Mathematician's lens.** Tokenization chooses a finite alphabet and represents strings as sequences over it; the language model is then a probability model over this discrete sequence space. The choice of representation changes the measure being estimated, not just the coordinates.

> **ML practitioner's warning.** Tokenization is not harmless preprocessing. It changes sequence length, vocabulary size, rare-word behavior, special-token handling, and the cost of attention through `T`.

### Self-checks

1. If a tokenizer doubles the average number of tokens per document, what happens to the size of a full attention score matrix for the same raw document?
2. Why is token ID order not meaningful as a numeric order?
3. Which model tensors change shape when `vocab_size` changes?
4. (*) Exhibit two distinct token sequences under a BPE vocabulary that decode to the same string, and explain why practical systems score only one of them.


In [ ]:
from llm_from_scratch.tokenization import CharTokenizer, train_bpe_tokenizer

text = (PROJECT_ROOT / "data" / "tiny_corpus.txt").read_text(encoding="utf-8")
char_tok = CharTokenizer.from_text(text)
ids = char_tok.encode(text[:80])
round_trip = char_tok.decode(ids)

bpe_tok = train_bpe_tokenizer([text], vocab_size=80)
bpe_ids = bpe_tok.encode(text[:80]).ids

assert round_trip == text[:80]
char_tok.vocab_size, len(ids), len(bpe_ids), ids[:12], bpe_ids[:12]


**Worked Example 2.1 (the running vocabulary).** Every worked example in this book uses one tiny alphabet, built by the repo's own `CharTokenizer` from the string `"a cat"`. The constructor sorts the distinct characters, so

$$
\operatorname{enc}: \quad \texttt{' '} \mapsto 0, \quad \texttt{a} \mapsto 1, \quad \texttt{c} \mapsto 2, \quad \texttt{t} \mapsto 3,
\qquad V = 4 .
$$

The word `"cat"` becomes the ID sequence

$$
\operatorname{enc}(\texttt{"cat"}) = (2,\ 1,\ 3) \in \mathcal{V}^3,
$$

and the lossless property of Definition 2.1 is checkable by eye: $\operatorname{dec}(2,1,3) = \texttt{"cat"}$. Note what the integers do *not* carry: $\texttt{t}=3$ is not "more" than $\texttt{a}=1$ in any sense the model may use. The next chapter replaces each ID with a learned vector precisely because the integer labels are semantically arbitrary.

In [ ]:
from llm_from_scratch.tokenization import CharTokenizer

# Worked Example 2.1 — the running vocabulary used by every worked example
tok = CharTokenizer.from_text("a cat")

assert tok.stoi == {" ": 0, "a": 1, "c": 2, "t": 3}
assert tok.vocab_size == 4

ids = tok.encode("cat")
assert ids == [2, 1, 3]
assert tok.decode(ids) == "cat"                       # dec ∘ enc = id  (Definition 2.1)
print("vocab      :", tok.stoi)
print("enc('cat') :", ids)
print("round trip :", repr(tok.decode(ids)))

### What Tokenization Does And Does Not Decide

Tokenization decides the model's discrete alphabet. It does not decide the architecture. A transformer can use character tokens, BPE tokens, byte tokens, or other discrete units. The rest of the model only sees integer IDs.

However, tokenization changes three mathematical quantities that matter throughout the notebook:

1. **Vocabulary size $V$.** This sets the dimension of the output simplex and the width of the final logit vector.
2. **Sequence length $T$.** This sets the number of positions in a context window and the size of the attention matrix.
3. **Empirical target distribution.** Token frequencies and conditional patterns depend on the tokenizer.

The connection to attention is especially direct. Vanilla causal self-attention builds a score matrix with shape `(T, T)` per head per example. If tokenization makes sequences longer, the quadratic term grows quickly:

$$
\text{scores per head} = T^2.
$$

So a tokenizer can affect both statistical efficiency and hardware cost.

### Connection to the toy LLM

`ModelConfig(vocab_size=...)` fixes the size of:

- `TinyGPT.token_embedding`, shape `(V, C)`,
- `TinyGPT.lm_head`, shape `(C, V)` conceptually, though PyTorch stores linear weights as `(V, C)`,
- cross-entropy logits, final axis length `V`.

The context length appears as `block_size`. It fixes:

- the maximum input window length,
- the position embedding table shape `(T_max, C)`,
- the registered causal mask shape `(1, 1, T_max, T_max)` in `CausalSelfAttention`.

> **Mathematician's lens.** Tokenization is a representation choice that changes the coordinate system of the discrete sample space before any neural network map is applied.

> **ML practitioner's warning.** Comparing models with different tokenizers can be misleading if you compare only per-token loss. A token is not a stable unit across tokenizers.


### CS124 Tokenization Lens: Types, Instances, Unknowns, And BPE

A useful CS124 distinction is:

| Term | Meaning in this notebook |
| --- | --- |
| Type | A vocabulary entry, such as a character, byte, word, or subword unit. |
| Instance | One occurrence in running text. |
| Token ID | The integer emitted by a tokenizer for one type occurrence. |
| Vocabulary size $V$ | The number of possible token IDs the model can predict. |

This matters because word vocabularies do not close neatly. As the corpus grows, more word types appear. A fixed word-level vocabulary therefore creates an unknown-word problem: a test string can contain a word type absent from training.

Subword tokenization is the compromise. A BPE-style tokenizer has two conceptually separate parts:

1. **Learner:** start from small units, count adjacent pairs in a corpus, and iteratively merge frequent pairs into longer units.
2. **Encoder:** apply the learned merge rules to segment new text into known token IDs.

Byte-level BPE pushes the base alphabet all the way down to bytes. Because there are only 256 possible byte values, every Unicode string can be represented without an unknown token. Learned merges then recover common multi-byte characters, morphemes, words, or word fragments when the corpus supports them.

The underappreciated consequence is that tokenization changes both statistics and compute. It changes the empirical distribution over labels, the sequence length $T$, and the attention cost proportional to $T^2$. A tokenizer is therefore not a preprocessing detail. It is part of the modeling contract.

Source note: this integrates CS124/Speech and Language Processing material on types, instances, vocabulary growth, unknown words, BPE learner/encoder structure, and byte-level BPE from `../data/cs124/slp3_chapters/2.pdf` and `../data/cs124/lectures/tokens_jan26.pdf`.


### Classical String Algorithms Before Token IDs

A transformer sees integer IDs, but the input begins as strings. CS124 puts several older NLP tools before neural modeling because they still define the contract that produces those IDs.

| Tool | Question it answers | Why it still matters |
| --- | --- | --- |
| Unicode and UTF-8 | what characters or bytes does this string contain? | different encodings change the raw units available to tokenization |
| Regular expressions | what surface patterns should be matched or segmented? | dataset cleaning, token boundaries, and filters often start here |
| Edit distance | how many insertions, deletions, or substitutions separate two strings? | spelling variation and approximate matching are not solved by token IDs alone |
| Noisy-channel correction | which intended word likely produced this observed typo? | probability can combine a source model with an error model |
| BPE | which adjacent units should be merged into reusable subword types? | subword vocabularies are learned from corpus statistics before LM training |

This matters for LLMs because tokenization is not semantic understanding. BPE can make frequent strings cheap to represent, and byte-level BPE can represent arbitrary Unicode text, but the model still receives a sequence of IDs whose boundaries were chosen by an algorithm outside the transformer.

A practical rule: when a model behaves oddly around whitespace, casing, spelling, code, emoji, or non-English text, inspect the string-to-token layer before blaming attention.

Source note: this section integrates the CS124 words/tokens/edit-distance material from `../data/cs124/slp3_chapters/2.pdf`, `../data/cs124/slp3_chapters/D.pdf`, `../data/cs124/lectures/med25.pdf`, and `../data/cs124/pa1-regular-expressions/`.


### Where We Stand

**What we now have.** The sample space is now honest: a tokenizer pair $(\operatorname{enc}, \operatorname{dec})$ turns strings into integer sequences and back, the vocabulary is fixed and finite, and Worked Example 2.1 pinned down the four-token alphabet every worked example uses. We also flagged a real subtlety most treatments skip: the model assigns probability to token sequences, not strings, and several token sequences can decode to one string (Remark 2.2).

> **Definition so far.** *An autoregressive language model is a smooth parameterized family of next-token distributions over a tokenizer-defined sample space $\mathcal{V} = \{0, \ldots, V-1\}$.*

**What is still missing.** Geometry, and supervision. The integers in $\mathcal{V}$ are bare labels — token 3 is not "more" than token 1, and no arithmetic on IDs is meaningful, so nothing can yet *compute* with them. And we know one sequence should yield many training terms, but we have not built the machinery that extracts input–target pairs from raw token streams.

**Where Chapter 3 goes.** Two constructions that give the pipeline its raw materials: the embedding map $E: \mathcal{V} \to \mathbb{R}^C$ that gives tokens coordinates (so that linear algebra can act on them), and label shifting, which turns one window of $T{+}1$ tokens into $T$ supervised examples evaluated in a single pass.

## 3. Embeddings And Next-Token Language Modeling

### Mathematical object

Token IDs are labels from a finite set. The integer `17` does not mean "larger" than token `8` in any semantic sense. The model therefore learns an embedding.

**Definition 3.1 (token embedding).** A token embedding is a map

$$
E: \mathcal{V} \longrightarrow \mathbb{R}^C,
$$

represented by a matrix $W_E \in \mathbb{R}^{V\times C}$ via $E(i) = W_E[i,:]$, the $i$-th row. It extends coordinatewise to sequences,

$$
E^{\otimes T}: \mathcal{V}^T \longrightarrow \mathbb{R}^{T\times C},
\qquad
\big(E^{\otimes T}(x)\big)_{t,:} = E(x_t).
$$

Equivalently, $E$ factors through the one-hot inclusion: let $\iota:\mathcal{V}\hookrightarrow \{0,1\}^V\subset\mathbb{R}^V$ send $i$ to the standard basis vector $e_i$; then

$$
E = (\,\cdot\, W_E)\circ \iota, \qquad e_i W_E = W_E[i,:].
$$

The lookup implementation is faster than forming one-hot vectors explicitly, but the mathematical object is a linear map $\mathbb{R}^V\to\mathbb{R}^C$ restricted to the vertices of the simplex — a learned coordinate system for a finite set.

**Definition 3.2 (input map of the toy model).** With a learned position table $W_P\in\mathbb{R}^{T_{\max}\times C}$, the input map of `TinyGPT` is

$$
\mathrm{In}: \mathcal{V}^T \longrightarrow \mathbb{R}^{T\times C},
\qquad
\mathrm{In}(x)_{t,:} = W_E[x_t,:] + W_P[t,:],
$$

for $T \le T_{\max}$. Token identity and position identity enter the same $C$-dimensional residual stream additively.

**Remark 3.3 (why $W_P$ must exist: symmetry breaking).** Every subsequent operation in the transformer except attention acts positionwise, and attention itself — without positional information — is *permutation-equivariant*: it commutes with every reordering of the sequence (Theorem 4.5, proved in the next chapter). A composition of permutation-equivariant and positionwise maps cannot distinguish `"dog bites man"` from `"man bites dog"`. The position table $W_P$ is an explicit symmetry-breaking term, exactly analogous to an external field breaking a degeneracy in a physical system: it is the *only* ingredient of the toy model through which the conditional distribution can depend on word order rather than word multiset.

### Dimensions and tensor shapes

For token IDs `ids` with shape `(B, T)`, PyTorch `nn.Embedding(V, C)` returns

```python
x = embedding(ids)
x.shape == (B, T, C)
```

The batched input map is $\mathrm{In}^{\otimes B}:\mathcal{V}^{B\times T}\to\mathbb{R}^{B\times T\times C}$; the position embedding tensor of shape `(T, C)` broadcasts across the batch axis when added to token embeddings of shape `(B, T, C)`.

### Statistical and geometric role

Embeddings let the model learn representation geometry. Two tokens whose rows are close in $\mathbb{R}^C$ behave similarly under all subsequent linear maps — closeness is transported functorially through the network. This similarity is not imposed by the token IDs; it is learned from the training objective: if moving an embedding row helps predict future tokens in contexts where that token appears, gradient descent moves it. The embedding table is trained by the same next-token loss as every other parameter.

### Code path

In the code cell below, `torch.nn.Embedding(V, C)` illustrates the lookup. In the model, the corresponding parameters are:

```python
self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
self.position_embedding = nn.Embedding(config.block_size, config.n_embd)
```

and the forward pass begins with:

```python
positions = torch.arange(seq_len, device=idx.device)
x = self.token_embedding(idx) + self.position_embedding(positions)
```

> **Mathematician's lens.** An embedding table is a map from a finite set into a vector space — equivalently, the restriction of a linear map to one-hot vertices. It turns discrete symbols into coordinates where linear algebra can operate; the position table breaks permutation symmetry (Remark 3.3).

> **ML practitioner's warning.** Embedding geometry is learned from data and objective, not from pure meaning. Nearby vectors can reflect syntax, frequency, formatting, artifacts, or spurious correlations.

### Self-checks

1. If `V=50_000` and `C=768`, how many trainable scalar parameters are in the token embedding table?
2. Why is an embedding lookup equivalent to one-hot matrix multiplication?
3. In `TinyGPT`, why can the position embedding of shape `(T, C)` be added to a token embedding tensor of shape `(B, T, C)`?
4. (*) Suppose $W_P = 0$ and the model has a single attention layer. Using Remark 3.3, describe exactly which pairs of input sequences must receive identical next-token distributions at the final position.


### Static Embeddings Versus Contextual Representations

The embedding table begins as a type-level lookup table. Token ID $i$ always selects the same row $W_E[i,:]$ before the transformer has seen context. This is analogous to a static embedding dictionary: one type, one vector.

After transformer layers, the vectors are no longer static type embeddings. The residual stream at position $t$ has been changed by position information, attention to earlier tokens, MLP transformations, layer normalization, and residual updates. It is better to think of the final hidden vector as a contextual token-instance representation:

$$
\text{same token ID} + \text{different context} \Rightarrow \text{different hidden vector}.
$$

This distinction prevents a common confusion. The row in the embedding matrix is a learned starting representation for a token type. The hidden state produced after attention is a representation of a token instance in a particular context.

Similarity in these spaces should also be interpreted carefully. Cosine similarity is a normalized dot product; it measures geometric angle, not truth, entailment, or probability. It is useful because it reduces raw vector-length effects, but it is still only one measurement on a learned representation space.

Source note: this follows the CS124 vector-semantics and contextual-embedding materials, especially the embedding-matrix shape view in SLP3 Chapter 6 and the static-vs-contextual distinction in SLP3 Chapter 10 and `../data/cs124/lectures/ContextualEmbeddings25.pdf`.


In [ ]:
B, T, V, C = 2, 4, 10, 6
ids = torch.tensor([[1, 2, 3, 4], [4, 3, 2, 1]])
embedding = torch.nn.Embedding(V, C)
x = embedding(ids)

assert ids.shape == (B, T)
assert x.shape == (B, T, C)
x.shape


**Worked Example 3.1 (embedding `"cat"`, with the one-hot factorization visible).** Work at width $C = 2$ so every tensor fits on the page. Fix the embedding table and position table

$$
W_E = \begin{pmatrix} 0 & 0.1 \\ 0.5 & -0.5 \\ 1 & 0.5 \\ -0.5 & 1 \end{pmatrix}
\begin{matrix} \leftarrow \texttt{' '} \\ \leftarrow \texttt{a} \\ \leftarrow \texttt{c} \\ \leftarrow \texttt{t} \end{matrix}
\qquad
W_P = \begin{pmatrix} 0.1 & 0 \\ 0 & 0.1 \\ -0.1 & 0 \end{pmatrix}
\begin{matrix} \leftarrow t=1 \\ \leftarrow t=2 \\ \leftarrow t=3 \end{matrix}
$$

The first token of `"cat"` is $\texttt{c} = 2$, and its lookup is literally the one-hot product of Definition 3.1:

$$
e_2 W_E = (0,\ 0,\ 1,\ 0)\,W_E
= 0\cdot(0, 0.1) + 0\cdot(0.5, -0.5) + 1\cdot(1, 0.5) + 0\cdot(-0.5, 1)
= (1,\ 0.5).
$$

Stacking the three lookups for $(2,1,3)$ and adding $W_P$ row-by-row gives the initial residual stream of Definition 3.2:

$$
X = \mathrm{In}(\texttt{"cat"}) =
\begin{pmatrix} 1 & 0.5 \\ 0.5 & -0.5 \\ -0.5 & 1 \end{pmatrix}
+
\begin{pmatrix} 0.1 & 0 \\ 0 & 0.1 \\ -0.1 & 0 \end{pmatrix}
=
\begin{pmatrix} 1.1 & 0.5 \\ 0.5 & -0.4 \\ -0.6 & 1 \end{pmatrix}
\in \mathbb{R}^{3\times 2}.
$$

Without the $W_P$ term, the two occurrences of any repeated token would enter the model as *identical* rows — Remark 3.3's symmetry, concretely. This matrix $X$ is the input to Worked Example 4.1, where it flows through one attention head.

In [ ]:
import torch

# Worked Example 3.1 — embedding "cat" = (2, 1, 3) at C = 2
W_E = torch.tensor([[0.0, 0.1], [0.5, -0.5], [1.0, 0.5], [-0.5, 1.0]])   # rows: ' ', a, c, t
W_P = torch.tensor([[0.1, 0.0], [0.0, 0.1], [-0.1, 0.0]])                # rows: positions 1..3
ids = torch.tensor([2, 1, 3])                                            # "cat"

# The lookup IS one-hot matrix multiplication (Definition 3.1):
one_hot = torch.nn.functional.one_hot(ids, num_classes=4).float()
assert torch.equal(one_hot @ W_E, W_E[ids])

X = W_E[ids] + W_P                                                       # Definition 3.2
expected_X = torch.tensor([[1.1, 0.5], [0.5, -0.4], [-0.6, 1.0]])
assert torch.allclose(X, expected_X)
print("X = In('cat') =")
print(X)

### Logits And Label Shifting

This section constructs the supervised examples the model actually trains on: where the target tokens come from, and why one window of length $T+1$ yields $T$ classification problems in a single forward pass.

### Mathematical object

A raw token sequence

$$
(x_1,x_2,\ldots,x_{T+1}) \in \mathcal{V}^{T+1}
$$

contains $T$ next-token prediction examples:

$$
(x_1) \mapsto x_2,\qquad
(x_1,x_2) \mapsto x_3,\qquad
\ldots,\qquad
(x_1,\ldots,x_T) \mapsto x_{T+1}.
$$

The windowing construction is a map

$$
\mathcal{V}^{T+1} \longrightarrow \mathcal{V}^{T}\times\mathcal{V}^{T},
\qquad
x_{1:T+1} \longmapsto \big(\underbrace{x_{1:T}}_{\text{input}},\ \underbrace{x_{2:T+1}}_{\text{target}}\big),
$$

i.e. targets are the inputs shifted one position left, $y_t := x_{t+1}$:

```text
text IDs:  [10, 11, 12, 13, 14]
input:     [10, 11, 12, 13]
target:    [11, 12, 13, 14]
```

At input position $t$, the target is the next token.

### Dimensions and tensor shapes

The data helper `get_batch(tokens, block_size, batch_size, device)` returns:

```python
x.shape == (B, T)
y.shape == (B, T)
```

with

```python
y[:, :-1] == x[:, 1:]
```

for each sampled window. The final target in each row is the token immediately after the input window in the original 1D token stream.

The model then returns

```python
logits.shape == (B, T, V)
```

and cross-entropy compares `logits[b, t, :]` with `y[b, t]`.

### Empirical risk from label shifting

The shifted-label construction turns one contiguous sequence into many supervised examples while preserving causal structure. The batch loss is

$$
L(\theta)
= \frac{1}{BT}\sum_{b=1}^{B}\sum_{t=1}^{T}
-\log p_\theta\big(y_{b,t}\mid x_{b,1:t}\big).
$$

The conditioning on $x_{b,1:t}$ — the prefix up to and including position $t$ — is exactly what the causal mask enforces. The transformer receives the full input window in parallel, but masking prevents position $t$ from reading positions greater than $t$. Without the mask, the position predicting $y_{b,t} = x_{b,t+1}$ could read $x_{b,t+1}$ directly from the input, and the objective would be corrupted by the answer being visible.

### Code path

The label shift is created in `src/llm_from_scratch/data.py`:

```python
x = torch.stack([tokens[i : i + block_size] for i in starts])
y = torch.stack([tokens[i + 1 : i + block_size + 1] for i in starts])
```

The loss is consumed in `TinyGPT.forward` by flattening logits and targets.

> **Mathematician's lens.** Label shifting is the step that converts density estimation on sequences (Proposition 0.4) into a sum of conditional classification losses, one per position, all evaluated in a single forward pass.

> **ML practitioner's warning.** Off-by-one label errors can produce a model that trains but learns the wrong task. Always inspect a small `x, y` pair by hand.

### Self-checks

1. For a token stream of length 100 and `block_size=8`, how many possible starting positions are valid in this repo's `get_batch` logic?
2. Why does next-token prediction require a causal mask once the whole input window is processed in parallel?
3. Which tensor axis indexes vocabulary logits, and which tensor indexes target class IDs?


In [ ]:
from llm_from_scratch.data import get_batch

all_ids = torch.arange(20)
x_batch, y_batch = get_batch(all_ids, block_size=5, batch_size=3, device=device)

assert x_batch.shape == y_batch.shape == (3, 5)
assert torch.equal(y_batch[:, :-1], x_batch[:, 1:])
x_batch, y_batch


**Worked Example 3.2 (label shifting on `"cat a"`, all four examples listed).** Encode the five-character string with the running vocabulary: $\operatorname{enc}(\texttt{"cat a"}) = (2, 1, 3, 0, 1)$. A window of width $T = 4$ yields the input–target pair

$$
x = (2,\ 1,\ 3,\ 0), \qquad y = (1,\ 3,\ 0,\ 1),
$$

which is four supervised examples evaluated in one forward pass:

| position $t$ | visible prefix | predict | in characters |
| --- | --- | --- | --- |
| 1 | $(2)$ | $1$ | $\texttt{c} \mapsto \texttt{a}$ |
| 2 | $(2, 1)$ | $3$ | $\texttt{ca} \mapsto \texttt{t}$ |
| 3 | $(2, 1, 3)$ | $0$ | $\texttt{cat} \mapsto \texttt{' '}$ |
| 4 | $(2, 1, 3, 0)$ | $1$ | $\texttt{cat[space]} \mapsto \texttt{a}$ |

The causal mask is what makes the table honest: at position 2 the model must predict $\texttt{t}$ while the input tensor *already contains* $\texttt{t}$ at position 3 — masking is the only thing standing between this construction and an answer key left open on the desk.

In [ ]:
import torch
from llm_from_scratch.tokenization import CharTokenizer

# Worked Example 3.2 — the shift on "cat a", explicitly
tok = CharTokenizer.from_text("a cat")
ids = torch.tensor(tok.encode("cat a"))
assert ids.tolist() == [2, 1, 3, 0, 1]

x, y = ids[:-1], ids[1:]                       # the whole construction
assert x.tolist() == [2, 1, 3, 0]
assert y.tolist() == [1, 3, 0, 1]
for t in range(len(x)):
    prefix = tok.decode(x[: t + 1].tolist())
    target = tok.decode([int(y[t])])
    print(f"position {t+1}: {prefix!r:9} → {target!r}")

### Where We Stand

**What we now have.** Tokens have coordinates. The embedding table realizes each ID as a learned vector, the position table stamps each slot with *where it is*, and their sum $\mathrm{In}(x) \in \mathbb{R}^{T\times C}$ is the residual stream — the tensor every subsequent layer reads and rewrites (Worked Example 3.1 built it by hand for `"cat"`). Label shifting turned raw streams into $(x, y)$ pairs, so the supervision side is complete too.

> **Definition so far.** *An autoregressive language model is a parameterized map that embeds a token prefix into $\mathbb{R}^{T\times C}$ and reads next-token distributions off the result — trained on shifted copies of its own input stream.*

**What is still missing.** The vectors do not talk to each other. Every operation so far acts position-by-position: token $t$'s representation knows its identity and its address, and *nothing about its neighbors*. A model built only from such maps assigns the same next-token distribution after `"the cat"` as after any reordering of unrelated words — its conditionals cannot actually condition. Remark 3.3 sharpened this into a symmetry statement that demands a theorem.

**Where Chapter 4 goes.** Attention: the one operation in the transformer that moves information *across* positions. We will type it precisely (a data-dependent stochastic matrix acting on values), prove the permutation-equivariance theorem that justifies position embeddings retroactively, derive the $1/\sqrt{D}$ temperature, and meet the Gibbs principle a second time — now over prefix positions instead of vocabulary entries.

## 4. Attention From Raw Tensors

### Mathematical object

**Why this chapter.** Chapter 3 ended with an indictment: every operation so far treats positions independently, so the model's "conditionals" cannot actually depend on context — `"the cat"` and `"cat the"` would be indistinguishable. Attention is the answer, and the reason the papers say it is *all* you need: it is the sole operation in the transformer through which one position reads another. This chapter builds it from raw dot products, with every design constant derived rather than accepted.

Attention is a learned weighted average. Each token position builds three vectors:

- a query $q_t$: what this position is looking for,
- a key $k_s$: what source position $s$ offers as an address,
- a value $v_s$: what source position $s$ contributes if selected.

**Definition 4.1 (causal mask).** The causal mask of size $T$ is the matrix

$$
M \in \{0,-\infty\}^{T\times T},
\qquad
M_{t,s} = \begin{cases} 0, & s\le t,\\ -\infty, & s>t, \end{cases}
$$

used *additively*: it is added to score matrices before the row-wise softmax. (The `masked_fill(..., -inf)` idiom in the code is the implementation of this addition.) By convention $e^{-\infty}=0$.

**Definition 4.2 (single-head scaled dot-product attention).** Fix a per-head dimension $D$. Single-head causal attention is the map

$$
\mathrm{A}: \mathbb{R}^{T\times D}\times\mathbb{R}^{T\times D}\times\mathbb{R}^{T\times D}
\longrightarrow \mathbb{R}^{T\times D},
\qquad
\mathrm{A}(Q,K,V) = W\,V,
$$

where the weight matrix $W = W(Q,K)$ is built in three steps:

$$
A = QK^{\top} \in \mathbb{R}^{T\times T}
\quad\text{(scores, } a_{t,s} = q_t\cdot k_s\text{)},
$$

$$
W_{t,:} = \operatorname{softmax}\!\left(\frac{A_{t,:}}{\sqrt{D}} + M_{t,:}\right)
\in \Delta^{T-1}
\quad\text{(row-wise, } t = 1,\ldots,T\text{)},
$$

$$
o_t = \sum_{s=1}^{T} W_{t,s}\, v_s
\quad\text{(output rows)}.
$$

Inside the model, $Q = XW_Q$, $K = XW_K$, $V = XW_V$ are linear images of the residual stream $X\in\mathbb{R}^{T\times C}$, so a head with its projections is a map $\mathbb{R}^{T\times C}\to\mathbb{R}^{T\times D}$, and multi-head attention with output projection is a map $\mathbb{R}^{T\times C}\to\mathbb{R}^{T\times C}$ (assembled in Chapter 5).

**Definition 4.3 (causal row-stochastic matrices).** Let

$$
\mathcal{S}_T = \Big\{ W\in\mathbb{R}^{T\times T} :\; W_{t,s}\ge 0,\ \ \sum_{s=1}^{T} W_{t,s}=1,\ \ W_{t,s}=0 \text{ for } s>t \Big\},
$$

the set of lower-triangular row-stochastic matrices. Definition 4.2 factors as

$$
(Q,K) \longmapsto W(Q,K) \in \mathcal{S}_T,
\qquad
(W, V) \longmapsto WV,
$$

so attention is *linear in the values* but the operator $W$ itself is a nonlinear (softmax) function of the data. (Worked Example 4.1 computes every entry of $W$ for the word `"cat"`.) Attention constructs a data-dependent linear operator on the sequence axis. Row $t$ of $W$ is a probability distribution over the prefix positions $\{1,\ldots,t\}$ — its support is the face of the simplex allowed by the mask.

**Proposition 4.4 (outputs are convex combinations of prefix values).** For every receiving position $t$,

$$
o_t \in \operatorname{conv}\{v_1,\ldots,v_t\} \subset \mathbb{R}^D.
$$

*Proof.* $o_t = \sum_{s\le t} W_{t,s} v_s$ with $W_{t,s}\ge 0$, $\sum_{s\le t}W_{t,s}=1$, which is the definition of a convex combination; the mask (Definition 4.1) zeroes the coefficients with $s>t$. $\blacksquare$

Attention can therefore *route and mix* information from the prefix, but a single head can never leave the convex hull of the value vectors it sees. Escaping the hull is the job of the residual connection and the MLP (Chapter 5).

### The symmetry theorem

The following theorem is the justification, promised in Remark 3.3, for the existence of position embeddings. State it for unmasked attention, where the symmetry is exact.

**Theorem 4.5 (permutation equivariance).** Let $\pi\in S_T$ be a permutation with matrix $P_\pi$ ($(P_\pi X)_{t,:} = X_{\pi^{-1}(t),:}$), and let $\mathrm{A}_0$ denote attention as in Definition 4.2 but with $M\equiv 0$ and $Q,K,V$ produced from $X$ by fixed linear maps $W_Q,W_K,W_V$. Then

$$
\mathrm{A}_0(P_\pi X) = P_\pi\, \mathrm{A}_0(X)
\qquad \text{for all } X \in \mathbb{R}^{T\times C},\ \pi\in S_T.
$$

The same holds for every positionwise map $f$ applied row-by-row: $f(P_\pi X) = P_\pi f(X)$. Consequently, any composition of unmasked attention layers and positionwise maps is permutation-equivariant, and the induced next-token distribution depends on the input only through its *multiset* of tokens.

*Proof.* The projections are positionwise: $(P_\pi X)W_Q = P_\pi Q$, and likewise for $K, V$. The scores transform by conjugation:

$$
(P_\pi Q)(P_\pi K)^{\top} = P_\pi\, QK^{\top} P_\pi^{\top}.
$$

Row-wise softmax commutes with simultaneous row and column permutation: permuting rows permutes which row is normalized, permuting columns permutes entries within a row, and softmax of a permuted vector is the permuted softmax. Hence the weight matrix transforms as $W \mapsto P_\pi W P_\pi^{\top}$, and

$$
(P_\pi W P_\pi^{\top})(P_\pi V) = P_\pi (WV),
$$

using $P_\pi^{\top}P_\pi = I$. Positionwise maps commute with $P_\pi$ by definition. Composition preserves the property. $\blacksquare$

**Remark.** The causal mask $M$ is *not* invariant under conjugation by general $P_\pi$ (it distinguishes "earlier" from "later"), so masked attention retains only a weaker symmetry; but the moral survives: without $W_P$, tokens carry no coordinate that says *where* they are, and order information can enter only through the mask's asymmetry — far too weak a channel to encode syntax. Position embeddings break the symmetry explicitly (Remark 3.3), the same way an external field selects a direction in an otherwise isotropic system.

### The Gibbs view of attention

**Proposition 4.6 (attention weights solve an entropy-regularized selection problem).** Fix a receiving position $t$, scores $a_s = q_t\cdot k_s$ for $s \le t$, and $\tau>0$. The problem

$$
\max_{w\,\in\,\Delta^{t-1}}\ \Big\{ \textstyle\sum_{s\le t} w_s a_s \;+\; \tau H(w) \Big\}
$$

has the unique solution $w^\ast = \operatorname{softmax}(a/\tau)$, and scaled dot-product attention is the case $\tau = \sqrt{D}$.

*Proof.* This is Theorem 1.6 with $V$ replaced by $t$ and $z$ by $a$: the same strictly concave objective, the same Lagrangian

$$
\mathcal{L}(w,\mu) = \sum_s w_s a_s - \tau\sum_s w_s\log w_s + \mu\Big(\sum_s w_s - 1\Big),
$$

whose stationarity condition $a_s - \tau(\log w_s + 1) + \mu = 0$ gives $w_s \propto e^{a_s/\tau}$. Comparing with Definition 4.2, the division of scores by $\sqrt D$ is division by the temperature. $\blacksquare$

So each attention row is a Gibbs distribution over prefix positions, with "energy" $-q_t\cdot k_s$ and temperature $\sqrt D$: attention *wants* to select the best-matching source position (score-seeking term) but is held back from a hard argmax by entropy. This is why softmax attention is the canonical differentiable relaxation of retrieval-by-address — the hard limit $\tau\to 0$ is exact lookup, non-differentiable and untrainable by gradients.

### Dimensions and tensor shapes

The teaching function `scaled_dot_product_attention` uses tensors shaped `(B, T, D)`:

```python
scores = q @ k.transpose(-2, -1)  # (B, T, T)
weights = stable_softmax(scores, dim=-1)
out = weights @ v                # (B, T, D)
```

The module `CausalSelfAttention` handles multiple heads. It starts with `x.shape == (B, T, C)`, applies one linear layer to obtain concatenated Q, K, and V, then reshapes each to `(B, H, T, D)` where `D = C // H`. The score tensor has shape `(B, H, T, T)`.

### Statistical and modeling role

Attention lets the conditional distribution at one position depend on earlier token representations. Without attention or recurrence, each position would be transformed independently after embeddings. With causal self-attention, position $t$ forms a context-dependent representation by reading positions $s \le t$. Attention does not introduce a new loss; it changes the function class used to compute logits.

### Code path

The notebook first uses `scaled_dot_product_attention` from `src/llm_from_scratch/functional.py`. The full model uses `CausalSelfAttention` in `src/llm_from_scratch/attention.py`.

> **Mathematician's lens.** Attention factors through the space $\mathcal{S}_T$ of causal row-stochastic matrices (Definition 4.3): a data-dependent linear operator on positions, whose rows are Gibbs distributions (Proposition 4.6), constrained to convex mixing (Proposition 4.4), and — absent positional terms — fully symmetric under reordering (Theorem 4.5).

> **ML practitioner's warning.** Attention weights are not automatically explanations. They are internal mixture weights that can be affected by scaling, masking, heads, layer composition, and downstream projections.

### Self-checks

1. If `q` and `k` have shape `(B, T, D)`, why does `q @ k.transpose(-2, -1)` have shape `(B, T, T)`?
2. Which axis should softmax normalize in the attention score tensor?
3. Why is `weights @ v` a weighted average over source positions rather than over channels?
4. (*) Complete the middle step of Theorem 4.5: prove that row-wise softmax satisfies $\operatorname{softmax}(P A P^{\top}) = P \operatorname{softmax}(A) P^{\top}$ for a permutation matrix $P$.


### Scaling And The Causal Mask

Definition 4.2 contained two ingredients beyond the raw dot products: division by $\sqrt{D}$ and the additive mask $M$. Neither is decorative — this section derives why each must be there, the first from a variance computation, the second from the causal structure of next-token prediction.

### Why $\sqrt{D}$: the variance lemma

**Lemma 4.7 (variance of a random dot product).** Let $q = (q_1,\ldots,q_D)$ and $k=(k_1,\ldots,k_D)$ have entries that are mutually independent with $\mathbb{E}[q_r]=\mathbb{E}[k_r]=0$ and $\operatorname{Var}(q_r)=\operatorname{Var}(k_r)=1$. Then

$$
\mathbb{E}[\,q\cdot k\,]=0,
\qquad
\operatorname{Var}(q\cdot k) = D,
\qquad
\operatorname{Var}\!\left(\frac{q\cdot k}{\sqrt{D}}\right) = 1 .
$$

*Proof.* Each term of $q\cdot k = \sum_{r=1}^D q_r k_r$ has mean $\mathbb{E}[q_r]\mathbb{E}[k_r] = 0$ by independence, and second moment

$$
\mathbb{E}[(q_rk_r)^2] = \mathbb{E}[q_r^2]\,\mathbb{E}[k_r^2] = 1\cdot 1 = 1,
$$

so $\operatorname{Var}(q_rk_r)=1$. Distinct terms are uncorrelated: for $r\ne r'$,

$$
\mathbb{E}[q_rk_r\,q_{r'}k_{r'}] = \mathbb{E}[q_rq_{r'}]\,\mathbb{E}[k_rk_{r'}] = 0 .
$$

Variances of uncorrelated terms add, giving $\operatorname{Var}(q\cdot k)=D$; dividing by $\sqrt D$ divides the variance by $D$. $\blacksquare$

The hypotheses are an *initialization model*, not a theorem about trained networks: at initialization, projections of normalized activations approximately satisfy them; after training the distributions drift. The design consequence stands: unscaled scores have standard deviation $\sqrt D$, and in the Gibbs picture (Proposition 4.6) score magnitude divided by temperature is what sets the entropy of the attention row. Without the $1/\sqrt D$ scaling, larger heads would start at effectively lower temperature — saturated, near-argmax rows whose softmax Jacobian $\operatorname{diag}(w)-ww^\top$ (Proposition 1.5) is nearly zero, killing gradients. Scaling by $1/\sqrt D$ keeps the initial score scale, hence the initial row entropy, independent of head width.

### Causal masking

For next-token prediction, receiving position $t$ may use positions $s\le t$ but not $s>t$. In the additive convention of Definition 4.1, the mask matrix $M\in\{0,-\infty\}^{T\times T}$ is added to the scaled scores before softmax. Since $e^{-\infty}=0$,

$$
\operatorname{softmax}\big((\ldots, a, \ldots, -\infty, \ldots)\big)
= \big(\ldots, \tfrac{e^{a}}{Z}, \ldots, 0, \ldots\big),
$$

forbidden positions receive *exactly* zero probability, and each row of $W$ is a distribution supported on the face $\{w \in \Delta^{T-1} : w_s = 0,\ s>t\} \cong \Delta^{t-1}$ of the simplex — a smaller simplex determined by causal order.

### Dimensions and code path

The helper `causal_mask(T)` returns a `(T, T)` Boolean lower-triangular mask (the Boolean form of $M$: `True` where $M=0$). In `CausalSelfAttention`, the mask is stored as

```python
self.mask.shape == (1, 1, block_size, block_size)
```

so it can broadcast over `(B, H, T, T)` attention scores — legitimate because every batch element and head obeys the same causal order. At runtime the module slices it to the actual sequence length:

```python
scores = scores.masked_fill(~self.mask[:, :, :seq_len, :seq_len], float("-inf"))
```

`masked_fill` with $-\infty$ is the implementation of the additive mask $M$.

The next code cell verifies three contracts:

1. output shape is `(B, T, D)`,
2. attention weights have shape `(B, T, T)`,
3. masked future weights are zero and each row sums to 1.

> **Mathematician's lens.** The mask restricts the support of each row distribution: attention weights still form a probability vector, but on the sub-simplex determined by causal order. The $1/\sqrt D$ factor is a temperature normalization (Lemma 4.7 + Proposition 4.6).

> **ML practitioner's warning.** Forgetting the causal mask can produce deceptively low training loss because the model can access information that should be unavailable at inference time.

### Self-checks

1. In Lemma 4.7, where exactly is independence used, and which step fails if $q$ and $k$ are correlated?
2. Why does setting a score to $-\infty$ before softmax force probability zero?
3. In a multi-head score tensor `(B, H, T, T)`, which two axes does the causal mask constrain?


In [ ]:
from llm_from_scratch.functional import causal_mask, scaled_dot_product_attention

B, T, D = 1, 5, 8
q = torch.randn(B, T, D)
k = torch.randn(B, T, D)
v = torch.randn(B, T, D)
mask = causal_mask(T)
out, weights = scaled_dot_product_attention(q, k, v, mask)

assert out.shape == (B, T, D)
assert weights.shape == (B, T, T)
assert torch.allclose(weights[0].triu(1), torch.zeros(T, T).triu(1))
assert torch.allclose(weights.sum(dim=-1), torch.ones(B, T))
out.shape, weights[0]


**Worked Example 4.1 (one attention head over `"cat"`, every number visible).** Take $X \in \mathbb{R}^{3\times 2}$ from Worked Example 3.1 and the simplest possible projections: $W_Q = W_V = I$ and $W_K = \left(\begin{smallmatrix}0&1\\1&0\end{smallmatrix}\right)$ (a column swap), so

$$
Q = X = \begin{pmatrix} 1.1 & 0.5 \\ 0.5 & -0.4 \\ -0.6 & 1 \end{pmatrix},
\qquad
K = XW_K = \begin{pmatrix} 0.5 & 1.1 \\ -0.4 & 0.5 \\ 1 & -0.6 \end{pmatrix},
\qquad
V = X .
$$

**Scores.** Each entry of $A = QK^{\top}$ is one dot product. The entry for receiving position 3 (token $\texttt{t}$) reading source position 1 (token $\texttt{c}$), written out:

$$
a_{3,1} = q_3 \cdot k_1 = (-0.6)(0.5) + (1)(1.1) = -0.30 + 1.10 = 0.80 .
$$

Doing all nine and scaling by $1/\sqrt{D} = 1/\sqrt{2}$ (Lemma 4.7's temperature):

$$
\frac{A}{\sqrt{2}} =
\begin{pmatrix}
 0.7778 & -0.1344 & 0.5657 \\
-0.1344 & -0.2828 & 0.5233 \\
 0.5657 &  0.5233 & -0.8485
\end{pmatrix}
\ \xrightarrow{\ +M\ }\
\begin{pmatrix}
 0.7778 & -\infty & -\infty \\
-0.1344 & -0.2828 & -\infty \\
 0.5657 &  0.5233 & -0.8485
\end{pmatrix}.
$$

**Softmax, row by row.** Row 1 has a single unmasked entry, so $W_{1,:} = (1, 0, 0)$ — position 1 has nothing else to read, and the mask forces its distribution to a *vertex* of the simplex. Row 3 in full: subtract the row max $0.5657$, exponentiate to $(1,\ 0.9585,\ 0.2431)$, sum to $Z = 2.2016$, divide:

$$
W =
\begin{pmatrix}
1 & 0 & 0 \\
0.5371 & 0.4629 & 0 \\
0.4542 & 0.4354 & 0.1104
\end{pmatrix}
\in \mathcal{S}_3 \quad (\text{Definition 4.3: rows sum to } 1,\ \text{upper triangle } 0).
$$

**Output.** Row 3 of $O = WV$ is the convex combination of Proposition 4.4, with the prefix's value vectors as the vertices:

$$
o_3 = 0.4542\,v_1 + 0.4354\,v_2 + 0.1104\,v_3 = (0.6511,\ 0.1634).
$$

The token $\texttt{t}$ has built its contextual representation by taking roughly $45\%$ of $\texttt{c}$, $44\%$ of $\texttt{a}$, and keeping $11\%$ of itself — that sentence *is* the content of $\mathrm{A}(Q,K,V) = WV$, at $T=3$, with nothing hidden.

In [ ]:
import torch
from llm_from_scratch.functional import causal_mask, scaled_dot_product_attention

# Worked Example 4.1 — one head over "cat", verified against the hand matrices
X = torch.tensor([[1.1, 0.5], [0.5, -0.4], [-0.6, 1.0]])
W_K = torch.tensor([[0.0, 1.0], [1.0, 0.0]])

q = X.unsqueeze(0)                     # W_Q = I        (1, 3, 2)
k = (X @ W_K).unsqueeze(0)             # column swap    (1, 3, 2)
v = X.unsqueeze(0)                     # W_V = I        (1, 3, 2)

# the expanded entry: a_31 = q_3 · k_1
assert abs((q[0, 2] @ k[0, 0]).item() - 0.80) < 1e-6

out, weights = scaled_dot_product_attention(q, k, v, causal_mask(3))

expected_W = torch.tensor([[1.0,    0.0,    0.0   ],
                           [0.5371, 0.4629, 0.0   ],
                           [0.4542, 0.4354, 0.1104]])
assert torch.allclose(weights[0], expected_W, atol=5e-5)
assert torch.allclose(weights.sum(dim=-1), torch.ones(1, 3))          # rows on the simplex

expected_o3 = torch.tensor([0.6511, 0.1634])
assert torch.allclose(out[0, 2], expected_o3, atol=5e-5)              # the convex combination
print("weights W =")
print(weights[0])
print("o_3 =", [round(v_, 4) for v_ in out[0, 2].tolist()])

### The `T x T` Bottleneck

The score matrix has one row for every receiving position and one column for every source position. That is $T^2$ scores per head per example. With batch and heads included, the attention score tensor has shape

$$
(B,H,T,T).
$$

The memory for scores alone is proportional to

$$
BHT^2.
$$

The matrix multiplication cost also grows quadratically in $T$ for each head. This is one of the central bottlenecks that motivates many alternatives to vanilla attention.

### Why the bottleneck matters

If `T=2048`, one attention head forms $2048^2 = 4,194,304$ scores for one example. If the model has many heads and layers, the total number of intermediate scores becomes large. During training, additional memory is needed for gradients and optimizer state. During decoding with a KV cache, the pattern changes: one new query attends over cached keys and values, but the cache itself grows with sequence length.

### Connection to later architecture ideas

Many post-transformer or transformer-variant ideas can be read as answers to a precise question:

- Can we avoid comparing every position with every other position?
- Can we share keys and values across heads?
- Can we compress history into a state rather than keep all pairwise interactions?
- Can retrieval supply relevant context without extending the model's internal sequence indefinitely?
- Can a different objective learn representations with fewer labeled or next-token samples?

The important habit is to identify the bottleneck before judging the architecture name.

> **Mathematician's lens.** Full attention constructs a dense kernel matrix over token positions. Sparse, local, recurrent, and state-space variants change the structure or representation of that operator.

> **ML practitioner's warning.** Reducing asymptotic cost does not automatically improve wall-clock speed or model quality. Hardware kernels, memory layout, batch size, and data distribution often decide whether a theoretical improvement matters.


### Residual-Stream View: What Attention Moves

The transformer block is easiest to read if you track the residual stream: one vector per token position, repeatedly modified by sublayers. Most sublayers act positionwise. The MLP transforms each token vector independently. Layer norm normalizes each token vector independently across its channel dimension.

Attention is the exceptional operation. It lets a receiving position build an update from source positions. In the CS124 framing, query, key, and value are three projected roles of the same residual-stream vectors:

- the **query** is what the receiving position asks for;
- the **key** is what each source position offers for matching;
- the **value** is the information copied, averaged, or mixed into the receiving position.

The causal mask determines which source positions are allowed to write into each receiver. Without the mask, a decoder-only model can leak future target information into the current representation. With the mask, attention becomes controlled information movement from the prefix into the current position.

This view also explains why attention dominates many architecture discussions. It is the main cross-position mixer, and its score tensor grows as `(B, H, T, T)`.

Source note: this integrates the CS124 transformer lecture's residual-stream, Q/K/V, causal attention, and quadratic-cost framing plus the PA6a attention scaffold in `../data/cs124/pa6a-transformers/model.py`.


In [ ]:
for T_value in [128, 512, 2048, 8192]:
    print(T_value, "full attention scores per head:", T_value * T_value)


### Where We Stand

**What we now have.** The conditioning mechanism. Each position emits a query, each earlier position offers a key and a value, and a softmax over match scores builds a distribution over the prefix — a row of the causal stochastic matrix $W \in \mathcal{S}_T$ (Definition 4.3) — which then mixes the values. Worked Example 4.1 ran every digit of this for `"cat"`. The two design constants earned their existence: the mask is what keeps the factorization of Chapter 0 valid under parallel evaluation, and $1/\sqrt{D}$ is a temperature that keeps rows from saturating (Lemma 4.7 + Proposition 4.6). Theorem 4.5 paid the debt from Chapter 3: without positional terms, attention literally cannot see order.

> **Definition so far.** *An autoregressive language model is an embedded token sequence transformed by data-dependent convex mixing over each prefix — attention — so that every position's representation, and hence its next-token distribution, is a function of everything before it.*

**What is still missing.** Expressive power and depth. One attention output is trapped in the convex hull of its value vectors (Proposition 4.4) — averaging can route information but cannot create new directions. There is no nonlinearity yet, no way to stack layers stably, and no account of how the pieces compose into one trainable map.

**Where Chapter 5 goes.** The transformer block: MLPs that escape the hull, layer norm that confines each token vector to a sphere (with an instructive catastrophe at $C = 2$), residual connections that make hundred-layer compositions trainable — and then the payoff, the entire model written as a single typed chain from $\mathcal{V}^T$ to $(\mathring{\Delta}^{V-1})^T$.

## 5. The Transformer Block

### Mathematical object

A decoder-only transformer block is a residual map on a sequence of vectors: a map $\mathbb{R}^{T\times C}\to\mathbb{R}^{T\times C}$ (batched: $\mathbb{R}^{B\times T\times C}\to\mathbb{R}^{B\times T\times C}$) of the form "identity plus update". Shape preservation is what makes blocks stackable.

**Definition 5.1 (layer normalization).** Layer norm with parameters $\gamma,\beta\in\mathbb{R}^C$ is the map

$$
\operatorname{LN}: \mathbb{R}^C \longrightarrow \mathbb{R}^C,
\qquad
\operatorname{LN}(u)_i = \gamma_i\,\frac{u_i-\mu(u)}{\sqrt{\sigma^2(u)+\epsilon}} + \beta_i,
$$

where

$$
\mu(u) = \frac{1}{C}\sum_{i=1}^{C} u_i,
\qquad
\sigma^2(u) = \frac{1}{C}\sum_{i=1}^{C}\big(u_i-\mu(u)\big)^2 .
$$

Applied to a residual stream in $\mathbb{R}^{T\times C}$, it acts row-by-row (positionwise), normalizing each token vector across its channel axis.

**Proposition 5.2 (the geometry of layer norm).** Set $\epsilon=0$ and let $N(u) = \frac{u - \mu(u)\mathbf{1}}{\sigma(u)}$ be the normalization before the affine reparameterization. Then $N$ maps $\mathbb{R}^C \setminus \mathbb{R}\mathbf{1}$ onto the sphere

$$
S = \Big\{ v\in\mathbb{R}^C : \textstyle\sum_i v_i = 0,\ \ \lVert v\rVert_2 = \sqrt{C} \Big\},
$$

a $(C-2)$-dimensional sphere of radius $\sqrt C$ inside the mean-zero hyperplane $\mathbf{1}^{\perp}$, and $N$ is invariant under the two-parameter group $u \mapsto au + b\mathbf{1}$, $a>0$, $b\in\mathbb{R}$.

*Proof.* Mean-zero: $\sum_i (u_i - \mu) = C\mu - C\mu = 0$. Norm: $\lVert u - \mu\mathbf{1}\rVert_2^2 = \sum_i (u_i-\mu)^2 = C\sigma^2$, so $\lVert N(u)\rVert_2^2 = C\sigma^2/\sigma^2 = C$. Invariance: $\mu(au+b\mathbf{1}) = a\mu(u)+b$ and $\sigma(au+b\mathbf{1}) = a\sigma(u)$ for $a>0$, so the shift and scale cancel. Surjectivity onto $S$: any $v \in S$ satisfies $N(v)=v$. $\blacksquare$

Layer norm thus quotients out exactly two degrees of freedom per token — overall offset and overall scale — projecting each token vector onto a fixed sphere, after which $\gamma$ and $\beta$ restore a learned, diagonal affine coordinate system. This is why activations cannot blow up or collapse in scale as depth grows: every block's sublayer reads its input from a compact set.

**Definition 5.3 (positionwise MLP with GELU).** The feed-forward sublayer is the positionwise map

$$
\operatorname{MLP}: \mathbb{R}^C \to \mathbb{R}^C,
\qquad
\operatorname{MLP}(u) = W_2\,\operatorname{GELU}(W_1u + b_1) + b_2,
\qquad
W_1 \in \mathbb{R}^{4C\times C},\ W_2\in\mathbb{R}^{C\times 4C},
$$

an expansion to width $4C$ followed by a nonlinearity and projection back. The nonlinearity is

$$
\operatorname{GELU}(u) = u\,\Phi(u), \qquad \Phi(u) = \tfrac{1}{2}\Big(1 + \operatorname{erf}\big(u/\sqrt{2}\big)\Big),
$$

$\Phi$ the standard normal CDF — a smooth interpolation between $0$ (for $u \ll 0$) and the identity (for $u \gg 0$), i.e. a mollified ReLU.

**Definition 5.4 (pre-norm block).** The block of `TinyGPT` is the map $\mathcal{B}: \mathbb{R}^{T\times C}\to\mathbb{R}^{T\times C}$,

$$
x' = x + \operatorname{Attn}(\operatorname{LN}_1(x)),
\qquad
\mathcal{B}(x) = x' + \operatorname{MLP}(\operatorname{LN}_2(x')),
$$

with $\operatorname{Attn}$ the multi-head causal attention of Chapter 4 and $\operatorname{LN}, \operatorname{MLP}$ acting positionwise. (Post-norm instead computes $\operatorname{LN}(x + F(x))$; this repo uses pre-norm.)

```python
x = x + self.attn(self.ln_1(x))
x = x + self.ff(self.ln_2(x))
```

### Residual composition: the flow picture

**Proposition 5.5 (residual Jacobian).** Write one sublayer as $x \mapsto x + f(x)$ with $f$ differentiable. Its Jacobian is

$$
\frac{\partial(x + f(x))}{\partial x} = I + J_f(x),
$$

and for a stack of $L$ sublayers the chain rule gives

$$
\frac{\partial x_L}{\partial x_0} = \prod_{l=L}^{1}\big(I + J_{f_l}(x_{l-1})\big)
= I + \sum_l J_{f_l} + \text{(higher products)} .
$$

*Proof.* Differentiate the update; expand the product. $\blacksquare$

The leading identity term is the point: gradients reaching layer $L$ propagate to layer $1$ along a direct additive path, regardless of how small the individual $J_{f_l}$ are. In a plain composition $f_L\circ\cdots\circ f_1$ the Jacobian is a bare product $\prod J_{f_l}$, which attenuates exponentially when the factors have small norm — the classical vanishing-gradient failure. Residual connections change the algebra from products to "identity plus perturbation".

**Remark (the ODE view).** The update $x_{l+1} = x_l + f_l(x_l)$ is the explicit Euler step, with $\Delta t = 1$, of the flow

$$
\dot x(t) = f(x(t), t),
$$

so a deep residual network is a discretized flow on $\mathbb{R}^{T\times C}$, each token vector following a trajectory driven alternately by an interaction term (attention: coupling across positions) and a self-interaction term (MLP: positionwise). If a sublayer is useless early in training it can learn $f\approx 0$ and the flow is near-identity — the network is a perturbation of the identity map, which is also why training deep residual stacks is possible at all.

### Multi-head attention inside the block

The attention module projects the residual stream to Q, K, V with a single linear layer. If `C=32` and `H=4`, then `D=8`. The channel space decomposes as a direct sum

$$
\mathbb{R}^C \cong \bigoplus_{h=1}^{H} \mathbb{R}^{D},
$$

each head running Definition 4.2 in its own $D$-dimensional subspace with its own weight matrix $W^{(h)}\in\mathcal{S}_T$; the outputs are concatenated back to width $C$ and mixed by the output projection $W_O\in\mathbb{R}^{C\times C}$. Shapes:

```text
x:      (B, T, C)
qkv:    (B, T, 3C)
q,k,v:  (B, T, C)
heads:  (B, H, T, D)
scores: (B, H, T, T)
out:    (B, T, C)
```

### The model as a typed composition

Assembling Chapters 2–5, the entire toy model is the composition

$$
\mathcal{V}^{T}
\;\xrightarrow{\ \mathrm{In}\ (\text{Def } 3.2)\ }\;
\mathbb{R}^{T\times C}
\;\xrightarrow{\ \mathcal{B}_1\ }\;
\mathbb{R}^{T\times C}
\;\xrightarrow{\ \mathcal{B}_2\ }\;
\cdots
\;\xrightarrow{\ \mathcal{B}_L\ }\;
\mathbb{R}^{T\times C}
\;\xrightarrow{\ \operatorname{LN}_f\ }\;
\mathbb{R}^{T\times C}
\;\xrightarrow{\ \cdot\,W_U\ }\;
\mathbb{R}^{T\times V}
\;\xrightarrow{\ \sigma\ (\text{rows})\ }\;
\big(\mathring{\Delta}^{V-1}\big)^{T},
$$

where $W_U$ is the language-model head ($W_U = W_E^{\top}$ under weight tying — see the next section) and the final softmax is applied to each row. Row $t$ of the output is $p_\theta(\,\cdot\mid x_{1:t})$: one forward pass evaluates all $T$ conditional distributions of Definition 0.3 simultaneously, which is exactly what the shifted labels of Chapter 3 consume. The batch dimension is a Cartesian power of this diagram.

### Code path

The relevant files are:

- `src/llm_from_scratch/attention.py`: `CausalSelfAttention`,
- `src/llm_from_scratch/model.py`: `FeedForward`, `GPTBlock`, `TinyGPT`,
- `src/llm_from_scratch/configs.py`: `ModelConfig` shape constraints.

The next code cell instantiates `TinyGPT`, sends a random `(B, T)` token-ID batch through it, and checks that logits have shape `(B, T, V)`.

> **Mathematician's lens.** A transformer stack is a composition of shape-preserving residual maps on $\mathbb{R}^{T\times C}$ — a discretized flow — with attention mixing across the sequence axis and MLPs mixing across the channel axis, layer norm confining each token vector to a sphere (Proposition 5.2) before every sublayer reads it.

> **ML practitioner's warning.** Residual connections and layer norm help optimization, but they do not fix bad data, label leakage, an impossible context length, or a learning rate that is too large.

### Self-checks

1. Why must `n_embd` be divisible by `n_head` in `ModelConfig`?
2. Which sublayer mixes information across token positions, and which sublayer acts positionwise?
3. Why does the residual addition require the sublayer output to have shape `(B, T, C)`?
4. (*) In Proposition 5.2, verify the dimension count: why is the image a $(C-2)$-dimensional sphere and not $(C-1)$-dimensional?


In [ ]:
from llm_from_scratch.model import TinyGPT, count_parameters

cfg = ModelConfig(vocab_size=64, block_size=16, n_embd=32, n_head=4, n_layer=2, dropout=0.0)
model = TinyGPT(cfg)
idx = torch.randint(0, cfg.vocab_size, (2, cfg.block_size))
logits, loss = model(idx, idx)

assert logits.shape == (2, cfg.block_size, cfg.vocab_size)
assert loss is not None and loss.item() > 0
count_parameters(model), logits.shape, loss.item()


**Worked Example 5.1 (LayerNorm at $C=2$: an instructive catastrophe).** Apply the normalization core $N(u) = (u - \mu\mathbf{1})/\sigma$ of Proposition 5.2 to the first row of our residual stream, $u = (1.1,\ 0.5)$:

$$
\mu = 0.8, \qquad u - \mu\mathbf{1} = (0.3,\ -0.3), \qquad \sigma = 0.3,
\qquad N(u) = (1,\ -1).
$$

Now the second row, $u' = (0.5,\ -0.4)$: $\mu = 0.05$, $u' - \mu\mathbf{1} = (0.45, -0.45)$, $\sigma = 0.45$ — and $N(u') = (1,\ -1)$ *again*. This is Proposition 5.2 in its degenerate case: at $C = 2$ the sphere $S$ is $(C-2)$-dimensional, i.e. zero-dimensional — exactly the two points $\pm(1,-1)$ — so normalization destroys everything about a token vector except which coordinate is larger. The information carried through a $C=2$ model would be one bit per token.

The lesson runs in both directions. It confirms the dimension count of Proposition 5.2 by exhibiting the sphere as two points; and it explains *why residual width matters*: at realistic $C$, the sphere $\sqrt{C}\,S^{C-2}$ is a huge manifold and normalization costs only the two nuisance degrees of freedom (offset and scale). The learned $\gamma, \beta$ then re-open the affine directions the model actually needs.

In [ ]:
import torch

# Worked Example 5.1 — LayerNorm at C=2 collapses to ±(1, -1)
X = torch.tensor([[1.1, 0.5], [0.5, -0.4], [-0.6, 1.0]])

def normalize(u, eps=0.0):
    mu = u.mean()
    sigma = u.var(unbiased=False).add(eps).sqrt()
    return (u - mu) / sigma

for row in X:
    n = normalize(row)
    assert torch.allclose(n.abs(), torch.ones(2), atol=1e-6)          # every output is ±(1,-1)
    assert abs(n.sum().item()) < 1e-6                                 # mean zero
    assert abs(n.norm().item() - 2 ** 0.5) < 1e-6                     # radius sqrt(C)
    print(f"{[round(v, 2) for v in row.tolist()]}  →  {[round(v, 4) for v in n.tolist()]}")

# torch's own LayerNorm (gamma=1, beta=0) agrees:
ln = torch.nn.LayerNorm(2, elementwise_affine=False, eps=1e-12)
assert torch.allclose(ln(X), torch.stack([normalize(r) for r in X]), atol=1e-5)

### Parameter Count Intuition

The toy model has the same categories of parameters as a larger GPT-style model:

| Parameter group | Shape intuition | Role |
| --- | --- | --- |
| Token embedding | `(V, C)` | map token IDs to vectors |
| Position embedding | `(block_size, C)` | encode absolute position |
| QKV projection | roughly `(C, 3C)` per block | create queries, keys, values |
| Attention output projection | roughly `(C, C)` per block | mix concatenated heads |
| MLP first projection | roughly `(C, 4C)` per block | expand channel dimension |
| MLP second projection | roughly `(4C, C)` per block | return to residual width |
| Layer norm parameters | vectors of length `C` | scale and shift normalized activations |
| LM head | conceptually `(C, V)` | map hidden states to vocabulary logits |

### Weight tying, precisely

In `TinyGPT`, the final language-model head shares its weight tensor with the token embedding table:

```python
self.lm_head.weight = self.token_embedding.weight
```

Since a PyTorch `Linear(C, V)` with weight $W\in\mathbb{R}^{V\times C}$ computes $h \mapsto hW^{\top}$, tying sets $W = W_E$, so the logits of a final hidden state $h\in\mathbb{R}^C$ are

$$
z = h\,W_E^{\top} \in \mathbb{R}^{V},
\qquad
z_i = \langle h,\ E(i)\rangle .
$$

The score of token $i$ is the inner product of the hidden state with token $i$'s own embedding: the output layer is a *log-linear model in the embedding geometry*, and

$$
p_\theta(i \mid x_{1:t}) = \frac{e^{\langle h_t, E(i)\rangle}}{\sum_j e^{\langle h_t, E(j)\rangle}}
$$

is a Gibbs distribution whose energies are negative matching scores between the context representation and the candidate token vectors. Reading and scoring use one shared geometry; tying removes $VC$ parameters and couples input and output representations.

### Scaling intuition

Increasing `n_embd` raises the cost of most matrix multiplications roughly quadratically in width. Increasing `n_layer` repeats the block. Increasing `block_size` increases the position table and the maximum attention mask, and increases attention cost quadratically in the sequence length actually used. Increasing `vocab_size` grows the embedding and output dimensions.

The toy model is not useful because it is large. It is useful because the categories of computation match the larger family.

> **Mathematician's lens.** Parameter count is the dimension of the parameter space $\Theta$ (Definition 1.1) — a crude measure of the hypothesis class. Architecture determines how those degrees of freedom are shared across positions and composed.

> **ML practitioner's warning.** More parameters can improve fit, but they also increase memory, compute, overfitting risk on small data, and the need for careful evaluation.


### Where We Stand

**What we now have.** The machine, complete. The block alternates two moves — attention mixes across positions, the MLP transforms each position — with layer norm regulating scale and residual connections providing the identity-plus-perturbation structure that keeps gradients alive through depth (Proposition 5.5). Composed and capped with the tied unembedding, the whole model is one typed line: $\mathcal{V}^T \to \mathbb{R}^{T\times C} \to \cdots \to (\mathring{\Delta}^{V-1})^T$. Every arrow has been derived. Nothing in the box is mysterious anymore.

> **Definition so far.** *An autoregressive language model is a specific smooth composition — embed, then $L$ rounds of (attend, transform), then normalize and score against the embedding geometry — mapping every prefix to a next-token distribution simultaneously in one pass.*

**What is still missing.** The parameters are random numbers. We possess the function class $\{F_\theta : \theta \in \Theta\}$ but have not moved through it: nothing yet connects the loss whose gradient we derived in Chapter 1 to an actual trajectory $\theta_0, \theta_1, \theta_2, \ldots$ that ends somewhere useful — nor any honest way to tell "learned" from "memorized".

**Where Chapter 6 goes.** Training as numerical integration of a noisy gradient flow: minibatch estimates (unbiased, Proposition 6.2), the AdamW update in full (Algorithm 6.3, executed by hand in Worked Example 6.1), the tiny-batch overfit as an implementation diagnostic, and perplexity as the exponential bridge from loss to interpretability.

## 6. Training: Loss, Batches, Optimizer, And Overfitting

### Mathematical object

Training minimizes the empirical risk $L:\Theta\to\mathbb{R}_{\ge 0}$,

$$
L(\theta)
= \frac{1}{BT}\sum_{b=1}^{B}\sum_{t=1}^{T}
-\log p_\theta\big(y_{b,t}\mid x_{b,1:t}\big),
$$

by following noisy derivative information along a trajectory in $\Theta$ (Definition 1.1).

**Definition 6.1 (minibatch gradient estimator).** Let $\widehat{R}(\theta) = \frac{1}{N}\sum_{n=1}^N \ell_n(\theta)$ be the full empirical risk over $N$ token positions. A minibatch estimator draws a uniformly random subset $S\subset\{1,\ldots,N\}$ of size $m$ and returns

$$
g_S(\theta) = \frac{1}{m}\sum_{n\in S}\nabla \ell_n(\theta).
$$

**Proposition 6.2 (unbiasedness).** $\mathbb{E}_S\big[g_S(\theta)\big] = \nabla \widehat{R}(\theta)$.

*Proof.* By symmetry, each index $n$ belongs to $S$ with probability $m/N$. By linearity of expectation,

$$
\mathbb{E}_S\big[g_S(\theta)\big]
= \frac{1}{m}\sum_{n=1}^{N}\Pr[n\in S]\,\nabla\ell_n(\theta)
= \frac{1}{m}\cdot\frac{m}{N}\sum_{n=1}^{N}\nabla\ell_n(\theta)
= \nabla\widehat{R}(\theta). \qquad\blacksquare
$$

So each SGD step follows the true empirical gradient *on average*; the batch size controls the variance, not the bias. In the flow picture of Chapter 1, SGD is a noisy Euler discretization of the gradient flow $\dot\theta = -\nabla\widehat R(\theta)$.

**Algorithm 6.3 (AdamW).** The repo trains with AdamW, whose update at step $k$, given gradient $g_k = g_{S_k}(\theta_k)$ and hyperparameters $\beta_1,\beta_2\in[0,1)$, $\eta>0$, $\lambda\ge 0$, $\varepsilon>0$, is (all operations elementwise on $\Theta\cong\mathbb{R}^P$):

$$
m_k = \beta_1 m_{k-1} + (1-\beta_1)\, g_k
\qquad\text{(first-moment EMA, } m_0=0\text{)},
$$

$$
v_k = \beta_2 v_{k-1} + (1-\beta_2)\, g_k^{2}
\qquad\text{(second-moment EMA, } v_0=0\text{)},
$$

$$
\hat m_k = \frac{m_k}{1-\beta_1^{k}},
\qquad
\hat v_k = \frac{v_k}{1-\beta_2^{k}}
\qquad\text{(bias corrections)},
$$

$$
\theta_{k+1} = \theta_k - \eta\left(\frac{\hat m_k}{\sqrt{\hat v_k}+\varepsilon} \;+\; \lambda\,\theta_k\right).
$$

Reading the update: $\hat m_k$ is a smoothed gradient (momentum); dividing by $\sqrt{\hat v_k}$ rescales each coordinate by its recent root-mean-square gradient, a *diagonal preconditioning* that makes the step size approximately $\pm\eta$ per coordinate regardless of raw gradient scale; the bias corrections fix the cold start $m_0 = v_0 = 0$ (without them, early $m_k$ underestimates the gradient by the factor $1-\beta_1^k$, as one sees by taking expectations of the EMA recursion with stationary $g$); and the decay term $\lambda\theta_k$ is applied *decoupled* from the preconditioner — that decoupling is the "W" in AdamW, distinguishing it from adding $\lambda\theta$ to the gradient, where the decay would get rescaled by $\sqrt{\hat v_k}$ and lose its meaning as uniform shrinkage toward the origin.

Worked Example 6.1 executes one full step of this algorithm by hand. None of this changes the objective. The objective is still average next-token cross-entropy; the optimizer is a rule for moving through parameter space.

### Training loop structure

The helper `train_steps` in `src/llm_from_scratch/train.py` implements the standard loop — one gradient step per pass, each step touching the object noted on the right:

```text
 ┌─▶  x, y = get_batch(tokens)                #  sample a window            x, y : (B, T)
 │        │
 │        ▼
 │    logits, loss = model(x, y)              #  forward pass               logits : (B, T, V),  loss : scalar
 │        │
 │        ▼
 │    optimizer.zero_grad(set_to_none=True)   #  clear stale gradients
 │        │
 │        ▼
 │    loss.backward()                         #  reverse-mode chain rule    ∇L(θ_k) ∈ Θ
 │        │
 │        ▼
 │    clip_grad_norm_(model.parameters(), c)  #  optional: bound the update
 │        │
 │        ▼
 │    optimizer.step()                        #  AdamW (Algorithm 6.3)      θ_k → θ_{k+1}
 │        │
 └────────┘   every eval_interval steps: estimate_loss() on train/val batches (no gradients)
```

The code cell below uses `overfit_tiny_batch`, which intentionally trains on one fixed random batch. This is a diagnostic, not a real training goal.

### Tiny-batch overfit as a diagnostic

A sufficiently expressive model should be able to drive loss down on a tiny fixed batch. If it cannot, likely causes include:

- labels are shifted incorrectly,
- gradients are not flowing,
- the model is in the wrong mode,
- the learning rate is too small or too large,
- logits and targets have incompatible shapes,
- the loss is averaged or masked incorrectly.

The diagnostic asks a narrow question: can the system reduce the objective on data it sees repeatedly? Passing does not prove generalization. Failing is a serious implementation warning.

### Optimization versus generalization

Optimization asks whether the training objective decreases. Generalization asks whether performance transfers to held-out data from the intended distribution — in the language of Theorem 1.9, whether the KL term is small on the *population*, not merely on the sample used for updates. Validation loss estimates the population risk on tokens not used for parameter updates.

> **Mathematician's lens.** Training is numerical integration of a noisy, preconditioned gradient flow on a high-dimensional nonconvex landscape. Proposition 6.2 guarantees the noise is centered; nothing guarantees the flow finds a global minimum — only that the objective's local structure is followed.

> **ML practitioner's warning.** A training curve can improve while the model learns leakage, memorizes duplicates, overfits formatting artifacts, or fails the behavior you actually care about.

### Self-checks

1. Why does a minibatch gradient estimate introduce noise compared with the full empirical gradient, and what does Proposition 6.2 say that noise cannot do?
2. What does tiny-batch overfitting prove, and what does it not prove?
3. Where in `train_steps` are gradients cleared, computed, and applied?
4. (*) In Algorithm 6.3, take $g_k \equiv g$ constant and show by solving the EMA recursion that $m_k = (1-\beta_1^k)\,g$, justifying the bias correction exactly.


In [ ]:
from llm_from_scratch.train import overfit_tiny_batch

set_seed(123)
small_cfg = ModelConfig(vocab_size=16, block_size=8, n_embd=32, n_head=4, n_layer=1, dropout=0.0)
small_model = TinyGPT(small_cfg)
x = torch.randint(0, small_cfg.vocab_size, (8, small_cfg.block_size))
y = torch.randint(0, small_cfg.vocab_size, (8, small_cfg.block_size))
first_loss, last_loss = overfit_tiny_batch(small_model, x, y, steps=30, lr=1e-2)

assert last_loss < first_loss
first_loss, last_loss


### Generalization Versus Memorization

Overfitting a tiny batch is a diagnostic, not the goal. A model that memorizes eight windows has shown that loss, gradients, and optimizer interact productively. It has not shown that the learned conditional distribution works on new text.

Validation loss is the first guard against fooling yourself. Training loss asks: how well did the model fit token positions used for updates? Validation loss asks: how well does the model predict held-out token positions sampled from a similar distribution? If training loss decreases while validation loss stalls or worsens, the model may be overfitting. If both are high, the model may be underfit, poorly optimized, too small, trained on too little data, or facing a harder tokenization problem.

### Perplexity

**Proposition 6.4 (perplexity is a geometric mean; the effective branching factor).** Let $L = \frac{1}{N}\sum_{n=1}^{N} -\log p_n$ be the average negative log-likelihood (in nats) over $N$ predicted tokens, where $p_n$ is the probability the model assigned to the $n$-th observed token. Then

$$
\operatorname{PPL} := e^{L}
= \exp\Big(\frac{1}{N}\sum_n \log \frac{1}{p_n}\Big)
= \Big(\prod_{n=1}^{N} \frac{1}{p_n}\Big)^{1/N},
$$

the geometric mean of the inverse predicted probabilities. In particular:

(i) if the model assigns probability $1$ to every observed token, $\operatorname{PPL}=1$;

(ii) if the model guesses uniformly over $V$ tokens, $p_n = 1/V$ for all $n$ and $\operatorname{PPL} = V$.

*Proof.* Exponentiate the average of logarithms; a mean of logs exponentiates to a geometric mean. (i) and (ii) are immediate substitutions. $\blacksquare$

Perplexity therefore reads as an *effective branching factor*: a perplexity of 20 means the model is, on geometric average, as uncertain as if it were choosing uniformly among 20 equally likely continuations at each step. By Theorem 1.9, its achievable floor is $e^{H}$ where $H$ is the intrinsic conditional entropy of the tokenized data — not 1, and dependent on the tokenizer (Chapter 2), which is one more reason perplexities across different tokenizations are not comparable.

Perplexity is tied directly to the probabilistic objective. It is still not a complete behavioral evaluation: instruction-following, factuality, retrieval, and safety need task-specific measurements.

### Code path

The repo's `evaluate.py` contains `estimate_loss` for train/validation averages and `perplexity(loss)` for the exponential conversion. Generation can look subjectively better or worse than validation loss suggests because decoding changes how probabilities are sampled.

> **Mathematician's lens.** Generalization is about expected loss under a data-generating distribution, while training observes only a finite sample. Perplexity is the exponential of the cross-entropy estimate — a change of scale, not of content.

> **ML practitioner's warning.** Validation sets can leak, become overused, or fail to represent deployment data. Treat validation as evidence, not as proof of broad competence.


**Worked Example 6.1 (one AdamW step, every quantity computed).** Take a single scalar parameter $\theta_0 = 1$, gradient $g_1 = 0.5$, and the common defaults $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\eta = 10^{-3}$, $\lambda = 0.01$, $\varepsilon = 10^{-8}$. Algorithm 6.3, line by line at $k=1$:

$$
m_1 = 0.9 \cdot 0 + 0.1 \cdot 0.5 = 0.05,
\qquad
v_1 = 0.999 \cdot 0 + 0.001 \cdot 0.25 = 0.00025,
$$

$$
\hat m_1 = \frac{0.05}{1 - 0.9} = 0.5,
\qquad
\hat v_1 = \frac{0.00025}{1 - 0.999} = 0.25,
\qquad
\sqrt{\hat v_1} = 0.5,
$$

$$
\theta_1 = 1 - 10^{-3}\left(\frac{0.5}{0.5 + 10^{-8}} + 0.01 \cdot 1\right)
= 1 - 10^{-3}(1 + 0.01) = 0.99899 .
$$

Two things are visible in the digits that the formulas only imply. First, the bias corrections at $k=1$ recover the raw gradient *exactly*: $\hat m_1 = g_1$ and $\hat v_1 = g_1^2$, so the preconditioned step is $\hat m_1/\sqrt{\hat v_1} = g_1/|g_1| = \pm 1$ — **the first AdamW step has magnitude $\eta$ regardless of the gradient's scale**. Replace $g_1 = 0.5$ by $g_1 = 500$ and $\theta_1$ is unchanged. That per-coordinate scale invariance is the entire point of the $\sqrt{\hat v}$ denominator. Second, the decay term $\lambda\theta$ enters *outside* the preconditioner: it contributed $10^{-5}$ here, independent of $v$ — decoupled shrinkage, the W in AdamW.

In [ ]:
import torch

# Worked Example 6.1 — one AdamW step, cross-checked against torch.optim.AdamW
eta, lam = 1e-3, 0.01

for g_val in (0.5, 500.0):                       # same first step for wildly different gradients
    theta = torch.tensor([1.0], requires_grad=True)
    opt = torch.optim.AdamW([theta], lr=eta, betas=(0.9, 0.999), eps=1e-8, weight_decay=lam)
    theta.grad = torch.tensor([g_val])
    opt.step()
    assert abs(theta.item() - 0.99899) < 1e-6, theta.item()
    print(f"g_1 = {g_val:>6}: theta_1 = {theta.item():.5f}   (first step is ±eta, scale-free)")

# hand values: m1=0.05, v1=0.00025, m_hat=0.5, v_hat=0.25 for g=0.5
m1, v1 = 0.1 * 0.5, 0.001 * 0.25
assert (m1 / (1 - 0.9), v1 / (1 - 0.999)) == (0.5, 0.25)

### Where We Stand

**What we now have.** A fitted model, and instruments to judge the fit. SGD follows the empirical gradient on average; AdamW preconditions it so every coordinate steps at scale $\eta$; the loss curve estimates the population risk, whose floor is the entropy of language itself (Theorem 1.9); and perplexity restates that loss as an effective branching factor (Proposition 6.4). The interactive training panel shows all of this on a real 400-step run — including the train/val gap opening, which is generalization theory made visible.

> **Definition so far.** *An autoregressive language model is the typed composition of Chapter 5 at a particular point $\theta^\ast$ of parameter space — the endpoint of a noisy descent trajectory — whose quality is an expected KL divergence estimated on held-out text.*

**What is still missing.** Text. A trained model is a machine for scoring continuations; it does not, by itself, produce any. Turning conditional distributions into sequences requires *decisions* — sample or argmax? at what temperature? from how much of the tail? — and none of those decisions is dictated by training.

**Where Chapter 7 goes.** Decoding as a separate object: the policy layer. Temperature moves along the Gibbs family (with proved limits), top-$k$ and top-$p$ are conditioning operations on the model's own distribution (Remark 7.2), and the prefill/decode split previews the systems reality of serving. The parameters never change; everything in the chapter is about *using* them.

## 7. Generation And Evaluation

### Mathematical object

**Why this chapter.** If you have ever set `temperature=0.7`, `top_p=0.9`, or `top_k=40` in an API call and wondered what you were actually doing — this chapter is those three fields. The trained model supplies distributions; it does not supply text. Between the two sits a policy layer of pure inference-time decisions, and every knob in it has an exact mathematical identity that this chapter derives.

After training, the model defines conditional distributions $p_\theta(\,\cdot\mid x_{1:t}) \in \mathring\Delta^{V-1}$. Generation uses these conditionals to sample a sequence. Starting from a prompt $x_{1:m}$, repeat:

1. compute logits $z \in \mathbb{R}^{V}$ for the final position,
2. transform logits into a probability vector,
3. draw or choose the next token,
4. append it to the context.

The parameters $\theta$ do not change during generation: decoding is a map from a trained model plus a *policy* to a stochastic process on $\mathcal{V}$, and everything in this chapter modifies the policy, never the parameterized family.

### Prefill versus decoding

A production decoder-only model separates inference into two phases.

**Prefill** processes the prompt tokens in parallel and builds the per-layer KV cache. If the prompt has length $m$, the model computes keys and values for those $m$ positions once. This phase is still causal — token $i$ cannot attend to token $j>i$ — but the whole prompt is known, so it packs into matrix operations.

**Decoding** generates one token at a time: compute the new token's query, key, and value; append key and value to the cache; attend the query over cached positions; sample; repeat.

The toy `generate` function does not implement a KV cache. It recomputes the model over the cropped context each step because that is simpler for teaching. The mathematical contract is the same conditional distribution; the systems contract differs, since real serving avoids recomputing past keys and values.

### Temperature

Temperature rescales logits before softmax, producing the Gibbs family of Theorem 1.6:

$$
p(\tau) = \sigma(z/\tau) \in \mathring\Delta^{V-1}, \qquad \tau>0 .
$$

**Proposition 7.1 (temperature limits).** Let $z\in\mathbb{R}^V$ and $p(\tau) = \sigma(z/\tau)$.

(i) As $\tau\to\infty$, $p(\tau) \to$ the uniform distribution $(1/V,\ldots,1/V)$.

(ii) As $\tau\to 0^+$, $p(\tau) \to$ the uniform distribution on the argmax set $\mathcal{M} = \{i : z_i = \max_j z_j\}$; in particular, if the maximum is unique, $p(\tau)\to e_{i^\ast}$, the point mass on the argmax.

*Proof.* Both follow from the ratio $p_i(\tau)/p_j(\tau) = e^{(z_i-z_j)/\tau}$. (i) As $\tau\to\infty$ every ratio tends to $e^0 = 1$; a normalized vector with all ratios $1$ is uniform. (ii) Take $i\in\mathcal{M}$ and any $j\notin\mathcal{M}$: then $z_i - z_j > 0$, so the ratio $p_j/p_i = e^{-(z_i-z_j)/\tau} \to 0$, and mass concentrates on $\mathcal{M}$; within $\mathcal{M}$ all ratios equal 1. $\blacksquare$

Between the limits, lowering $\tau$ sharpens (raises the largest probabilities, lowers entropy) and raising $\tau$ flattens; the entropy $H(p(\tau))$ increases monotonically in $\tau$ from $\log|\mathcal{M}|$ to $\log V$.

### Greedy and beam decoding

The simplest deterministic decoding rule is greedy search, the $\tau\to 0$ limit of Proposition 7.1:

$$
\hat{x}_{t+1}=\arg\max_i\; p_\theta(i\mid x_{1:t}).
$$

Greedy search is fast, but a locally best token can lead to a poor full sequence. Beam search keeps the top $K$ partial sequences by accumulated log probability:

$$
\log p(y_{1:i}\mid x) = \log p(y_{<i}\mid x) + \log p(y_i\mid x,y_{<i}).
$$

Beam search is a search procedure over sequence prefixes. Top-k and top-p are different: they modify the next-token distribution and then sample.

### Truncation as conditioning

**Remark 7.2 (top-k and top-p are conditional distributions).** Both truncation rules select a subset $S\subseteq\mathcal{V}$ of allowed tokens and sample from the renormalized restriction

$$
p(i \mid i \in S) = \frac{p_i\,\mathbf{1}[i\in S]}{\sum_{j\in S} p_j},
$$

i.e. from the model's own distribution *conditioned* on the event that the token falls in $S$ — geometrically, projection of $p$ onto a face of the simplex followed by renormalization. Top-k takes $S$ = the $k$ highest-probability tokens (implemented by setting all other logits to $-\infty$, the additive-mask idiom of Definition 4.1 applied to the vocabulary axis). Top-p (nucleus) sorts tokens by probability and takes the smallest prefix whose cumulative mass exceeds $p$, so $|S|$ adapts to the distribution's shape: a peaked distribution keeps few tokens, a flat one keeps many.

### Categorical sampling

After filtering and softmax, generation samples from a categorical distribution over token IDs. In PyTorch, the toy code uses `torch.multinomial(probs, num_samples=1)`.

### Code path

`src/llm_from_scratch/generate.py` contains:

- `top_k_filter`,
- `top_p_filter`,
- `sample_next_token`,
- `generate`.

The `generate` function crops context to `model.config.block_size`, runs the model, takes `logits[:, -1, :]`, samples one next token, and concatenates it to `idx`.

> **Mathematician's lens.** Decoding is a procedure for drawing a path through a sequence of conditional categorical distributions. Temperature moves along the Gibbs family (Proposition 7.1); truncation conditions on a subset (Remark 7.2). Neither alters the learned parameterized family.

> **ML practitioner's warning.** Decoding settings can hide or amplify model weaknesses. A better sample does not mean the model learned a better distribution; it may only mean the sampling procedure avoided low-quality regions.

### Self-checks

1. Why does `generate` use only `logits[:, -1, :]` at each step?
2. What happens to probabilities of filtered-out tokens after setting logits to $-\infty$?
3. Why does changing temperature not require backpropagation?
4. (*) Prove the entropy monotonicity claim after Proposition 7.1 for $V=2$: show $\frac{d}{d\tau}H(p(\tau)) > 0$ when $z_1\neq z_2$.


### Decoding Policies: Greedy, Sampling, Temperature, And Truncation

Training defines a conditional distribution. Generation requires a policy for turning that distribution into the next token.

| Policy | Rule | Benefit | Failure mode |
| --- | --- | --- | --- |
| Greedy decoding | choose `argmax` token | deterministic and simple | can be repetitive and brittle |
| Multinomial sampling | sample from the full softmax distribution | respects model uncertainty | can sample low-quality tail tokens |
| Temperature sampling | divide logits by $\tau$ before softmax | controls sharpness/entropy | bad settings distort behavior |
| Top-k sampling | keep only the `k` highest-logit tokens | removes low-probability tail | fixed `k` may be too narrow or too broad |
| Top-p sampling | keep the smallest set whose mass exceeds `p` | adapts to distribution shape | still needs tuning and evaluation |

Temperature is especially easy to connect to the softmax math. If logits are $z$, generation samples from

$$
\operatorname{softmax}(z/\tau).
$$

For $0 < \tau < 1$, large logits become larger relative to the rest, so the distribution sharpens. For $\tau > 1$, differences shrink and the distribution flattens. The model parameters have not changed. Only the decoding policy has changed.

This is why qualitative samples are not a pure model metric. They depend on prompt, random seed, temperature, truncation, stopping rules, and the model itself.

Source note: this follows SLP3 Chapter 7 and CS124's LLM lecture on greedy decoding, random sampling, temperature, and the PA6a `generate` implementation.


In [ ]:
from llm_from_scratch.generate import generate
from llm_from_scratch.evaluate import perplexity

set_seed(123)
gen_cfg = ModelConfig(vocab_size=20, block_size=8, n_embd=16, n_head=4, n_layer=1, dropout=0.0)
gen_model = TinyGPT(gen_cfg)
prompt = torch.zeros((1, 3), dtype=torch.long)
out_ids = generate(gen_model, prompt, max_new_tokens=5, temperature=1.0, top_k=5)

assert out_ids.shape == (1, 8)
perplexity_of_one_nat = perplexity(1.0)
out_ids, perplexity_of_one_nat


**Worked Example 7.1 (temperature and truncation on real digits).** Reuse the logit vector of Worked Example 1.1, $z = (2, 0, -1, 1)$ over $(\texttt{' '}, \texttt{a}, \texttt{c}, \texttt{t})$, and sweep the Gibbs family $p(\tau) = \sigma(z/\tau)$:

| $\tau$ | $p_{\texttt{' '}}$ | $p_{\texttt{a}}$ | $p_{\texttt{c}}$ | $p_{\texttt{t}}$ | entropy $H$ (nats) |
| --- | --- | --- | --- | --- | --- |
| $0.5$ | $0.8650$ | $0.0158$ | $0.0021$ | $0.1171$ | $0.4554$ |
| $1$ | $0.6439$ | $0.0871$ | $0.0321$ | $0.2369$ | $0.9475$ |
| $2$ | $0.4551$ | $0.1674$ | $0.1015$ | $0.2760$ | $1.2451$ |
| $\infty$ | $0.25$ | $0.25$ | $0.25$ | $0.25$ | $\ln 4 = 1.3863$ |

The entropy column climbs monotonically toward $\ln V$, and the mass visibly migrates from the argmax to the tail — Proposition 7.1 as a table.

**Truncation.** Top-$k$ with $k = 2$ keeps $S = \{\texttt{' '}, \texttt{t}\}$ (logits $2$ and $1$). Remark 7.2's conditional distribution renormalizes their probabilities $0.6439, 0.2369$ to

$$
p(\,\cdot \mid S) = \left(\frac{0.6439}{0.8808},\ \frac{0.2369}{0.8808}\right) = (0.7311,\ 0.2689) = \sigma\big((2, 1)\big).
$$

The last equality is worth noticing: renormalizing a softmax over a subset is *the same* as taking softmax of the surviving logits — conditioning commutes with the Gibbs map. What the model believed about $\texttt{' '}$ versus $\texttt{t}$ is exactly preserved; only the tail was deleted.

In [ ]:
import torch
from llm_from_scratch.functional import stable_softmax
from llm_from_scratch.generate import top_k_filter

# Worked Example 7.1 — temperature sweep and top-k conditioning
z = torch.tensor([[2.0, 0.0, -1.0, 1.0]])

for tau, expected_H in [(0.5, 0.4554), (1.0, 0.9475), (2.0, 1.2451)]:
    p = stable_softmax(z / tau, dim=-1)
    H = -(p * p.log()).sum().item()
    assert abs(H - expected_H) < 5e-5
    print(f"tau={tau}: p={[round(v, 4) for v in p[0].tolist()]}  H={H:.4f}")

# top-k=2 keeps logits (2, 1); conditioning commutes with softmax:
p_trunc = stable_softmax(top_k_filter(z.clone(), k=2), dim=-1)
assert torch.allclose(p_trunc[0, [0, 3]], stable_softmax(torch.tensor([[2.0, 1.0]]), dim=-1)[0], atol=1e-6)
assert p_trunc[0, 1] == p_trunc[0, 2] == 0.0
print("top-2:", [round(v, 4) for v in p_trunc[0].tolist()], " = softmax of surviving logits")

### Why Qualitative Samples Can Mislead

Generated text is useful for intuition, but it is noisy evidence. A few good samples do not prove that a model has a good distribution. A few bad samples do not fully characterize it either. Sampling is stochastic, prompt-sensitive, and decoding-sensitive.

For the tiny curriculum model, generated text will often look poor. That is expected. The model is small, the data are tiny, and the training budget is minimal. The goal is to understand the mechanics, not to produce a useful assistant.

### Evaluation hierarchy

A rigorous evaluation habit separates several questions:

| Question | Example evidence | Failure mode |
| --- | --- | --- |
| Does the code run? | notebook execution, tests | says little about learning quality |
| Does the objective decrease? | training loss | can reflect memorization or leakage |
| Does held-out likelihood improve? | validation loss, perplexity | may not match downstream behavior |
| Does task behavior improve? | task-specific metrics | can overfit benchmark format |
| Does deployment behavior hold? | monitoring, red-team tests, latency checks | distribution shift and system effects |

The toy repo mostly covers the first three. Modern LLM practice adds the rest.

> **Mathematician's lens.** A sample is one realization from a distribution. Estimating properties of the distribution requires repeated measurements and a defined statistic.

> **ML practitioner's warning.** Cherry-picked generations are not evaluation. Always ask how prompts were selected, how many samples were tried, and what metric or rubric was fixed before looking at outputs.


### Where We Stand

**What we now have.** A generator. Model plus decoding policy yields text, and the policy knobs are now typed objects: temperature is the Gibbs parameter (τ→0 argmax, τ→∞ uniform, Proposition 7.1), truncation is conditioning on a subset (Worked Example 7.1 showed conditioning *commutes* with softmax), and qualitative samples were demoted to what they are — single draws, not evaluations.

> **Definition so far.** *An autoregressive language model is a trained conditional distribution wrapped in a sampling policy: a stochastic process on $\mathcal{V}$ whose law is the model's but whose behavior is co-authored by decoding choices that touch no parameters.*

**What is still missing.** The right *distribution to imitate*. Our generator continues text the way the corpus talks — ask it a question and it may answer with another question, because completion, not assistance, is what next-token training on raw text rewards. Changing behavior means changing what the empirical risk averages over.

**Where Chapter 8 goes.** Supervised fine-tuning as a change of measure: instruction-formatted data, and a loss mask that makes prompt tokens context-only while response tokens carry the gradient (Definition 8.1). Same architecture, same loss function pointwise — different measure, different behavior.

## 8. Toy Supervised Fine-Tuning

### Mathematical object

Supervised fine-tuning changes the data distribution and often the formatting. Instead of raw text windows alone, examples may have instruction-response structure:

```text
### Instruction:
<user task>

### Response:
<desired answer>
```

The model is still trained with next-token prediction. What changes is which sequences are sampled and which token positions are included in the loss.

**Definition 8.1 (masked empirical risk).** Let $m \in \{0,1\}^{B\times T}$ be a loss mask. The masked empirical risk is

$$
L_m(\theta)
= \frac{\sum_{b,t} m_{b,t}\,\big[-\log p_\theta(y_{b,t}\mid x_{b,1:t})\big]}
{\sum_{b,t} m_{b,t}},
$$

i.e. the expectation of the per-token loss under the reweighted empirical measure

$$
\mu_m = \frac{\sum_{b,t} m_{b,t}\,\delta_{(b,t)}}{\sum_{b,t} m_{b,t}}
$$

on the set of token positions. Ordinary cross-entropy is the case $m\equiv 1$, where $\mu_m$ is uniform.

Prompt positions get mask $0$, response positions mask $1$: the model conditions on the instruction (those tokens are still in context and still attended to) but spends its loss budget only on generating the response. Masking changes the *measure* in the empirical average; the model class and the pointwise loss are untouched.

### Dimensions and tensor shapes

For a batch of formatted examples:

```python
x.shape == (B, T)
y.shape == (B, T)
loss_mask.shape == (B, T)
per_token_loss.shape == (B, T)
```

The mask has the same token-position shape as targets. It weights which next-token losses contribute to the scalar objective.

### Statistical role

SFT does not create a new architecture. It changes the empirical distribution and the target behavior. The model sees examples where the desired continuation has the style and structure of an answer; masking makes the objective precise — condition on the instruction, optimize the response. The denominator in Definition 8.1 matters: without it, examples with more unmasked tokens would contribute larger total loss simply by length.

### Code path

The repo keeps SFT as a toy orientation topic. `toy_instruction_examples()` returns a few prompt-response pairs. The notebook formats them and then demonstrates masked averaging directly with tensors. A production SFT pipeline would need a tokenizer, special tokens, padding, sequence packing, mask construction, validation splits, and careful data audits.

> **Mathematician's lens.** Loss masking is a change of measure — multiplication by an indicator inside the empirical average, then renormalization. The model class is unchanged; the objective's support is not.

> **ML practitioner's warning.** Formatting is part of the training signal. Inconsistent templates, wrong masks, or response leakage can teach brittle behavior even when the loss decreases.

### Self-checks

1. If every mask value is 1, how does the masked loss of Definition 8.1 relate to ordinary cross-entropy?
2. Why might prompt tokens be included in context but excluded from loss?
3. What happens if the response mask is shifted by one position incorrectly?


In [ ]:
from llm_from_scratch.data import toy_instruction_examples

examples = toy_instruction_examples()
formatted = []
for prompt, response in examples[:2]:
    formatted.append(f"### Instruction:\n{prompt}\n\n### Response:\n{response}")

formatted


### Loss Masking Intuition

The code cell below uses fixed per-token losses so the masking formula is visible. With

```text
per_token_loss = [2.0, 2.0, 1.0, 0.5, 0.25]
loss_mask      = [0,   0,   1,   1,   1]
```

the masked average is

$$
\frac{1 + 0.5 + 0.25}{3}.
$$

The first two token positions still exist in the sequence. The model may attend to them if they are in context. They simply do not contribute to the scalar loss.

### Weighted empirical risk

More generally, masks need not be only 0 or 1 (Definition 8.1 extends verbatim). A weighted loss could use $m_{b,t}\ge 0$:

$$
L(\theta)
= \frac{\sum_{b,t}m_{b,t}\ell_{b,t}(\theta)}{\sum_{b,t}m_{b,t}}.
$$

Binary prompt-response masking is the simplest case. The denominator matters: without it, examples with more unmasked tokens would automatically contribute larger total loss.

### Connection to implementation practice

PyTorch's built-in cross-entropy can ignore a special target index, or you can compute unreduced per-token losses and multiply by a mask. The toy notebook uses the second idea directly because it makes the mathematics explicit.

> **Mathematician's lens.** Masking is multiplication by an indicator function inside an empirical integral or sum.

> **ML practitioner's warning.** Always inspect masks on decoded examples. A correct-looking tensor shape does not prove the right tokens are supervised.


In [ ]:
targets = torch.tensor([[5, 6, 7, 8, 9]])
loss_mask = torch.tensor([[0, 0, 1, 1, 1]], dtype=torch.float32)
per_token_loss = torch.tensor([[2.0, 2.0, 1.0, 0.5, 0.25]])
masked_average = (per_token_loss * loss_mask).sum() / loss_mask.sum()

assert abs(masked_average.item() - ((1.0 + 0.5 + 0.25) / 3)) < 1e-6
masked_average.item()


### Where We Stand

**What we now have.** The two-stage recipe in miniature: pretrain on raw text for the language prior, then fine-tune on formatted instruction–response pairs with a mask that spends the loss budget only where behavior should change. The mask is mathematically slight — multiplication by an indicator, renormalized — and behaviorally decisive.

> **Definition so far.** *An autoregressive language model is a pretrained conditional distribution re-shaped by a reweighted empirical measure — an imitator of curated demonstrations, not just of the raw corpus.*

**What is still missing.** Contact with practice. Everything so far was built from scratch precisely so it could be understood; but the working ecosystem runs on `datasets`, `tokenizers`, and `AutoModelForCausalLM`, and someone who knows only our from-scratch objects still cannot read production code — or know which contracts the libraries silently handle.

**Where Chapter 9 goes.** The translation table: each hand-built object mapped to its Hugging Face/PyTorch abstraction, the questions to ask any library model (who shifts the labels? which positions are ignored?), and the encoder/decoder/encoder–decoder family tree so that `AutoModel` names stop being opaque.

## 9. Translation To Hugging Face And PyTorch Abstractions

After building from scratch, library abstractions become easier to interpret. The point is not that libraries are unimportant. The point is that abstractions are safer when you know the contracts they hide.

| From scratch | Library abstraction | Mathematical or tensor contract hidden |
| --- | --- | --- |
| text list | `datasets.Dataset` | finite sample from a data distribution; split and map operations |
| `CharTokenizer` or BPE helper | `Tokenizer` / `AutoTokenizer` | text-to-ID map, vocabulary, special tokens, offsets |
| `ModelConfig` | `GPT2Config` or another config object | dimensions `V`, `T`, `C`, `H`, number of layers |
| `TinyGPT` | `AutoModelForCausalLM` | embedding, blocks, logits, optional loss |
| manual loop | `Trainer` or Accelerate loop | batching, optimizer, scheduler, device placement, checkpointing |
| manual sampling | `.generate()` | temperature, top-k, top-p, beams, stopping criteria |

### What should transfer from the toy implementation

When you use a library model, the same questions still matter:

1. What shape are `input_ids`, `attention_mask`, and labels?
2. Does the model shift labels internally, or must the data collator do it?
3. Which positions are ignored in the loss?
4. What does the model return: logits, loss, hidden states, attentions?
5. Which generation settings modify decoding but not parameters?
6. Is evaluation measuring likelihood, task accuracy, preference, latency, or something else?

### Code path

The next code cell instantiates a tiny Hugging Face `GPT2LMHeadModel` and a `datasets.Dataset`. It does not train a real model. It is a translation check: the objects are larger abstractions around the same ideas already built by hand.

> **Mathematician's lens.** A library config is a compact specification of dimensions and module choices. It is not a replacement for the underlying map from token sequences to conditional distributions.

> **ML practitioner's warning.** High-level APIs can silently handle shifting, masking, padding, and generation defaults. Read the shape and loss contracts before trusting results.


### Encoder, Decoder, And Encoder-Decoder Families

This notebook builds a decoder-only, GPT-style causal language model. That is one transformer family, not the whole transformer family.

| Family | Attention pattern | Typical objective | Common use |
| --- | --- | --- | --- |
| Encoder-only | bidirectional self-attention over the input | masked-token prediction or representation learning | classification, retrieval encoders, tagging features |
| Decoder-only | causal self-attention over the prefix | next-token prediction | completion, chat base models, autoregressive generation |
| Encoder-decoder | encoder self-attention plus decoder causal self-attention and cross-attention | conditional generation with teacher forcing | translation, summarization, input-conditioned generation |

The distinction is about which information each position is allowed to use. A decoder-only LM predicts the next token from the prefix, so future tokens must be masked. An encoder-only masked LM can look left and right because it is not trained to generate left-to-right continuations in the same way. An encoder-decoder model first represents a source sequence, then lets the decoder generate a target sequence while attending back to the encoded source.

When you see `AutoModelForCausalLM`, assume the decoder-only contract unless the model documentation says otherwise. When you see BERT-style encoders, do not assume they can be used as drop-in autoregressive generators. When you see translation models, look for the encoder outputs, decoder inputs, label shifting, and cross-attention boundary.

Source note: this map comes from the transformer, masked-LM, and machine-translation sources in `../data/cs124/slp3_chapters/8.pdf`, `../data/cs124/slp3_chapters/10.pdf`, `../data/cs124/slp3_chapters/12.pdf`, `../data/cs124/slp3_slides/transformer_jan26.pdf`, and `../data/cs124/slp3_slides/mlmjan25.pdf`.


In [ ]:
from datasets import Dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from transformers import GPT2Config, GPT2LMHeadModel

hf_dataset = Dataset.from_dict({"text": ["attention reads earlier tokens", "loss trains logits"]})
hf_config = GPT2Config(vocab_size=128, n_positions=32, n_embd=64, n_layer=2, n_head=4)
hf_model = GPT2LMHeadModel(hf_config)

len(hf_dataset), Tokenizer(BPE()).get_vocab_size(), hf_model.num_parameters()


### Where We Stand

**What we now have.** Bilinguality. Every from-scratch object now has a library name — `ModelConfig` is a `GPT2Config`, `TinyGPT` is an `AutoModelForCausalLM`, the training loop is a `Trainer` — and, more usefully, the *contracts* those abstractions hide (label shifting, ignored indices, generation defaults) are things we built ourselves and can interrogate.

> **Definition so far.** *An autoregressive language model is the same typed composition, now recognized inside industrial wrappers whose conveniences are theorems we have already proved.*

**What is still missing.** Deployability. A real model's weights at float32 are gigabytes; its KV cache grows linearly in context and dominates serving memory; and none of our mathematics yet says how to trade precision for footprint — or what that trade costs.

**Where Chapter 10 goes.** Quantization from first principles: the affine map onto a finite grid (Definition 10.1), the half-step error bound proved and *achieved* (Lemma 10.2, Worked Example 10.1), per-channel remedies for outliers, and the KV-cache arithmetic — including the MQA/GQA head-sharing dial — that decides what fits on a card.

## 10. Quantization: Lower Precision Representations

### Mathematical object

**Why this chapter.** If you have downloaded a model file whose name ends in `Q4_K_M` or `int8`, you have already used this chapter's subject: the same weights, stored at a quarter of the memory, with an error you are trusting someone to have controlled. Here is what the 4 means, where the error comes from, exactly how large it can be — a provable bound, achieved in Worked Example 10.1 — and why the KV cache, not the weights, is often what actually decides whether a context fits.

Quantization replaces a continuum by a finite grid and studies the error of the projection.

**Definition 10.1 (affine quantizer).** Fix a scale $s>0$, a zero-point $z\in\mathbb{Z}$, and an integer range $\mathcal{Q} = \{q_{\min}, q_{\min}+1,\ldots, q_{\max}\}$. The quantizer and dequantizer are the maps

$$
Q_{s,z}: \mathbb{R} \longrightarrow \mathcal{Q},
\qquad
Q_{s,z}(x) = \operatorname{clip}\!\Big(\operatorname{round}\Big(\frac{x}{s} + z\Big),\, q_{\min},\, q_{\max}\Big),
$$

$$
D_{s,z}: \mathcal{Q} \longrightarrow \mathbb{R},
\qquad
D_{s,z}(q) = s\,(q-z),
$$

applied elementwise to tensors. The reconstruction is $\hat{x} = (D_{s,z}\circ Q_{s,z})(x)$. The image of $D_{s,z}$ is the arithmetic grid $\{s(q-z): q\in\mathcal{Q}\}\subset\mathbb{R}$ with spacing $s$; the scale is the bin width in real units, and the zero-point chooses which integer represents real zero (so that $x = 0$ is exactly representable).

**Lemma 10.2 (half-step error bound).** If $x$ lies in the representable interval,

$$
x \in \big[\, s(q_{\min}-z),\; s(q_{\max}-z) \,\big],
$$

so that no clipping occurs, then

$$
|x - \hat{x}| \;\le\; \frac{s}{2}.
$$

Outside this interval, $\hat x$ is the nearest endpoint and the error grows linearly in the distance to the interval.

*Proof.* Without clipping, $q = \operatorname{round}(x/s + z)$, and rounding to the nearest integer satisfies $\big|\tfrac{x}{s} + z - q\big| \le \tfrac{1}{2}$. Multiply by $s$:

$$
\big|x - s(q - z)\big| = \big|x - \hat x\big| \le \frac{s}{2}.
$$

For $x$ above the interval, $q = q_{\max}$ is forced, $\hat x = s(q_{\max}-z)$ is the largest grid point, and the error is $x - \hat x$, growing linearly; symmetrically below. $\blacksquare$

The lemma exposes the fundamental tradeoff through the single parameter $s$: covering a wide range $[x_{\min},x_{\max}]$ with a fixed number of levels forces a large $s$ (coarse resolution, larger worst-case rounding error $s/2$ for *every* value); a small $s$ gives fine resolution but clips outliers, whose error is unbounded by $s/2$. One extreme outlier in a tensor therefore degrades the representation of all typical values — the practical reason for per-channel schemes below.

### Choosing scale and zero-point

For asymmetric unsigned quantization with `num_bits` $= b$, one takes $q_{\min}=0$, $q_{\max}=2^b - 1$. Given the observed range $[x_{\min},x_{\max}]$, the toy helper uses

$$
s = \frac{x_{\max}-x_{\min}}{q_{\max}-q_{\min}},
\qquad
z = \operatorname{round}\!\Big(q_{\min} - \frac{x_{\min}}{s}\Big),
$$

then clips $z$ into the integer range. This makes the representable interval of Lemma 10.2 coincide with the observed range. Symmetric quantization instead fixes $z=0$ and uses the maximum absolute value; this is convenient for signed integer arithmetic but may waste levels when the real range is very asymmetric.

### Per-tensor versus per-channel

Per-tensor quantization uses one $(s,z)$ pair for the whole tensor; per-channel uses a separate pair along a chosen axis — a *product of quantizers*, one per slice. For a weight matrix whose rows have very different ranges, per-channel scales let each row meet Lemma 10.2 with its own small $s$, often reducing error substantially at the cost of metadata and kernel complexity.

### KV-cache memory formula

During autoregressive decoding, transformers cache keys and values for previous tokens. The cache size is a pure shape product:

$$
\text{bytes}
= 2 \cdot L \cdot H \cdot D \cdot T \cdot B \cdot (\text{bytes per value}),
$$

where the factor 2 counts keys and values, $L$ is the number of layers, $H$ heads, $D$ head dimension, $T$ cached length, $B$ batch size. The repo helper `estimate_kv_cache_bytes` implements exactly this product.

Axis-level interpretation: a standard multi-head cache stores keys and values for every layer, head, cached token, and head dimension, so its shape cost is $O(L H D T)$. Multi-query attention (MQA) shares one key/value set across all query heads, reducing the head factor to 1. Grouped-query attention (GQA) uses $G$ key/value groups:

$$
O(L\cdot G\cdot D\cdot T), \qquad 1\le G\le H,
$$

interpolating between MQA ($G=1$) and full MHA ($G=H$). This is a serving-time memory and bandwidth tradeoff, not a change to the next-token loss.

### Code path

`src/llm_from_scratch/quantization.py` contains:

- `QuantizationParams`,
- `quantize_tensor`,
- `dequantize_tensor`,
- `fake_quantize_tensor`,
- `quantize_per_channel`,
- `quantization_error`,
- `estimate_kv_cache_bytes`.

The next code cell quantizes a tiny tensor, dequantizes it, measures error, and computes a KV-cache estimate.

> **Mathematician's lens.** Quantization is projection onto a finite lattice: Definition 10.1 fixes the lattice, Lemma 10.2 bounds the projection error, and all engineering choices (bit width, symmetric/asymmetric, per-channel granularity) are choices of lattice and of how the error budget is distributed between rounding and clipping.

> **ML practitioner's warning.** Quantization quality depends on tensor distributions, calibration data, hardware kernels, and which tensors are quantized. A lower bit-width is not automatically faster or acceptable.

### Self-checks

1. If `num_bits=4` for unsigned affine quantization, what are `qmin` and `qmax`?
2. Using Lemma 10.2, explain quantitatively why one outlier increases rounding error for many non-outlier values in per-tensor quantization.
3. In the KV-cache formula, why is there a factor of 2?
4. (*) Show that with the scale choice above, the representable interval of Lemma 10.2 equals $[x_{\min}, x_{\max}]$ exactly when $z$ needs no clipping.


In [ ]:
from llm_from_scratch.quantization import (
    quantize_tensor,
    dequantize_tensor,
    quantization_error,
    estimate_kv_cache_bytes,
)

x = torch.tensor([-1.0, -0.1, 0.0, 0.5, 1.0])
q, params = quantize_tensor(x, num_bits=8)
x_hat = dequantize_tensor(q, params)
err = quantization_error(x, x_hat)
kv_bytes = estimate_kv_cache_bytes(batch_size=1, n_layer=2, n_head=4, seq_len=128, head_dim=16, bytes_per_value=2)

assert err["mae"] < 0.01
q, x_hat, err, kv_bytes


**Worked Example 10.1 (3-bit quantization, grid and errors explicit).** Quantize $x = (-1,\ -0.1,\ 0,\ 0.5,\ 1)$ with an asymmetric 3-bit quantizer: $q_{\min} = 0$, $q_{\max} = 7$. The calibration of Definition 10.1 gives

$$
s = \frac{x_{\max} - x_{\min}}{q_{\max} - q_{\min}} = \frac{2}{7} = 0.2857,
\qquad
z = \operatorname{round}\!\Big(0 - \frac{-1}{2/7}\Big) = \operatorname{round}(3.5) = 4,
$$

(round-half-to-even sends $3.5 \to 4$). The grid is $\{s(q - 4) : q = 0,\ldots,7\}$ — eight points from $-1.1429$ to $0.8571$, spaced $0.2857$ apart. Quantize each value via $q = \operatorname{round}(x/s + 4)$:

| $x$ | $x/s + z$ | $q$ | $\hat x = s(q-4)$ | $\lvert x - \hat x\rvert$ |
| --- | --- | --- | --- | --- |
| $-1.0$ | $0.5$ | $0$ | $-1.1429$ | $0.1429 = s/2$ |
| $-0.1$ | $3.65$ | $4$ | $0$ | $0.1$ |
| $0$ | $4$ | $4$ | $0$ | $0$ |
| $0.5$ | $5.75$ | $6$ | $0.5714$ | $0.0714$ |
| $1.0$ | $7.5$ | $7$ | $0.8571$ | $0.1429 = s/2$ |

Every error respects Lemma 10.2's bound $s/2 = 0.1429$, and the endpoints *achieve* it — the bound is tight. (The first row is a nice trap: $0.5$ rounds to $0$, not $1$, because both PyTorch and IEEE round halves to even; the reconstruction still lands within half a grid step.) Note also that real zero is exactly representable ($q = 4 \mapsto \hat x = 0$), which is what the zero-point exists to guarantee.

In [ ]:
import torch
from llm_from_scratch.quantization import quantize_tensor, dequantize_tensor

# Worked Example 10.1 — 3-bit asymmetric quantization, verified against the hand table
x = torch.tensor([-1.0, -0.1, 0.0, 0.5, 1.0])
q, params = quantize_tensor(x, num_bits=3)

assert (params.qmin, params.qmax) == (0, 7)
assert abs(params.scale - 2 / 7) < 1e-9
assert params.zero_point == 4
assert q.tolist() == [0, 4, 4, 6, 7]

x_hat = dequantize_tensor(q, params)
err = (x - x_hat).abs()
assert torch.allclose(x_hat, torch.tensor([-1.1429, 0.0, 0.0, 0.5714, 0.8571]), atol=5e-5)
assert (err <= params.scale / 2 + 1e-6).all()                     # Lemma 10.2, verified
assert abs(err.max().item() - params.scale / 2) < 1e-6            # ...and the bound is achieved
print("q     :", q.tolist())
print("x_hat :", [round(v, 4) for v in x_hat.tolist()])
print("errors:", [round(v, 4) for v in err.tolist()], f"  bound s/2 = {params.scale/2:.4f}")

**Worked Example 10.2 (the toy model's KV cache, by hand).** Evaluate the cache formula for the exact configuration trained in the visualization pipeline — $L = 2$ layers, $H = 4$ heads, $D = 8$ per head, context $T = 16$, batch $B = 1$, fp16 ($2$ bytes per value):

$$
\text{bytes} = 2 \cdot L \cdot H \cdot D \cdot T \cdot B \cdot 2
= 2 \cdot 2 \cdot 4 \cdot 8 \cdot 16 \cdot 1 \cdot 2
= 4096 = 4\,\text{KB}.
$$

Four kilobytes — nothing. Now do a 7B-class model in the same formula: $L = 32$, $H = 32$, $D = 128$, $T = 8192$, fp16 gives $2 \cdot 32 \cdot 32 \cdot 128 \cdot 8192 \cdot 2 \approx 4.3\,\text{GB}$ — *per sequence in the batch*, before a single weight is loaded. The formula did not change; the shape factors did. This is why the head-sharing dial (MQA/GQA) and cache quantization exist, and the interactive calculator below Chapter 12 lets you feel every factor.

In [ ]:
from llm_from_scratch.quantization import estimate_kv_cache_bytes

# Worked Example 10.2 — cache bytes: toy config by hand, then a 7B-class config
toy = estimate_kv_cache_bytes(n_layer=2, n_head=4, head_dim=8,
                              seq_len=16, batch_size=1, bytes_per_value=2)
assert toy == 2 * 2 * 4 * 8 * 16 * 1 * 2 == 4096
print(f"toy model cache: {toy} bytes = {toy // 1024} KB")

big = estimate_kv_cache_bytes(n_layer=32, n_head=32, head_dim=128,
                              seq_len=8192, batch_size=1, bytes_per_value=2)
print(f"7B-class, T=8192, fp16: {big / 2**30:.2f} GB per sequence")

### Per-Tensor Versus Per-Channel Quantization

Per-tensor quantization uses one scale for the whole tensor. Per-channel quantization uses separate scales along a chosen axis. The axis decision is a shape decision, just like choosing the softmax axis in attention.

Suppose a weight tensor has rows with very different ranges. A single scale must cover the largest range, so small-range rows get coarse resolution. Per-channel quantization can give each row its own scale:

$$
q_{i,j} = \operatorname{round}\left(\frac{x_{i,j}}{s_i}+z_i\right).
$$

This often reduces reconstruction error, especially for weight matrices. The cost is extra scale and zero-point metadata and possibly more complex kernels.

### Practical tradeoffs

| Method | Benefit | Cost |
| --- | --- | --- |
| Per-tensor | simple metadata and kernels | sensitive to outliers and mixed ranges |
| Per-channel | lower error when channels have different ranges | more metadata and axis-specific implementation |
| Groupwise | compromise between the two | group size becomes another hyperparameter |

The toy helper `quantize_per_channel(x, axis, num_bits)` makes the axis explicit. It iterates over slices along the chosen axis and applies `quantize_tensor` to each slice.

> **Mathematician's lens.** Per-channel quantization uses a product of smaller quantizers instead of one global quantizer for the whole tensor.

> **ML practitioner's warning.** Quantization reports should state bit-width, symmetric/asymmetric choice, calibration method, which tensors were quantized, and the hardware path. Otherwise comparisons are underspecified.


### Where We Stand

**What we now have.** The numerics layer. Real weights live on finite grids with a provable, achievable error bound; the scale parameter arbitrates rounding against clipping; and cache memory is a closed-form product of shape factors we can evaluate by hand (Worked Example 10.2). The quantization panel shows the grid coarsening live as bits drop.

> **Definition so far.** *An autoregressive language model is a trained, policy-wrapped composition whose tensors are stored and moved at finite precision — an object with a memory footprint and a bandwidth budget, not just a mathematical signature.*

**What is still missing.** Everything that surrounds the model in a product: alignment beyond imitation (what if good behavior is easier to *compare* than to demonstrate?), prompting and retrieval, evaluation that matches claims, and the serving machinery of prefill, batching, and caches at scale.

**Where Chapter 11 goes.** The modern practice map — and the Gibbs principle's third appearance: the KL-regularized alignment objective is solved by exponentially tilting the reference policy (Proposition 11.1), and DPO falls out by inverting that formula. Pretraining, SFT, preference optimization, RAG, and serving each classified by *which object they change*.

## 11. Modern LLM Practice: What Gets Added Around The Core Model

The toy GPT path teaches the central language-modeling mechanism: tokenize text, embed IDs, apply causal transformer blocks, produce logits, optimize next-token cross-entropy, and decode from the resulting conditional distribution. Modern LLM systems add objectives, data pipelines, evaluation procedures, retrieval systems, compression methods, and serving constraints around that core.

### Pretraining

Pretraining fits a broad next-token model on a large text distribution. The objective is still average negative log-likelihood,

$$
\mathbb{E}_{x\sim \mathcal{D}_{\text{pretrain}}}\Big[\sum_t -\log p_\theta(x_{t+1}\mid x_{1:t})\Big],
$$

whose population decomposition is Theorem 1.9. What changes is scale: data volume, model size, training time, distributed systems, filtering, deduplication, and evaluation.

### Supervised fine-tuning

SFT changes the data distribution to curated instruction-response examples, with the masked risk of Definition 8.1. The objective is still token-level likelihood; the empirical measure is now concentrated on the desired interaction format.

### Preference optimization

Preference methods use comparisons between candidate outputs rather than raw likelihood. The RLHF-style objective is: given a reward model $r(x,y)$ trained from human comparisons, find a policy $\pi_\theta$ maximizing reward while staying close to a reference policy $\pi_{\mathrm{ref}}$ (usually the SFT model):

$$
\max_{\pi}\;
\mathbb{E}_{y\sim\pi(\cdot\mid x)}\big[r(x,y)\big]
\;-\;
\beta\,\mathrm{KL}\big(\pi(\cdot\mid x)\,\big\|\,\pi_{\mathrm{ref}}(\cdot\mid x)\big).
$$

This is the third appearance of the Gibbs variational principle promised in Chapter 1:

**Proposition 11.1 (the KL-regularized optimum is a Gibbs reweighting).** For each prompt $x$, the maximizer of the objective above over all distributions on responses is

$$
\pi^{\ast}(y\mid x)
= \frac{1}{Z(x)}\;\pi_{\mathrm{ref}}(y\mid x)\;
\exp\!\Big(\frac{r(x,y)}{\beta}\Big),
\qquad
Z(x) = \sum_{y}\pi_{\mathrm{ref}}(y\mid x)\,e^{\,r(x,y)/\beta}.
$$

*Proof.* Write the objective as

$$
\sum_y \pi(y)\,r(x,y) - \beta \sum_y \pi(y)\log\frac{\pi(y)}{\pi_{\mathrm{ref}}(y)}
= \sum_y \pi(y)\Big[r(x,y) + \beta\log\pi_{\mathrm{ref}}(y)\Big] + \beta H(\pi),
$$

which is exactly the objective of Theorem 1.6 with scores $z_y = r(x,y) + \beta\log\pi_{\mathrm{ref}}(y)$ and temperature $\beta$. By that theorem, the unique maximizer is

$$
\pi^\ast(y) = \operatorname{softmax}(z/\beta)_y
\;\propto\; e^{r(x,y)/\beta}\, \pi_{\mathrm{ref}}(y),
$$

as claimed. $\blacksquare$

The reference model plays the role of a prior; the reward tilts it exponentially; $\beta$ is the temperature of the tilt. Inverting Proposition 11.1 expresses the reward in terms of the optimal policy,

$$
r(x,y) = \beta \log\frac{\pi^{\ast}(y\mid x)}{\pi_{\mathrm{ref}}(y\mid x)} + \beta\log Z(x),
$$

and this inversion is the heart of **DPO** (direct preference optimization): substituting it into the Bradley–Terry model of pairwise preferences makes the intractable $\log Z(x)$ *cancel* between the preferred response $y^{+}$ and the rejected response $y^{-}$, leaving a loss expressed directly in policy log-ratios with no explicit reward model:

$$
\mathcal{L}_{\mathrm{DPO}}(\theta)
= -\,\mathbb{E}_{(x,y^{+},y^{-})}\left[
\log \sigma\!\Big(
\beta \log\frac{\pi_\theta(y^{+}\mid x)}{\pi_{\mathrm{ref}}(y^{+}\mid x)}
- \beta \log\frac{\pi_\theta(y^{-}\mid x)}{\pi_{\mathrm{ref}}(y^{-}\mid x)}
\Big)\right],
$$

where $\sigma(u) = 1/(1+e^{-u})$ is the logistic sigmoid. The common idea across RLHF and DPO: desired behavior is not fully specified by next-token likelihood on raw text, so preference data introduces a statistical signal over complete outputs — while the KL anchor keeps the policy in the neighborhood of the reference distribution.

### Prompting and in-context learning

Prompting changes the input sequence, not the parameters. In-context learning is the special case where the prompt includes demonstrations or instructions that specify a task. Mathematically, the model is still sampling from $p_\theta(\text{continuation}\mid \text{prompt})$; operationally, the prompt acts as task conditioning. This is why prompt length, example order, formatting, and irrelevant context can change behavior without any gradient update.

### Retrieval-augmented generation and tool use

Retrieval changes the conditional context at inference time by fetching documents and inserting them into the prompt. The model samples from

$$
p_\theta\big(x_{\text{answer}}\mid x_{\text{question}}, r_1,\ldots,r_k\big),
$$

where the $r_i$ are retrieved passages. System quality depends on both retrieval and generation. A useful view of RAG is problem decomposition: retrieve evidence, condition on it, generate faithfully. Tool use is a related decomposition where the system calls an external function *during* inference — a calculator, database, search engine, simulator, or API — producing new observations mid-generation, whereas basic RAG inserts retrieved text before generation. The failure modes follow the decomposition: irrelevant evidence, over- or under-constraining prompts, models that ignore evidence or fail to abstain.

### Evaluation

Evaluation must match the claim. Held-out language-model loss measures likelihood. Multiple-choice benchmarks measure a narrow decision format. Human preference ratings measure judged output quality under a sampling and prompt protocol. Serving metrics measure latency, throughput, memory, and reliability.

### Inference serving: prefill, decode, and scheduling

Modern serving distinguishes prefill from decoding (Chapter 7). Prefill builds the KV cache for the prompt with parallel matrix operations; decoding updates the cache one token at a time and is often memory-bandwidth-bound, because every generated token reads all cached keys and values. This explains why a long prompt and a long generated answer stress different parts of the system. Serving systems also batch requests: request-level batching groups full requests; continuous batching updates the active batch as requests arrive or finish; paged KV-cache systems split cache storage into blocks so variable-length sequences avoid large contiguous allocations. These are systems techniques around the same attention tensors studied earlier.

### Serving constraints

- memory for weights and KV cache,
- latency per generated token,
- batching and scheduling,
- quantization and kernels,
- context-window management,
- monitoring and rollback.

These constraints connect back to the same shapes: `V`, `T`, `C`, `H`, `D`, number of layers, and bytes per value.

### Code path

The `stage_map` in the next cell is intentionally small: a map from each practice area back to the objective, data distribution, or system constraint it modifies.

> **Mathematician's lens.** Modern LLM work can be classified by which object changes: the model family, the training distribution, the loss, the inference context, the decoding rule, or the deployment constraint. Preference alignment in particular is the Gibbs principle a third time (Proposition 11.1), now on the space of whole responses with the reference policy as prior.

> **ML practitioner's warning.** Avoid vague claims like "the model understands" without stating the objective, data, evaluation, and failure cases. Modern systems are composites, and failures can come from any component.

### Self-checks

1. Which parts of SFT change the model architecture, and which parts change the data/objective?
2. Why is retrieval an inference-time conditioning strategy rather than necessarily a parameter update?
3. What metric would you use for memory pressure during long-context decoding?
4. (*) Carry out the cancellation in the DPO derivation: substitute $r = \beta\log(\pi^\ast/\pi_{\mathrm{ref}}) + \beta\log Z$ into the Bradley–Terry probability $\Pr[y^{+}\succ y^{-}] = \sigma\big(r(x,y^{+})-r(x,y^{-})\big)$ and verify that $\log Z(x)$ drops out.


### Post-Training Stage Boundary

CS124 separates the base language model from post-training. The boundary is useful because each stage changes a different object.

| Stage | What changes | Training or inference signal | What it cannot guarantee alone |
| --- | --- | --- | --- |
| Pretraining | base model weights | next-token likelihood over broad text | instruction following, safety, freshness, task reliability |
| Supervised fine-tuning | model weights | curated instruction-response examples | preference quality beyond the demonstrations |
| Preference alignment | model weights or policy behavior | comparisons, rewards, or direct preference losses | truthfulness without evidence or perfect tool use |
| Test-time compute | inference procedure | extra reasoning steps, search, verification, or tool calls | better base knowledge or fixed bad retrieval |
| Retrieval/tool wrapping | external context and actions | search results, tool observations, memory state | correctness if sources or tools are wrong |

The undergraduate mistake is to call all of this "the LLM." The cleaner view is a pipeline: a pretrained conditional distribution is adapted by later data and then embedded in an inference system. If behavior changes because you added search, memory, a system prompt, or a verifier, the base next-token objective did not suddenly become a database or a planner. The system around it changed.

Source note: this section follows the CS124 post-training chapter `../data/cs124/slp3_chapters/9.pdf`, the 2025/2026 final LLM lectures, and the PA7 agent materials.


### Retrieval, Tools, And Memory Are Outside The Base LM

A base decoder-only language model maps context tokens to next-token probabilities. Modern applications often wrap that model in additional systems:

| System piece | Mathematical or engineering role | Not the same as |
| --- | --- | --- |
| Retrieval | map a query to external documents or chunks | changing model weights |
| Tool use | choose and call external functions | next-token prediction alone |
| Memory | persist state across interactions | the current context window |
| Web search | access time-sensitive external information | parametric knowledge |
| Agent loop | coordinate reasoning traces, tool calls, observations, and final responses | a single forward pass |

The distinction matters. If a model answers from retrieved context, the answer depends on the retriever and source text. If an agent calls a function, correctness depends on tool schemas, arguments, side effects, permissions, and observations. If memory is used, the system now has a state-management problem in addition to a language-modeling problem.

The CS124 PA7 agent assignment makes this boundary concrete: the agent decides which tool is relevant, calls it, observes the result, and then produces a response. That trajectory is a system behavior built around an LM, not a property of the LM alone.

Source note: this integrates PA7's tool-use, web-search, memory, and nondeterminism framing, plus Lab 3's distinction between lexical tf-idf retrieval and dense retrieval.


### Sparse Retrieval, Dense Retrieval, And RAG Failure Points

Retrieval is its own model or algorithmic subsystem. A useful first split is sparse versus dense retrieval.

| Retriever type | Representation | Strength | Common failure |
| --- | --- | --- | --- |
| Sparse lexical retrieval | term-count vectors with tf-idf, BM25, or related weights | exact words, rare terms, transparent inverted indexes | misses paraphrases and semantic matches with little word overlap |
| Dense retrieval | learned embedding vectors for queries and documents | can match related meanings without exact surface overlap | opaque similarities, domain drift, embedding bias, hard-to-debug misses |

RAG composes retrieval with generation:

$$
\text{question} \to \text{retriever} \to \text{passages} \to \text{LM context} \to \text{answer}.
$$

That decomposition gives you a debugging checklist. A bad RAG answer might come from missing source documents, poor chunking, weak query rewriting, a sparse/dense mismatch, bad ranking, context-window packing, prompt confusion, or generation that ignores the retrieved evidence. The final answer is produced by the LM, but the evidence available to the LM is controlled upstream.

For notebook 99, the key lesson is modest: retrieval changes the context, not the pretrained weights. A source-grounded system should therefore report both the model output and the retrieval evidence used to condition it.

Source note: this section comes from `../data/cs124/slp3_chapters/11.pdf`, `../data/cs124/slp3_slides/ir_nov25.pdf`, `../data/cs124/lectures/ir_jan17_2026.pdf`, `../data/cs124/labs/Lab3_InformationRetrieval.md`, and `../data/cs124/pa3-information-retrieval/`.


In [ ]:
stage_map = {
    "pretraining": "learn next-token prediction over broad text",
    "supervised fine-tuning": "teach instruction-response formats",
    "preference optimization": "shape outputs using preference signals",
    "retrieval": "condition generation on fetched context",
    "quantization": "reduce memory or bandwidth with lower precision",
    "evaluation": "test behavior without fooling yourself",
}
stage_map


### Where We Stand

**What we now have.** The system view. Each practice-area changes a specific object: pretraining the weights via raw likelihood, SFT the measure, preference optimization the policy relative to a reference (with the Gibbs tilt as its closed form), prompting the conditioning context, retrieval the evidence in that context, and serving the latency/memory envelope. "The LLM" in a product is this whole pipeline, and failures can originate at any stage.

> **Definition so far.** *An autoregressive language model is the probabilistic core of a system: a pretrained, fine-tuned, preference-tilted conditional distribution, conditioned at inference on engineered context, sampled through a policy, and served under memory and latency constraints.*

**What is still missing.** A view beyond the transformer. Everything so far took the architecture as given. But the $T \times T$ score matrix is a *choice* with a quadratic price, the KV cache is a *choice* of exact memory, and next-token likelihood is a *choice* of objective — and the research frontier is precisely the space of alternatives to each.

**Where Chapter 12 goes.** The bottleneck lens: sparse and local attention, head-sharing, state-space models, retrieval-as-memory, and representation-first objectives — each read as a hypothesis about which constraint of the standard transformer can be relaxed without losing what makes it work.

## 12. Beyond Transformers And World Models

The attention section exposed the dense `(T, T)` score matrix. That matrix is the reference point for many architecture questions. Beyond-transformer work is easier to read when you ask which bottleneck is being targeted.

### Sparse and local attention

Sparse attention changes the allowed attention graph. Instead of every position reading every earlier position, each position may read a local window or a selected pattern of positions. If a local window has size $w$, score count becomes closer to

$$
O(Tw)
$$

instead of

$$
O(T^2).
$$

This targets compute and memory. The risk is that important long-range dependencies may be inaccessible or require many layers to propagate.

### Grouped-query and multi-query attention

During decoding, keys and values are cached. Standard multi-head attention stores a separate key/value stream for every head. If there are $H$ heads, $D$ dimensions per head, $L$ layers, and $T$ cached positions, the head-related cache cost is proportional to

$$
L\cdot H\cdot D\cdot T.
$$

Multi-query attention keeps separate query heads but shares one key/value stream across all heads. Its cache cost is proportional to

$$
L\cdot 1\cdot D\cdot T.
$$

Grouped-query attention divides the query heads into $G$ groups. Each group has one shared key/value stream, so the cost is proportional to

$$
L\cdot G\cdot D\cdot T.
$$

When $G=H$, grouped-query attention becomes ordinary multi-head attention. When $G=1$, it becomes multi-query attention. This is a concrete example of a shape tradeoff: reduce the cache along the head axis, hopefully without losing too much expressiveness.

### State-space and recurrent alternatives

State-space models and recurrent models try to summarize history in a state rather than materialize all pairwise token interactions. The mathematical response is different: instead of a dense attention kernel over positions, maintain and update a compressed state. This targets long-context efficiency, but the representation must preserve the information needed for future predictions.

### Retrieval, external memory, and compressed context

Retrieval systems move some memory outside the model's fixed context. Instead of storing everything in parameters or the KV cache, the system searches an external corpus and conditions the model on selected results. This targets knowledge freshness, context length, and data access, but introduces retrieval errors and citation/evidence challenges.

Long-sequence methods can also be framed as memory design. The standard KV cache is an exact memory of prior keys and values, but it grows with sequence length. Sliding-window attention keeps only recent keys and values. Compressive or recurrent memory tries to store a fixed-size summary. k-NN or retrieval memory stores external vectors and retrieves relevant ones. Each option chooses a different answer to the same question: what information from the past should be available to the next query?

### JEPA, world-model, and representation-learning ideas

JEPA-style and world-model ideas shift attention from predicting the next token directly to learning predictive representations. The question becomes: what latent variables or representations should be preserved so that future observations, actions, or abstractions are predictable? This targets sample efficiency, planning, and representation quality, but evaluation is harder because the objective may be farther from observed text likelihood.

### A bottleneck table

| Direction | Mathematical response | Bottleneck targeted | Main risk |
| --- | --- | --- | --- |
| Sparse/local attention | restrict attention graph | compute and memory | missed long-range dependencies |
| Grouped-query attention | share K/V across query heads | KV-cache memory and bandwidth | reduced head expressivity |
| State-space/recurrent models | compress history into state | long-context scaling | state may forget needed details |
| Retrieval | condition on external selected text | knowledge and context limits | retrieval quality controls answer quality |
| JEPA/world models | predict representations or latent structure | sample efficiency and abstraction | objective/evaluation mismatch |

### Code path

The next code cell compares full attention score counts with a local window count. It is not a benchmark. It is a shape-based calculation that makes the bottleneck visible.

> **Mathematician's lens.** Architecture proposals are hypotheses about which structure in the computation can be constrained, shared, approximated, or moved outside the model without losing too much predictive power.

> **ML practitioner's warning.** Do not judge architecture work by name alone. Ask for the exact objective, complexity, hardware path, training data, evaluation protocol, and failure cases.

### Self-checks

1. If local attention uses window size 256, what is the score-count ratio between full and local attention at `T=8192`?
2. Which bottleneck does KV-cache sharing target during decoding?
3. Why might a representation-learning objective be harder to evaluate than next-token likelihood?


In [ ]:
sequence_lengths = [128, 512, 2048, 8192]
window = 256
comparison = []
for n in sequence_lengths:
    full = n * n
    local = n * min(n, window)
    comparison.append({
        "T": n,
        "full_attention_scores": full,
        "local_window_scores": local,
        "full_to_local_ratio": full / local,
    })
comparison


### How To Read A Future Architecture Paper

When you read a paper about sparse attention, state-space models, recurrent memory, retrieval, world models, or post-transformer architectures, classify the claim before judging it.

Ask:

1. **Objective:** What loss or training signal is optimized?
2. **Data distribution:** What data are used, and what filtering or supervision is assumed?
3. **Architecture:** What map is changed relative to a standard transformer?
4. **Shapes:** How do `B`, `T`, `C`, `H`, `D`, and number of layers affect memory and compute?
5. **Bottleneck:** Is the target compute, memory, sample efficiency, representation quality, planning, grounding, or evaluation?
6. **Complexity:** What is the asymptotic claim, and what constants or kernels matter?
7. **Evaluation:** Which benchmark or measurement supports the claim?
8. **Ablation:** What comparison isolates the proposed idea from scale, data, or tuning?
9. **Serving:** Does the method improve prefill, decoding, batching, KV-cache size, or hardware utilization?
10. **Failure cases:** What does the method make worse or leave unresolved?
11. **Inference phase:** Does the method primarily affect prefill, decoding, or both?
12. **Cache policy:** Does it store exact KV states, share them, compress them, page them, retrieve them, or recompute them?

This habit prevents architecture names from becoming buzzwords. It turns them into specific hypotheses about computation and learning.

### Small measurable experiment

Before expanding scope, design the smallest experiment that could falsify the claim you care about. For this repo, examples include:

- replacing full attention with a local mask and comparing tiny validation loss at fixed parameter count,
- measuring KV-cache bytes under different head-sharing assumptions,
- checking whether a masked SFT objective supervises exactly the intended response tokens,
- comparing top-k and top-p decoding on the same fixed logits,
- measuring the cache-size difference between MHA, MQA, and GQA for the same `L`, `H`, `D`, and `T`,
- timing a toy generation loop with and without recomputing the full context, while clearly labeling that the current repo does not implement a real KV cache.

The experiment should have a clear metric, a controlled baseline, and a reason it answers the bottleneck question.

> **Mathematician's lens.** A good architecture paper proposes a constrained function class or optimization procedure and then argues that the constraint improves some tradeoff.

> **ML practitioner's warning.** A benchmark win without a clear comparison can be caused by more data, more compute, better tuning, leakage, or evaluation choices rather than the named architectural idea.


### Where We Stand

**What we now have.** A reading method. Architecture proposals stopped being names and became classified hypotheses: identify the bottleneck (compute, cache, sample efficiency, objective), the mathematical response, and the risk — then demand the ablation that isolates the idea from scale and tuning. The twelve questions turn any future paper into an hour of structured work instead of an act of faith.

> **Definition so far — final form.** *An autoregressive language model is a parameterized family of conditional next-token distributions (Chapter 0), realized as a typed composition of embeddings, data-dependent convex mixing, and positionwise maps (Chapters 3–5), fitted by preconditioned stochastic KL descent (Chapters 1, 6), sampled through a Gibbs-family policy (Chapter 7), re-measured and preference-tilted toward desired behavior (Chapters 8, 11), stored at finite precision (Chapter 10), and embedded in a retrieval-and-serving system (Chapter 11) — one particular, historically contingent point (Chapter 12) in the space of sequence models.*

**What is still missing.** Only practice. The definitions are stable; fluency comes from moving among the four representations — sentence, object, shape, code — until the motion is automatic.

**Where Chapter 13 goes.** The study path: what to re-run, what to break on purpose, which exercises close which gaps, and how to keep the implementation thread when reading beyond this book.

## 13. How To Study From Here

Use this notebook as the overview pass, then return to the detailed notebooks and exercises. The main skill is not memorizing formulas. It is moving cleanly between four representations of the same idea:

1. a sentence, such as "the model attends causally over previous tokens";
2. a mathematical object, such as the masked, scaled score matrix and its row-Gibbs distributions (Definition 4.2, Proposition 4.6);
3. a tensor shape, such as `(B, H, T, T)`;
4. a code path, such as `CausalSelfAttention.forward`.

Recommended order:

1. Run this full notebook once without edits.
2. Re-run chapters 1-4 and write down every tensor shape by hand.
3. Complete `exercises/01_first_principles.md`.
4. Re-run chapters 5-8 and intentionally break one assertion in each chapter, then fix it.
5. Complete `exercises/02_training_and_generation.md`.
6. Spend a separate session on `10_quantization_deep_dive.ipynb` and `exercises/quantization.md`.
7. Finish with `12_beyond_transformers_and_world_models.ipynb` and `docs/notes/beyond_transformers_reading_map.md`.
8. Use Xiao and Zhu's *Foundations of Large Language Models* as a second-pass reference for topics this toy repo only orients around: pretraining variants, prompting, preference alignment, inference scheduling, and long-context memory.

### A compact checklist for each new concept

For every new LLM concept you study, write down:

- the mathematical object, with domain and codomain,
- the tensor shape,
- the axis of any softmax, mean, mask, or gather,
- the loss or inference rule it affects,
- the exact implementation location,
- one failure mode.

For example, for attention:

- mathematical object: data-dependent operator valued in the causal row-stochastic matrices $\mathcal{S}_T$ (Definition 4.3),
- tensor shape: `(B, H, T, T)` for weights,
- softmax axis: source-position axis `dim=-1`,
- affects: hidden states used to compute next-token logits,
- implementation: `src/llm_from_scratch/attention.py`,
- failure mode: missing causal mask leaks future tokens.

For cross-entropy:

- mathematical object: $\ell(z,y) = A(z) - z_y$ (Definition 1.7), gradient $p - e_y$ (Proposition 1.8),
- tensor shape: logits `(B, T, V)`, targets `(B, T)`,
- gather axis: vocabulary axis,
- affects: scalar training objective,
- implementation: `cross_entropy_from_logits` and `TinyGPT.forward`,
- failure mode: shifted labels or masks are wrong.

### Final perspective

The toy model is small, but it contains the core contracts. In one typed line (Chapter 5):

$$
\mathcal{V}^{T}
\xrightarrow{\ \mathrm{In}\ }
\mathbb{R}^{T\times C}
\xrightarrow{\ \mathcal{B}_1\ }
\cdots
\xrightarrow{\ \mathcal{B}_L\ }
\mathbb{R}^{T\times C}
\xrightarrow{\ \operatorname{LN}_f\ }
\mathbb{R}^{T\times C}
\xrightarrow{\ \cdot W_E^{\top}\ }
\mathbb{R}^{T\times V}
\xrightarrow{\ \sigma\ }
\big(\mathring{\Delta}^{V-1}\big)^{T},
$$

trained by minimizing an empirical estimate of $H(q) + \mathrm{KL}(q\|p_\theta)$ (Theorem 1.9), sampled from by a decoding policy (Chapter 7), and aligned, when preference data exist, by tilting a reference policy along a Gibbs reweighting (Proposition 11.1). Modern systems add scale, data engineering, alignment objectives, retrieval, quantization, serving, and evaluation. Those additions are easier to understand when the core composition is already clear.

And the single recurring structure worth remembering: the Gibbs variational principle of Theorem 1.6 appears at the output head (softmax over the vocabulary), inside every attention row (softmax over prefix positions, temperature $\sqrt D$), and at the alignment stage (softmax over whole responses, temperature $\beta$, prior $\pi_{\mathrm{ref}}$). One theorem, three scales.

> **Mathematician's lens.** Treat each model component as a map with a domain, codomain, invariants, and composition rules — and notice that the same variational principle organizes the probabilistic choices at every level.

> **ML practitioner's warning.** Understanding the formulas is necessary but not sufficient. Real model quality depends on data, implementation details, compute, evaluation design, and deployment constraints.


### NLP Around The Core LM

The rest of CS124 is a reminder that next-token prediction is a central objective, not the entire field. Many NLP tasks ask for explicit structure, grounding, or interaction.

| Area | What it asks the system to do | Why it matters for LLM understanding |
| --- | --- | --- |
| Sequence labeling | assign tags such as POS or named-entity labels to token positions | many outputs are structured labels, not free-form continuations |
| Parsing | recover constituency or dependency structure | grammar can involve relations that are not adjacent in the string |
| Information extraction | identify entities, relations, events, and times | useful answers often need schema-grounded facts |
| Semantic role labeling | identify who did what to whom, where, when, and how | fluent text can still confuse event roles |
| Sentiment and connotation | model affective meaning and social associations | word meaning includes stance, affect, and bias, not just reference |
| Coreference and entity linking | track mentions and connect them to entities | long outputs need stable referents across sentences |
| Discourse coherence | model how sentences fit together | paragraph quality is more than local next-token plausibility |
| Speech | map between waveforms and linguistic units | language systems often start or end outside text |
| Dialogue and agents | coordinate turns, goals, tools, memory, and state | applications are loops around models, not one forward pass |

This table should keep future reading grounded. If a paper claims an LLM "understands language," ask which of these behaviors was actually measured and which were only implied by fluent generation.

Source note: this boundary map comes from the later CS124/SLP3 chapters, especially `../data/cs124/slp3_chapters/17.pdf` through `25.pdf`, appendices `E.pdf` through `K.pdf`, the speech materials, and the PA7/lab agent sources.


### CS124 Source-Backed Second Pass

The local CS124 bundle can be used as a depth pass after this notebook. The full corpus review for this pass is recorded in `docs/notes/cs124_full_corpus_review_for_notebook_99.md`.

| If this notebook section feels weak | Read or inspect next | What to extract |
| --- | --- | --- |
| Tokenization | `../data/cs124/slp3_chapters/2.pdf`, `../data/cs124/lectures/tokens_jan26.pdf`, `../data/cs124/pa1-regular-expressions/pa1.ipynb` | type/instance distinctions, regex tokenization, BPE learner/encoder, Unicode and byte-level tokenization |
| Language modeling | `../data/cs124/slp3_chapters/3.pdf`, `../data/cs124/lectures/lm_jan25.pdf` | n-gram baselines, smoothing, interpolation, train/dev/test protocol, perplexity |
| Embeddings | `../data/cs124/slp3_chapters/6.pdf`, `../data/cs124/lectures/vector26jan.pdf`, `../data/cs124/pa4-embeddings/pa4.ipynb` | embedding matrices, cosine, static embeddings, dense retrieval intuition |
| Neural networks | `../data/cs124/lectures/7_NN.pdf`, `../data/cs124/pa5-neural-networks/pa5.ipynb` | units, hidden layers, backprop, cross-entropy as training signal |
| Transformers | `../data/cs124/lectures/transformer_cs124_26.pdf`, `../data/cs124/pa6a-transformers/model.py` | Q/K/V roles, residual stream, causal masking, attention implementation contracts |
| Evaluation and generation | `../data/cs124/slp3_chapters/7.pdf`, `../data/cs124/pa6a-transformers/perplexity.py` | greedy versus sampling, temperature, perplexity computation, detector caveats |
| Retrieval and agents | `../data/cs124/labs/Lab3_InformationRetrieval.md`, `../data/cs124/pa7-agent/README.md` | tf-idf versus dense retrieval, tool calls, web search, memory, nondeterminism |
| Architecture families | `../data/cs124/slp3_chapters/8.pdf`, `../data/cs124/slp3_chapters/10.pdf`, `../data/cs124/slp3_chapters/12.pdf` | decoder-only, encoder-only, masked LM, encoder-decoder, cross-attention, teacher forcing |
| Post-training | `../data/cs124/slp3_chapters/9.pdf`, `../data/cs124/lectures/final124_26.pdf`, `../data/cs124/lectures/finallecture_cs124_2025.pdf` | instruction tuning, preference alignment, test-time compute, stage boundaries |
| Broader NLP context | `../data/cs124/slp3_chapters/17.pdf` through `25.pdf`, `../data/cs124/slp3_chapters/K.pdf` | sequence labeling, parsing, semantics, coreference, discourse, conversation, dialogue state |

The study rule is the same as the notebook rule: turn every source idea into a mathematical object, a tensor shape, a code path, and one checkable question.
